# ScamShield BD — Reproducible Final Pipeline

> **Generated file:** Built by `tools/build_notebook.py`. Do not hand-edit this notebook.

## 1. Libraries & Environment
- Python 3.10–3.12; PyPI packages: `numpy`, `pandas`, `scipy`, `scikit-learn`, `joblib`, `ftfy`, `regex`, `unidecode`, `sentence-transformers`, `torch`, `python-dotenv` (exact runtime versions printed in Cell 2).
- Build-time dependency: `nbformat`. Clean-run packages are installed/verified in Cell 2.
- Licences: scientific Python libraries are BSD/MIT-compatible; the pretrained encoder is Apache-2.0 (below).

## 2. External Datasets, Pretrained Models & APIs (from `MEMORY.md` §11)
| Resource | Type | Source / provider | Licence | Purpose | Declared in notebook? |
|---|---|---|---|---|---|
| scikit-learn, pandas, numpy, scipy, ftfy, regex, unidecode, torch, python-dotenv | libraries | PyPI | BSD/MIT | modelling, preprocessing, data handling, clean-run install | [x] |
| sentence-transformers + sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2 | pretrained model | Hugging Face | Apache-2.0 | Model 2 OOF + full-test embeddings and OOD similarity | [x] |
| LLM API (if used) | hosted API | Configurable via `.env` (`LLM_PROVIDER`, `LLM_MODEL`, `LLM_API_KEY`) | Provider terms | frozen `llm_labels_v1.jsonl` is embedded; `REGENERATE_SYNTH=False` needs no key; key only for explicit regeneration | [x] |
| Hand-written extra examples (84 rows / 14 groups) | `data/extra/handwritten.py` | our own, authored by A from PRD + labeled sample patterns only; zero overlap with test | n/a | declared only; not used in final E028/Step 5 pipeline | [x] |
| Pseudo-labeling on `test.csv` text | method | our code | n/a | semi-supervised (adopted in E025/Step 5; frozen file embedded) | [x] |

- **TF-IDF Vocabulary Policy:** In the final `prediction.main()` path, `fit_all()` passes no `df_test`, so TF-IDF word/char vocabulary and IDF are fit on **labeled + augmented training text only**. `TfidfBlock.fit()` can optionally include test text for unsupervised IDF when a caller supplies `test_d`, but that path is not used to produce the submitted candidate.
- **Model Features:** `prediction.py` uses word and char TF-IDF on `text_norm` (preserved text) and `text_skel` (skeleton), without message blocks or a raw-char branch.
- **Secrets / Keys:** Read via `os.getenv` / `python-dotenv`. `REGENERATE_SYNTH=False`, so the clean run uses frozen LLM/pseudo-label files and needs **no API key**. A key is required only when explicitly enabling regeneration.

## 3. Method & Validation Summary
- **Final model (Step 5 / E028):** 5x3 grouped CV with LR (E026) and LinearSVC (E027), promoted LLM + pseudo-label extras, full-test MiniLM Model 2 (E020), equal M1/M2 blend weights (0.50/0.50) for Tasks A/B/C, cross-fitted Task B thresholds, and a constant-confidence fallback. It writes `outputs/predictions/submission_E<mmdd>_<hhmm>.csv`, copied to `submission.csv`.
- **Validation:** Deterministic saved 5-fold × 3-repeat grouped CV (`outputs/validation_results/folds.csv`) with duplicate/template isolation and leave-cluster-out checks. Nested cross-fitted F_B=0.7103; calibrator Brier=0.1283 (constant baseline).
- **Corruption Robustness:** Set A augmentation is applied to training folds only (inheriting source `group_id`); Set B corruption is evaluation-only.

## 4. Reproducibility & Execution
- **Random Seed:** `PYTHONHASHSEED=42` and `config.set_seed(42)` for `random`, `numpy`, and `torch`; the embedded `folds.csv` is loaded, not rebuilt differently.
- **Run Order:** Install/version cell → setup → module and frozen-data `%%writefile` cells → imports/seeding → data location → Model 2 OOF + test embeddings (500,000 rows; GPU if available) → `prediction.main()` → strict validation → `submission.csv` → optional comparison with `SCAMSHIELD_COMPARE_SUBMISSION`.
- **Runtime:** Model 2 embeddings: about 6–9 min on a Kaggle T4/P100 GPU, roughly 3 h on CPU. The 500k-row preprocessing is cached and `predict_df` runs in 50,000-row chunks; emit a loud CPU warning before the full run. `SCAMSHIELD_TEST_SLICE=1000` executes a fast smoke test.


In [ ]:
import importlib
import importlib.metadata
import os
import platform
import subprocess
import sys

os.environ["PYTHONHASHSEED"] = "42"

REQUIRED_PACKAGES = [
    ("numpy", "numpy"),
    ("pandas", "pandas"),
    ("scipy", "scipy"),
    ("sklearn", "scikit-learn"),
    ("joblib", "joblib"),
    ("regex", "regex"),
    ("ftfy", "ftfy"),
    ("unidecode", "Unidecode"),
    ("sentence_transformers", "sentence-transformers"),
    ("torch", "torch"),
    ("dotenv", "python-dotenv"),
]

for import_name, pip_name in REQUIRED_PACKAGES:
    try:
        importlib.import_module(import_name)
    except ImportError:
        print(f"Installing missing package: {pip_name}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])

print(f"Python:       {sys.version.split()[0]} ({platform.platform()})")
for pip_name in (
    "numpy", "pandas", "scipy", "scikit-learn", "joblib", "regex", "ftfy",
    "Unidecode", "sentence-transformers", "torch", "python-dotenv",
):
    try:
        print(f"{pip_name + ':':<22}{importlib.metadata.version(pip_name)}")
    except importlib.metadata.PackageNotFoundError:
        print(f"{pip_name + ':':<22}unknown")


In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

# Load environment variables without hardcoding any keys
load_dotenv()

for pkg_dir in ['data', 'data/extra', 'src', 'src/evaluation', 'src/features', 'src/models', 'src/training', 'tools']:
    p = Path(pkg_dir)
    p.mkdir(parents=True, exist_ok=True)
    init_file = p / "__init__.py"
    if not init_file.exists():
        init_file.write_text("", encoding="utf-8")

for out_sub in (
    "outputs/validation_results",
    "outputs/oof",
    "outputs/predictions",
):
    Path(out_sub).mkdir(parents=True, exist_ok=True)

print("Package layout and environment initialized.")


In [ ]:
%%writefile src/config.py
"""Shared configuration: seeds, paths, constants.

Owned by Teammate B (T-001). Everyone imports this; create first.
"""
import os
import random
from pathlib import Path

import numpy as np


# ── Root ──────────────────────────────────────────────────────────────────────
ROOT = Path(__file__).resolve().parents[1]
IS_KAGGLE = os.path.exists("/kaggle")


# ── Helpers ───────────────────────────────────────────────────────────────────
def find_data_dir() -> Path:
    """Search for the data directory containing sample.csv (works locally and on Kaggle)."""
    candidates = [
        ROOT / "data",
        ROOT.parent / "data",
        Path("..") / "data",
        Path(".") / "data",
        Path("/kaggle/input"),
    ]
    for c in candidates:
        if (c / "sample.csv").exists():
            return c
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        for sub in kaggle_input.iterdir():
            if sub.is_dir() and (sub / "sample.csv").exists():
                return sub
        nested = find_nested_data_dir(kaggle_input)
        if nested is not None:
            return nested
    return ROOT / "data"   # default; loader will raise a clear error if missing


def find_nested_data_dir(base: Path) -> Path | None:
    """Return the sorted-first directory under base that contains sample.csv."""
    if not base.exists():
        return None
    for path in sorted(base.rglob("sample.csv")):
        if path.is_file():
            return path.parent
    return None


def set_seed(seed: int = 42) -> None:
    """Seed random, numpy, and torch (if installed) for full reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    try:
        import torch
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
    except ImportError:
        pass


# ── Run-time constants ────────────────────────────────────────────────────────
SEED = 42
DELIMITER = "|||"
N_SPLITS = 5
N_REPEATS = 3
MAX_LEN = 256
CONF_CLIP = (0.03, 0.97)
USE_LLM = bool(os.getenv("LLM_API_KEY"))
REGENERATE_SYNTH = False   # True = call API; False = load frozen file

# ── Paths ─────────────────────────────────────────────────────────────────────
DATA_DIR = find_data_dir()
OUT_DIR = Path("/kaggle/working") if IS_KAGGLE else ROOT / "outputs"
OUTPUT_DIR = ROOT / "outputs"
VALIDATION_DIR = OUTPUT_DIR / "validation_results"
PREDICTION_DIR = OUTPUT_DIR / "predictions"
CACHE_DIR = OUTPUT_DIR / "cache"
OOF_DIR = OUTPUT_DIR / "oof"

SAMPLE_PATH = DATA_DIR / "sample.csv"
TEST_PATH = DATA_DIR / "test.csv"
SAMPLE_SUBMISSION_PATH = DATA_DIR / "sample_submission.csv"

# ── Task definitions ──────────────────────────────────────────────────────────
NUM_CLASSES = 6
NUM_ACTIONS = 6
TECHNIQUES = ["T1", "T2", "T3", "T4", "T5", "T6"]

LABEL_NAMES = {
    0: "Legitimate",
    1: "MFS/OTP scam",
    2: "Prize/lottery scam",
    3: "Fake job/investment",
    4: "Delivery/parcel phishing",
    5: "Promotional spam",
}

ACTION_NAMES = {
    0: "No action",
    1: "Share credentials",
    2: "Pay / transfer money",
    3: "Click link",
    4: "Install / join / task",
    5: "Contact sender",
}

# ── Bootstrap output directories ──────────────────────────────────────────────
for _d in (VALIDATION_DIR, PREDICTION_DIR, CACHE_DIR, OOF_DIR):
    _d.mkdir(parents=True, exist_ok=True)


In [ ]:
%%writefile src/data_loader.py
import numpy as np
import pandas as pd

from src import config

LABEL_NAMES = {
    0: "Legitimate",
    1: "MFS/OTP scam",
    2: "Prize/lottery scam",
    3: "Fake job/investment",
    4: "Delivery/parcel phishing",
    5: "Promotional spam",
}

ACTION_NAMES = {
    0: "No action",
    1: "Share credentials",
    2: "Pay / transfer money",
    3: "Click link",
    4: "Install / join / task",
    5: "Contact sender",
}


def _read(path):
    # keep_default_na=False: texts like "NA", "null" or "nan" (possible noise-only
    # rows) must stay strings, and the techniques value "NONE" must not become NaN.
    df = pd.read_csv(path, keep_default_na=False, dtype={"text": str})
    df["text"] = df["text"].astype(str)
    return df


def load_sample():
    df = _read(config.SAMPLE_PATH)
    expected = ["id", "text", "label", "techniques", "action"]
    assert list(df.columns) == expected, f"sample.csv columns: {list(df.columns)}"
    df["label"] = df["label"].astype(int)
    df["action"] = df["action"].astype(int)
    return df


def load_test():
    df = _read(config.TEST_PATH)
    assert list(df.columns) == ["id", "text"], f"test.csv columns: {list(df.columns)}"
    return df


def load_submission_template():
    df = pd.read_csv(config.SAMPLE_SUBMISSION_PATH, keep_default_na=False)
    assert list(df.columns) == ["id", "label", "techniques", "action", "confidence"]
    return df


def parse_techniques(value):
    """'T1|T2' -> ['T1', 'T2'];  'NONE' -> []"""
    value = str(value).strip()
    if value == "" or value.upper() == "NONE":
        return []
    return [t.strip() for t in value.split("|") if t.strip()]


def techniques_to_multihot(series):
    """Series of 'T1|T2' strings -> array of shape (n, 6)."""
    out = np.zeros((len(series), len(config.TECHNIQUES)), dtype=int)
    for i, value in enumerate(series):
        for t in parse_techniques(value):
            out[i, config.TECHNIQUES.index(t)] = 1
    return out


def multihot_to_techniques(row):
    """array of 6 zeros/ones -> 'T1|T2' or 'NONE'"""
    codes = [t for t, on in zip(config.TECHNIQUES, row) if on]
    return "|".join(codes) if codes else "NONE"


In [ ]:
%%writefile data/extra/handwritten.py
"""Hand-written hard-legit and minimal-pair examples for A5.

Written from the competition definitions and labeled sample patterns only; no test rows were read or copied. Each six-row group shares a `group_id` so folds never split a minimal pair.
"""
import pandas as pd

VALID_TECHNIQUES = {"T1", "T2", "T3", "T4", "T5", "T6"}
COLUMNS = ["id", "text", "label", "techniques", "action", "group_id", "source", "weight", "family", "language"]


def _tech(*codes):
    if not codes:
        return "NONE"
    return "|".join(sorted(codes))


ROWS = [
    (90001, 9001, "bKash support: your account will be paused tomorrow. Reply with the 6-digit code we just sent to confirm you are the owner.", 1, _tech("T1", "T2", "T5"), 1, "otp", "en"),
    (90002, 9001, "Nagad helpline: apnar account e sobchhe chole gese. KYC update korte PIN ar OTP ta ei number e pathan, na hole block hobe.", 1, _tech("T1", "T2", "T5"), 1, "otp", "banglish"),
    (90003, 9001, "S: We detected a login on your wallet ||| R: I am not at home ||| S: Then share the code we sent to keep access", 1, _tech("T1", "T2", "T5"), 1, "otp", "en"),
    (90004, 9001, "bKash: Your new device login was approved at 18:42. If this was not you, call 16247 from the app. Never share your OTP with anyone, including our staff.", 0, _tech(), 0, "otp", "en"),
    (90005, 9001, "বিকাশ: আপনার অ্যাকাউন্টে নতুন ডিভাইস যোগ করা হয়েছে। যদি আপনি না করেন, অ্যাপ থেকে ১৬২৪৭ কল করুন। কাউকে ওটিপি বা পিন দেবেন না।", 0, _tech(), 0, "otp", "bn"),
    (90006, 9001, "S: bKash support here. We will never ask for your OTP. ||| R: I got a code, should I read it? ||| S: No. Keep it private and report the message in the app.", 0, _tech(), 0, "otp", "en"),

    (90007, 9002, "Daraz refund desk: Tk 1,250 refund approved. Send card number, expiry, CVV and OTP to release it.", 1, _tech("T1", "T2", "T5"), 1, "refund", "en"),
    (90008, 9002, "Bhai, bhul kore apnar number e 5,000 taka gese. Ferot pathat hole bKash PIN ta bolen.", 1, _tech("T2", "T5"), 1, "refund", "banglish"),
    (90009, 9002, "S: I sent money to the wrong number by mistake ||| R: I did not receive it ||| S: Please check and send it back; verify with your PIN first", 1, _tech("T2", "T5"), 1, "refund", "en"),
    (90010, 9002, "bKash: Tk 1,250 refund from Daraz was credited to your wallet. TrxID BX91K7. No action needed; we never ask for PIN or OTP.", 0, _tech(), 0, "refund", "en"),
    (90011, 9002, "আপনার বিকাশ ওয়ালেটে ১,২৫০ টাকা রিফান্ড জমা হয়েছে। ট্রানজেকশন আইডি BX91K7। কোনো পিন বা ওটিপি দিতে হবে না।", 0, _tech(), 0, "refund", "bn"),
    (90012, 9002, "S: Did you receive the wrong-number transfer? ||| R: Yes, I will return it through bKash myself. ||| S: Good. Never ask anyone for your PIN to fix it.", 0, _tech(), 0, "refund", "en"),

    (90013, 9003, "SA Courier: parcel CX8821 stuck at customs. Pay Tk 450 clearance fee at hxxps://sa-clearing[.]top before 6 pm.", 4, _tech("T1", "T3", "T5"), 2, "delivery", "en"),
    (90014, 9003, "আপনার পার্সেল আটকে আছে। আজই ৫০ টাকা ডেলিভারি ফি দিতে লিংকে ক্লিক করুন: hxxps://parcel-fee[.]top", 4, _tech("T1", "T3", "T5"), 2, "delivery", "bn"),
    (90015, 9003, "S: DHL here, customs hold ||| R: I did not order anything ||| S: Pay Tk 900 duty today or the parcel returns", 4, _tech("T1", "T3", "T5"), 2, "delivery", "en"),
    (90016, 9003, "Pathao Parcel: delivery attempt 2 for tracking BD4471. Reschedule from the official app. Our riders never ask for advance payment by SMS link.", 0, _tech(), 0, "delivery", "en"),
    (90017, 9003, "পাঠাও পার্সেল: আপনার BD4471 প্যারাসেল আজ বিকেল ৪টায় ডেলিভারি হবে। সময় বদলাতে অ্যাপ ব্যবহার করুন; এসএমএস লিংকে টাকা দেবেন না।", 0, _tech(), 0, "delivery", "bn"),
    (90018, 9003, "S: Courier text says pay a fee via a link ||| R: Did you check the tracking number in the app? ||| S: Yes, it is already out for delivery. The link was fake.", 0, _tech(), 0, "delivery", "en"),

    (90019, 9004, "Lucky draw: your number won a Tk 50,000 voucher. Claim at hxxps://win-bd[.]top within 60 minutes.", 2, _tech("T4", "T5"), 3, "prize", "en"),
    (90020, 9004, "Oi apni 10 lakh taka jitechen. Claim korte age 300 tk processing fee pathan.", 2, _tech("T3", "T4", "T5"), 2, "prize", "banglish"),
    (90021, 9004, "S: Congrats, you won a smartphone ||| R: really? ||| S: Pay Tk 500 delivery charge and confirm your address", 2, _tech("T3", "T4"), 2, "prize", "en"),
    (90022, 9004, "Robi: EID bonus 2 GB added to your number. Check balance with *121*3#. No claim link or fee is required.", 5, _tech(), 0, "prize", "en"),
    (90023, 9004, "জিপির ঈদ অফার: ১ জিবি কিনলে ১ জিবি ফ্রি। *১২১*৩# ডায়াল করে দেখুন; কোনো ফি বা লিংক নেই।", 5, _tech(), 0, "prize", "bn"),
    (90024, 9004, "S: Is the cashback message real? ||| R: I checked the operator app, it is a genuine 10% top-up cashback. ||| S: Good, no need to click the SMS link.", 5, _tech(), 0, "prize", "en"),

    (90025, 9005, "Part-time task: rate 5 apps daily, earn Tk 800. Join hxxps://t[.]me/task_bd and deposit Tk 2,000 as security.", 3, _tech("T3", "T6"), 2, "job", "en"),
    (90026, 9005, "Freelance job: protidin 3000 tk income. WhatsApp e join korte fee 1500 tk pathan.", 3, _tech("T3", "T6"), 2, "job", "banglish"),
    (90027, 9005, "S: We saw your CV, data entry role ||| R: how much? ||| S: Tk 40,000 monthly, pay Tk 3,000 training fee first", 3, _tech("T1", "T3", "T6"), 2, "job", "en"),
    (90028, 9005, "Bdjobs: Your application for Data Entry Officer was received. Employer will contact you through Bdjobs messages only. Never pay to get a job.", 0, _tech(), 0, "job", "en"),
    (90029, 9005, "বিডিজবস: আপনার আবেদন গৃহীত হয়েছে। নিয়োগকর্তা শুধু বিডিজবস মেসেজে যোগাযোগ করবেন। টাকা দিয়ে চাকরি কেনা যাবে না।", 0, _tech(), 0, "job", "bn"),
    (90030, 9005, "S: Recruiter asking for a training fee ||| R: That is a scam; real employers never charge you ||| S: Thanks, I reported the chat.", 0, _tech(), 0, "job", "en"),

    (90031, 9006, "Invest Tk 10,000 in our crypto bot, get 25% weekly profit guaranteed. Send to bKash 01XXXXXXXXX now.", 3, _tech("T3", "T5", "T6"), 2, "investment", "en"),
    (90032, 9006, "Ami investment group e 300% profit dekhechi. Join korte Tk 5,000 fee dite hobe.", 3, _tech("T3", "T6"), 4, "investment", "banglish"),
    (90033, 9006, "S: Sir, double your money in 7 days ||| R: is it safe? ||| S: 100% guaranteed, transfer today only", 3, _tech("T3", "T5", "T6"), 2, "investment", "en"),
    (90034, 9006, "Bangladesh Bank advisory: verify any investment offer against the licensed list. High 'guaranteed' returns are a warning sign.", 0, _tech(), 0, "investment", "en"),
    (90035, 9006, "বাংলাদেশ ব্যাংক সতর্কতা: বিনিয়োগের আগে লাইসেন্স তালিকা যাচাই করুন। 'নিশ্চিত লাভ' প্রতারণার লক্ষণ হতে পারে।", 0, _tech(), 0, "investment", "bn"),
    (90036, 9006, "S: This investment chat promises guaranteed profit ||| R: Do not transfer; check the regulator list first ||| S: Agreed, I will report it.", 0, _tech(), 0, "investment", "en"),

    (90037, 9007, "Robi: your SIM will be blocked tomorrow for unverified KYC. Reply with NID number and OTP to keep it active.", 1, _tech("T1", "T2", "T5"), 1, "bill", "en"),
    (90038, 9007, "নগদ: অ্যাকাউন্ট বন্ধ হতে পারে। এখনই পিন নম্বর পাঠান।", 1, _tech("T1", "T2", "T5"), 1, "bill", "bn"),
    (90039, 9007, "S: Telecom support here ||| R: why? ||| S: Confirm your SIM by sharing the OTP we just sent", 1, _tech("T1", "T2", "T5"), 1, "bill", "en"),
    (90040, 9007, "Titas Gas bill reminder: Tk 860 due on 30 Nov. Pay through the official app or bank branch. Ignore SMS links asking for OTP.", 0, _tech(), 0, "bill", "en"),
    (90041, 9007, "টিটাস গ্যাস: আপনার বিল ৮৬০ টাকা ৩০ নভেম্বরের আগে দিতে হবে। অফিশিয়াল অ্যাপ বা ব্যাংক ব্যবহার করুন; ওটিপি চাওয়া বার্তা এড়িয়ে চলুন।", 0, _tech(), 0, "bill", "bn"),
    (90042, 9007, "S: Gas bill text says pay via link ||| R: I paid in the app; the link was not official ||| S: Good, never share OTP for bill payment.", 0, _tech(), 0, "bill", "en"),

    (90043, 9008, "Nagad app update required. Download hxxps://nagad-update[.]top/app.apk and enter old PIN to keep wallet active.", 1, _tech("T1", "T2", "T4", "T5"), 4, "service_link", "en"),
    (90044, 9008, "Voucher activate korte app install korun hxxps://gp-voucher[.]top/a.apk ar PIN din.", 1, _tech("T1", "T2", "T4"), 4, "service_link", "banglish"),
    (90045, 9008, "S: Support says install an update ||| R: from where? ||| S: this link, then share your old PIN", 1, _tech("T1", "T2", "T4"), 4, "service_link", "en"),
    (90046, 9008, "bKash: Scheduled maintenance tonight 01:00-03:00. Details: hxxps://bkash[.]com/notice. No app download or PIN entry is needed.", 0, _tech(), 0, "service_link", "en"),
    (90047, 9008, "বিকাশ: আজ রাত ১টা থেকে ৩টা রক্ষণাবেক্ষণ। বিবরণ hxxps://bkash[.]com/notice; কোনো পিন বা অ্যাপ ইনস্টলের প্রয়োজন নেই।", 0, _tech(), 0, "service_link", "bn"),
    (90048, 9008, "S: Maintenance notice has a link ||| R: I opened the official domain, not the forwarded one ||| S: Correct; never install apps from SMS links.", 0, _tech(), 0, "service_link", "en"),

    (90049, 9009, "Mama, ami bipode. Amar bKash e 3,000 taka nai. Tomar PIN ta dao, ami send kore nei.", 1, _tech("T2", "T5"), 1, "friend_warning", "banglish"),
    (90050, 9009, "Bhai, ami emergency e achhi. OTP ta patha diben, amar account block hoye jabe.", 1, _tech("T2", "T5"), 1, "friend_warning", "banglish"),
    (90051, 9009, "S: It is me, new number ||| R: really? ||| S: yes, send the OTP I just texted you", 1, _tech("T2", "T5"), 1, "friend_warning", "en"),
    (90052, 9009, "Ami ekta fake bKash message dekhechi. Kauke OTP ba PIN bolben na; ami amar number change korini.", 0, _tech(), 0, "friend_warning", "banglish"),
    (90053, 9009, "আমি একটা জাল bKash বার্তা দেখেছি। কাউকে ওটিপি বা পিন বলবেন না; আমি আমার নম্বর পরিবর্তন করিনি।", 0, _tech(), 0, "friend_warning", "bn"),
    (90054, 9009, "S: Bro, someone texted from my number asking for OTP ||| R: I did not share it ||| S: Good, block and report that number.", 0, _tech(), 0, "friend_warning", "en"),

    (90055, 9010, "You won Tk 7,000 cashback. Confirm by clicking hxxps://cb-bd[.]top and pay Tk 250 verification fee.", 2, _tech("T3", "T4", "T5"), 2, "cashback", "en"),
    (90056, 9010, "Cashback 5,000 tk jitechen. Link e click kore address confirm korun.", 2, _tech("T4", "T5"), 3, "cashback", "banglish"),
    (90057, 9010, "S: Congrats, cashback winner ||| R: wow ||| S: Click this link and pay Tk 200 processing", 2, _tech("T3", "T4"), 3, "cashback", "en"),
    (90058, 9010, "Aarong EID offer: 20% off on all dresses this week. Valid in store and on the official app; no fee or link needed.", 5, _tech(), 0, "cashback", "en"),
    (90059, 9010, "আরংগ ঈদ অফার: সব পোশাকে ২০% ছাড়। দোকান ও অফিশিয়াল অ্যাপে কার্যকর; কোনো ফি বা লিংক নেই।", 5, _tech(), 0, "cashback", "bn"),
    (90060, 9010, "S: Is this cashback real? ||| R: I checked the official app, yes, 10% top-up cashback. ||| S: Great, no need to reply to the SMS.", 5, _tech(), 0, "cashback", "en"),

    (90061, 9011, "Courier tracking BD9911 failed. Confirm address at hxxps://bd-track[.]top; pay Tk 60 redelivery now.", 4, _tech("T1", "T3", "T5"), 2, "tracking", "en"),
    (90062, 9011, "আপনার পার্সেলের ঠিকানা ভুল। hxxps://bd-track[.]top এ ক্লিক করে ৬০ টাকা দিন।", 4, _tech("T1", "T3"), 3, "tracking", "bn"),
    (90063, 9011, "S: Courier here ||| R: I have no parcel ||| S: It is in your name, pay Tk 60 customs today", 4, _tech("T1", "T3", "T5"), 2, "tracking", "en"),
    (90064, 9011, "Sundarban Courier: tracking BD9911 out for delivery today. Check status on the official website; no payment requested.", 0, _tech(), 0, "tracking", "en"),
    (90065, 9011, "সুন্দরবন কুরিয়ার: BD9911 আজ ডেলিভারিতে আছে। অফিশিয়াল ওয়েবসাইটে দেখুন; কোনো টাকা চাওয়া হয়নি।", 0, _tech(), 0, "tracking", "bn"),
    (90066, 9011, "S: Delivery message asks for a fee ||| R: Official tracking says out for delivery ||| S: Then it is phishing; ignore the link.", 0, _tech(), 0, "tracking", "en"),

    (90067, 9012, "HR: your salary is on hold. Share bank account and OTP to release Tk 45,000 today.", 1, _tech("T1", "T2", "T5"), 1, "payroll", "en"),
    (90068, 9012, "HR office: salary release korte account details ar OTP pathan.", 1, _tech("T1", "T2"), 1, "payroll", "banglish"),
    (90069, 9012, "S: Payroll team here ||| R: why OTP? ||| S: to verify salary account, send now", 1, _tech("T1", "T2", "T5"), 1, "payroll", "en"),
    (90070, 9012, "Payroll: salary Tk 45,000 credited to account ending 4412. Payslip is in the office portal. HR never asks for OTP.", 0, _tech(), 0, "payroll", "en"),
    (90071, 9012, "বেতন: ৪৫,০০০ টাকা ৪৪১২ নম্বর হিসাব জমা হয়েছে। পেসলিপ অফিস পোর্টালে আছে; এইচআর কখনো ওটিপি চায় না।", 0, _tech(), 0, "payroll", "bn"),
    (90072, 9012, "S: HR text asks for bank OTP ||| R: Never share it; I checked the portal ||| S: Reported to security.", 0, _tech(), 0, "payroll", "en"),

    (90073, 9013, "Customer loyalty survey: you qualified for Tk 15,000 gift. Pay Tk 100 handling fee at hxxps://survey-bd[.]top to claim.", 2, _tech("T1", "T3", "T4"), 2, "survey", "en"),
    (90074, 9013, "Survey e 15,000 tk gift. Claim korte hxxps://survey-bd[.]top e click korun.", 2, _tech("T4", "T5"), 3, "survey", "banglish"),
    (90075, 9013, "S: Loyalty reward selected ||| R: what? ||| S: Pay Tk 100 shipping for your gift", 2, _tech("T3", "T4"), 2, "survey", "en"),
    (90076, 9013, "Customer survey: tell us about network quality in the app and get 500 bonus points. No payment or personal data required.", 5, _tech(), 0, "survey", "en"),
    (90077, 9013, "কাস্টমার সার্ভে: অ্যাপে নেটওয়ার্ক মতামত দিন ৫০০ বোনাস পয়েন্ট পাবেন। কোনো টাকা বা ব্যক্তিগত তথ্য লাগবে না।", 5, _tech(), 0, "survey", "bn"),
    (90078, 9013, "S: Survey offers bonus points ||| R: I filled it in the official app ||| S: Good, no link or payment needed.", 5, _tech(), 0, "survey", "en"),

    (90079, 9014, "bKash support: apnar РIN expire today. Reply with currentP1N and new PIN to continue.", 1, _tech("T1", "T2", "T5"), 1, "mixed", "mixed"),
    (90080, 9014, "N0gad helpline: ver1fy account, send 0TP + P1N nowwww.", 1, _tech("T1", "T2", "T5"), 1, "mixed", "mixed"),
    (90081, 9014, "S: Assalamu alaikum, bKash agent ||| R: kotha bolun ||| S: apnar account verify korte PIN bolen ||| R: no ||| S: na hole SIM block", 1, _tech("T1", "T2", "T5"), 1, "mixed", "mixed"),
    (90082, 9014, "bKash notice: your PIN was changed today at 14:10. If not you, call 16247. We will never ask for OTP in chat.", 0, _tech(), 0, "mixed", "en"),
    (90083, 9014, "বিকাশ নোটিশ: আজ ১৪:১০ এ পিন বদল হয়েছে। আপনি না করলে ১৬২৪৭ কল করুন। চ্যাটে কখনো ওটিপি চাইব না।", 0, _tech(), 0, "mixed", "bn"),
    (90084, 9014, "S: Message says SIM will block ||| R: I checked in app, nothing pending ||| S: Good, delete it and never send OTP.", 0, _tech(), 0, "mixed", "en"),
]


def validate():
    """Check schema and label/technique/action consistency; return True."""
    assert len(ROWS) == 84, f"expected 84 rows, got {len(ROWS)}"
    ids = [r[0] for r in ROWS]
    assert len(ids) == len(set(ids)), "duplicate handwritten id"
    assert min(ids) >= 90001, "handwritten ids must not overlap sample/test ids"
    groups = {}
    for row in ROWS:
        _id, group, text, label, techniques, action, family, language = row
        groups.setdefault(group, []).append(_id)
        assert isinstance(text, str) and text.strip(), f"empty text at {_id}"
        assert text.count("|||") <= 11, f"too many messages at {_id}"
        assert label in range(6), f"bad label at {_id}"
        assert action in range(6), f"bad action at {_id}"
        assert family and language, f"missing metadata at {_id}"
        codes = [] if techniques == "NONE" else techniques.split("|")
        assert techniques == "NONE" or all(c in VALID_TECHNIQUES for c in codes), f"bad technique at {_id}"
        if techniques != "NONE":
            assert techniques == "|".join(sorted(codes)), f"techniques not sorted at {_id}"
        if label in (0, 5):
            assert techniques == "NONE" and action == 0, f"non-scam row has suspicious fields at {_id}"
        else:
            assert techniques != "NONE", f"scam row without technique at {_id}"
    assert len(groups) == 14, f"expected 14 groups, got {len(groups)}"
    assert all(len(v) == 6 for v in groups.values()), "each group must have 6 rows"
    return True


def load_handwritten(weight=0.75):
    """Return a DataFrame in the training schema with group/source/weight columns."""
    validate()
    df = pd.DataFrame(ROWS, columns=["id", "group_id", "text", "label", "techniques", "action", "family", "language"])
    df["source"] = "handwritten"
    df["weight"] = float(weight)
    return df[COLUMNS]


if __name__ == "__main__":
    out = load_handwritten()
    print(f"Validated {len(out)} handwritten rows in {out['group_id'].nunique()} groups")
    print(out["label"].value_counts().sort_index().to_dict())


In [ ]:
%%writefile src/preprocessing.py
"""Stage 2: normalisation, noise handling, conversation parsing, and multi-view builder.

Run from the project root:
    python -m src.preprocessing
"""
from collections import Counter
from functools import lru_cache
import math
import re
import time
import unicodedata

import ftfy
import pandas as pd

from src import config

# --- Unicode lookalikes (Cyrillic / Greek -> Latin) -------------------------
_LOOKALIKE_PAIRS = {
    "а": "a", "е": "e", "о": "o", "р": "p", "с": "c", "у": "y", "х": "x",
    "і": "i", "ј": "j", "ѕ": "s", "һ": "h", "ԁ": "d", "ԛ": "q", "ԝ": "w", "ı": "i",
    "А": "A", "В": "B", "Е": "E", "К": "K", "М": "M", "Н": "H", "О": "O",
    "Р": "P", "С": "C", "Т": "T", "Х": "X", "І": "I", "Ј": "J", "Ѕ": "S", "У": "Y",
    "ο": "o", "ν": "v", "ρ": "p", "ι": "i", "Α": "A", "Β": "B", "Ε": "E",
    "Ζ": "Z", "Η": "H", "Ι": "I", "Κ": "K", "Μ": "M", "Ν": "N", "Ο": "O",
    "Ρ": "P", "Τ": "T", "Χ": "X",
}
LOOKALIKES = str.maketrans(_LOOKALIKE_PAIRS)
BANGLA_DIGITS = str.maketrans("০১২৩৪৫৬৭৮৯", "0123456789")

BANGLA = "\u0980-\u09FF"
BANGLA_CHAR_RE = re.compile(rf"[{BANGLA}]")
LOOKALIKE_DETECT_RE = re.compile("[\u0400-\u04FF\u0370-\u03FF]")
ZW_ALWAYS_RE = re.compile("[\u200b\u2060\ufeff\u00ad]")
# ZWNJ is noise; ZWJ is kept only inside র + ZWJ + ্
ZWNJ_RE = re.compile("\u200c")
ZWJ_RE = re.compile("\u200d(?!্)|(?<!র)\u200d")
EMOJI_UNIT_RE = re.compile("[\U0001F000-\U0001FFFF\u2600-\u27BF]\ufe0f?")
EMOJI_RE = re.compile("[\U0001F000-\U0001FFFF\u2600-\u27BF\uFE00-\uFE0F]")
LEADING_EMOJI_RE = re.compile(r"^(?:[\U0001F000-\U0001FFFF\u2600-\u27BF\uFE00-\uFE0F]+\s*)+")
MOJIBAKE_OR_ENTITY_RE = re.compile(r"[ÃÂâð\ufffd]|&(?:amp|lt|gt|quot|apos|nbsp|#\d+);")

# Preserve masked 'x'/'X' sequences (e.g. 01XXXXXXXXX), collapse stretched 'x' inside words
REPEAT_RE = re.compile(r"(?![xX])([^\W\d_])\1{2,}|(?<=[A-WYZa-wyz])([xX])\2{2,}(?=[A-WYZa-wyz])")
PUNCT_REPEAT_RE = re.compile(r"([^\w\s|])\1{2,}")
WS_RE = re.compile(r"\s+")
DIGITS_RE = re.compile(r"\d+")
BANGLA_OR_DIGIT_RE = re.compile(rf"[{BANGLA}0-9]")
HAS_LETTERS_RE = re.compile(rf"[A-Za-z{BANGLA}]{{2,}}")
ANY_WORD_TOKEN_RE = re.compile(rf"[A-Za-z0-9{BANGLA}]+")

# Skeleton & digit-shape helpers (strict boundaries avoid splitting 40K, 10GB, 017XXXXXX73)
_SPACED_RE = re.compile(r"(?<![A-Za-z0-9])(?:[A-Za-z][ .\-_]){3,}[A-Za-z](?![A-Za-z0-9])")
_SPACED_SEP_RE = re.compile(r"[ .\-_]")
_NUM_RE = re.compile(r"(?<![A-Za-z0-9])\d+(?![A-Za-z0-9])")
_LOWER_N_TAG_RE = re.compile(r"<n(\d+)>")
_OTP_RE = re.compile(r"(?<!\d)\d{6}(?!\d)")

HXXP_RE = re.compile(r"hxxp(s?)://", re.I)
BRACKET_DOT_RE = re.compile(r"\[+\.+\]+|\(+\.+\)+")
EMPTY_BRACKET_DOT_RE = re.compile(r"(?<=[A-Za-z0-9-])\[+\.?\]+(?=[A-Za-z0-9-])")

# Lookahead (?=[a-z]) prevents testing all 22 TLDs at sentence-ending periods
URL_RE = re.compile(
    r"(?:https?://|www\.)[^\s<>\"']+"
    r"|(?<![a-z0-9-])[a-z0-9-]+\.(?=[a-z])(?:com|net|org|bd|co|io|xyz|link|info|top|vip|me|ly|site|click|online|shop|store|live|club|app|icu)\b[^\s<>\"']*",
    re.I,
)

SPEAKER_RE = re.compile(r"^\s*[\[\(]?([A-Za-z]{1,10})[\]\)]*\s*:\s*")
SPEAKER_CANON = {
    "S": "S", "R": "R", "A": "A", "B": "B",
    "SENDER": "SENDER", "YOU": "YOU", "YO": "YOU",
    "CALLER": "CALLER", "RECEIVER": "RECEIVER", "USER": "USER", "AGENT": "AGENT",
}

WORD_RE = re.compile(r"[A-Za-z]+")
ALNUM_TOKEN_RE = re.compile(r"[A-Za-z0-9]+")
LEET_MAP = {"0": "o", "1": "i", "3": "e", "4": "a", "5": "s", "7": "t", "8": "b"}
LEET_INNER_RUN_RE = re.compile(r"(?<=[A-Za-z])[0134578]+(?=[A-Za-z])")
LEET_ANY_RE = re.compile(r"[0134578]")
NON_LEET_DIGIT_RE = re.compile(r"[269]")
UNIT_OR_ID_RE = re.compile(
    r"^(?:\d+(?:gb|mb|kb|tb|k|m|b|tk|bdt|hrs?|mins?|sec|days?|st|nd|rd|th|x)"
    r"|(?:tk|bdt|rs|usd|ec|id|txn|trx|ref|ord|inv|no)\d+)$",
    re.I,
)
LEET_EXACT_MAP = {
    "m3": "me", "fr33": "free", "0tp": "otp", "p1n": "pin", "n1d": "nid",
    "5cam": "scam", "1ink": "link", "b4nk": "bank", "c4sh": "cash",
    "c0d3": "code", "pr1z3": "prize", "cl41m": "claim", "sh4r3": "share",
}

# Known noise tokens ONLY from the problem statement and sample.csv (RULES A6 compliant)
NOISE_VOCAB = {
    "wkrjfejejrn",  # problem statement §7
    "xkcdpl",       # sample.csv id 10022
    "asdkjh",       # sample.csv id 10033
    "zzqwpo",       # sample.csv id 10038
    "mnbvcx",       # sample.csv id 10053
    "lkjsdfoiu",    # sample.csv id 10058
    "xcvbnm",       # sample.csv id 10058
    "zzqpwoeiru",   # sample.csv id 10081
    "qzxvbn",       # sample.csv id 10082
}

# Domain/merged fragments that must never be dropped as noise
_KEEP_SUBSTRINGS = (
    "bkash", "bksh", "nagad", "mcash", "rocket", "upay", "paperfly", "dhl",
    "kyc", "otp", "pin", "cvv", "nid", "phone", "month", "crypt", "track",
    "claim", "thank", "passw", "cash", "card", "fraud", "bank", "taka",
    "offer", "spring", "instant", "transfer", "salary", "commiss",
)

# Rare consonant clusters for short gibberish-only detection (allows standard endings like -nks/-rds)
_GIBBERISH_CLUSTER_RE = re.compile(r"q(?!u)|[bcdfghjklmnpqrstvwxz]{4,}|^[bcdfghjklmnpqrstvwxz]{3,}")
_STD_ENDING_RE = re.compile(r"(?:nks|rds|sts|nds|nts|ths|rts|cts|pts|fts|sks|ght)$")
_COMMON_SHORT = {
    "ok", "hi", "ki", "na", "no", "yes", "call", "pay", "pin", "otp", "din", "jan",
    "mom", "bhai", "thanks", "bkash", "regards", "hello", "sorry", "please", "urgent",
}

VIEW_CONTRACT_COLS = (
    "text_raw", "text_norm", "text_skel", "n_msgs",
    "first_msg", "last_msg", "mid_msgs",
    "first_msg_len", "mean_msg_len", "max_msg_len",
    "has_url", "n_urls",
    "digit_ratio", "upper_ratio", "script_latin_ratio", "script_bangla_ratio",
    "emoji_count", "excl_count", "qmark_count", "asks_question",
    "has_otp_like_number", "noise_ratio",
)


def _is_missing(text):
    if text is None:
        return True
    if isinstance(text, float) and math.isnan(text):
        return True
    return str(text).strip() == ""


def to_skeleton(clean: str) -> str:
    """Cleaned text -> lowercase skeleton with spaced letters re-joined and digit runs as <N#>."""
    if not clean:
        return ""
    t = _SPACED_RE.sub(lambda m: _SPACED_SEP_RE.sub("", m.group(0)), clean)
    t = t.lower()
    # Keep existing <N#> tags uppercase so to_skeleton is strictly idempotent
    t = _LOWER_N_TAG_RE.sub(r"<N\1>", t)
    return _NUM_RE.sub(lambda m: f"<N{len(m.group(0))}>", t)


def _normalise_url_obfuscation(text):
    if "hxxp" in text.lower():
        text = HXXP_RE.sub(r"http\1://", text)
    if "[" in text or "(" in text:
        text = BRACKET_DOT_RE.sub(".", text)
        text = EMPTY_BRACKET_DOT_RE.sub(".", text)
    return text


def _decode_leet_token(match):
    tok = match.group(0)
    if tok.isalpha() or tok.isdigit():
        return tok
    low = tok.lower()
    if low in LEET_EXACT_MAP:
        repl = LEET_EXACT_MAP[low]
        return repl.upper() if tok.isupper() else repl
    if "xxx" in low or UNIT_OR_ID_RE.match(tok):
        return tok
    if not LEET_ANY_RE.search(tok):
        return tok
    # Case-independent guard for TrxIDs / reference codes (e.g. 9K4L2M1X / 9k4l2m1x, A4B1C7)
    if NON_LEET_DIGIT_RE.search(tok):
        return tok
    transitions = sum(tok[i].isdigit() != tok[i - 1].isdigit() for i in range(1, len(tok)))
    if transitions >= 4:
        return tok
    n_digits = sum(c.isdigit() for c in tok)
    if tok.isupper() and n_digits >= 2 and not tok.startswith(("P1N", "0TP")):
        return tok

    def _replace_match(m):
        out = []
        s = m.group(0)
        start = m.start()
        prev_up = start > 0 and tok[start - 1].isupper()
        next_up = m.end() < len(tok) and tok[m.end()].isupper()
        all_up = tok.isupper() or (prev_up and next_up)
        for ch in s:
            dec = LEET_MAP[ch]
            out.append(dec.upper() if all_up else dec)
        return "".join(out)

    if LEET_INNER_RUN_RE.search(tok):
        return LEET_ANY_RE.sub(_replace_match, tok)
    return tok


@lru_cache(maxsize=100_000)
def _is_noise_lower(t, use_vocab=True):
    if use_vocab and t in NOISE_VOCAB:
        return True
    if set(t) == {"x"}:
        return False

    n = len(t)
    vowels_y = 0
    run = 0
    longest_run = 0
    for ch in t:
        if ch in "aeiouy":
            vowels_y += 1
            run = 0
        else:
            run += 1
            if run > longest_run:
                longest_run = run

    ratio = vowels_y / n
    is_suspect = (
        vowels_y == 0
        or (n >= 8 and (longest_run >= 7 or (longest_run >= 6 and ratio < 0.20)))
    )
    if not is_suspect:
        return False
    return not any(k in t for k in _KEEP_SUBSTRINGS)


def is_noise_token(tok, use_vocab=True):
    """Known noise strings, 0-vowel >=6 char consonant strings, or long gibberish tokens."""
    if len(tok) < 6:
        return False
    return _is_noise_lower(tok.lower(), use_vocab=use_vocab)


def _is_gibberish_token(w):
    """True only for clearly unpronounceable/gibberish Latin tokens (protects Thanks/Bkash/Regards)."""
    if len(w) < 4:
        return False
    vowels = sum(c in "aeiouy" for c in w)
    if vowels == 0:
        return True
    if any(k in w for k in _KEEP_SUBSTRINGS):
        return False
    standard_ending = bool(_STD_ENDING_RE.search(w))
    if not standard_ending and _GIBBERISH_CLUSTER_RE.search(w):
        return True
    if not standard_ending and re.search(r"[bcdfghjklmnpqrstvwxz]{3,}$", w):
        return True
    return len(w) >= 7 and not standard_ending and (vowels / len(w)) < 0.18


def _is_gibberish_only(body, n_noise_removed=0):
    """Conservative whole-record check for short random-letter noise rows without Bangla, digits, or URLs."""
    if len(body) > 90 or body.count(" ") > 5:
        return False
    if BANGLA_OR_DIGIT_RE.search(body) or (("." in body or "://" in body) and URL_RE.search(body)):
        return False
    tokens = WORD_RE.findall(body.lower())
    if not tokens or len(tokens) > 3:
        return False
    if any(w in _COMMON_SHORT for w in tokens):
        return False
    if n_noise_removed > 0 and len(tokens) <= 2 and all(_is_gibberish_token(w) for w in tokens):
        return True
    suspicious = sum(_is_gibberish_token(w) for w in tokens)
    if suspicious == len(tokens):
        return True
    return suspicious > 0 and max(len(w) for w in tokens) >= 7 and len(tokens) <= 2


def _normalise_body_core(t, strip_emojis=True):
    """Internal core normalisation returning (text_with_url_placeholders, urls_list)."""
    if not t.isascii():
        if MOJIBAKE_OR_ENTITY_RE.search(t):
            t = ftfy.fix_text(t)
        t = unicodedata.normalize("NFKC", t)
        t = ZW_ALWAYS_RE.sub("", t)
        t = ZWNJ_RE.sub("", t)
        t = ZWJ_RE.sub("", t)
        t = t.translate(LOOKALIKES)
        t = t.translate(BANGLA_DIGITS)
        if strip_emojis:
            t = EMOJI_RE.sub(" ", t)
    elif "&" in t and MOJIBAKE_OR_ENTITY_RE.search(t):
        t = ftfy.fix_text(t)

    t = _normalise_url_obfuscation(t)

    urls = []
    if "." in t or "://" in t:
        def _stash_url(match):
            urls.append(match.group(0))
            return f" __URL{len(urls) - 1}__ "
        t = URL_RE.sub(_stash_url, t)

    if len(t) >= 3:
        t = REPEAT_RE.sub(lambda m: (m.group(1) or m.group(2)) * 2, t)
        t = PUNCT_REPEAT_RE.sub(r"\1", t)

    if LEET_ANY_RE.search(t):
        t = ALNUM_TOKEN_RE.sub(_decode_leet_token, t)

    return t, urls


def normalise_chars(text, strip_emojis=True):
    """Careful text normalisation preserving masked X-runs, IDs, URLs, Bangla ZWJ, and ASCII digits."""
    if _is_missing(text):
        return ""
    t, urls = _normalise_body_core(str(text), strip_emojis=strip_emojis)
    for idx, url in enumerate(urls):
        t = t.replace(f"__URL{idx}__", url)
    return WS_RE.sub(" ", t).strip()


def _strip_leading_noise_and_speaker(text):
    """Strip leading noise tokens (and leading emojis) so turn-initial speakers are always parsed."""
    n_noise = 0
    lead_emojis = []
    t = text.lstrip()
    while True:
        em = LEADING_EMOJI_RE.match(t)
        if em:
            lead_emojis.append(em.group(0).strip())
            t = t[em.end():].lstrip()
            continue
        m = WORD_RE.match(t)
        if m and is_noise_token(m.group(0)) and not t[m.end():].lstrip().startswith(":"):
            n_noise += 1
            t = t[m.end():].lstrip()
            continue
        break

    speaker = None
    sm = SPEAKER_RE.match(t)
    if sm:
        canon = SPEAKER_CANON.get(sm.group(1).upper())
        if canon:
            speaker = canon
            t = t[sm.end():]
    return speaker, t, n_noise, lead_emojis


@lru_cache(maxsize=250_000)
def _clean_turn(msg):
    """Single-pass cached turn cleaner -> (speaker, clean_tx, preserved_tx, n_noise, n_raw_tokens)."""
    t, urls = _normalise_body_core(msg, strip_emojis=False)
    speaker, t, lead_noise, lead_emojis = _strip_leading_noise_and_speaker(t)
    n_noise = lead_noise
    n_raw_tokens = len(ANY_WORD_TOKEN_RE.findall(t)) + lead_noise

    def _drop(match):
        nonlocal n_noise
        tok = match.group(0)
        if tok.startswith("__URL"):
            return tok
        if is_noise_token(tok):
            n_noise += 1
            return " "
        return tok

    t = WORD_RE.sub(_drop, t)
    for idx, url in enumerate(urls):
        t = t.replace(f"__URL{idx}__", url)

    if lead_emojis:
        pres_tx = WS_RE.sub(" ", " ".join(lead_emojis) + " " + t).strip()
    else:
        pres_tx = WS_RE.sub(" ", t).strip()

    if t.isascii():
        clean_tx = WS_RE.sub(" ", t).strip()
    else:
        clean_tx = WS_RE.sub(" ", EMOJI_RE.sub(" ", t)).strip()

    return speaker, clean_tx, pres_tx, n_noise, n_raw_tokens


def clean_message(msg, strip_emojis=True):
    """One message -> (speaker or None, cleaned text, n_noise_tokens)."""
    if _is_missing(msg):
        return None, "", 0
    speaker, clean_tx, pres_tx, n_noise, _ = _clean_turn(str(msg))
    return speaker, (clean_tx if strip_emojis else pres_tx), n_noise


def preprocess(text):
    """Raw record -> dict with cleaned views, skeleton, conversation turns, and structural features."""
    if _is_missing(text):
        return {
            "clean": "",
            "preserved": "",
            "skel": "",
            "first_msg": "",
            "last_msg": "",
            "n_messages": 0,
            "n_noise_tokens": 0,
            "noise_ratio": 1.0,
            "noise_only": True,
            "raw_len": 0,
            "emoji_count": 0,
            "has_url": False,
            "n_urls": 0,
            "script_bangla_ratio": 0.0,
            "had_lookalike": False,
            "had_zero_width": False,
            "had_emoji": False,
        }

    raw = str(text)
    if raw.isascii():
        emoji_count = 0
        had_emoji = False
        had_lookalike = False
        had_zero_width = False
    else:
        emoji_count = len(EMOJI_UNIT_RE.findall(raw))
        had_emoji = emoji_count > 0 or bool(EMOJI_RE.search(raw))
        had_lookalike = bool(LOOKALIKE_DETECT_RE.search(raw))
        had_zero_width = bool(ZW_ALWAYS_RE.search(raw))

    raw_msgs = raw.split(config.DELIMITER)
    parsed = [_clean_turn(m) for m in raw_msgs]

    clean_parts = []
    pres_parts = []
    body_parts = []
    n_noise_tokens = 0
    n_raw_tokens = 0

    for sp, clean_tx, pres_tx, n_noise, turn_tokens in parsed:
        n_noise_tokens += n_noise
        n_raw_tokens += turn_tokens
        if clean_tx:
            clean_parts.append(f"{sp}: {clean_tx}" if sp else clean_tx)
            body_parts.append(clean_tx)
        if had_emoji and pres_tx:
            pres_parts.append(f"{sp}: {pres_tx}" if sp else pres_tx)

    delim = f" {config.DELIMITER} "
    clean = delim.join(clean_parts)
    preserved = delim.join(pres_parts) if had_emoji else clean
    skel = to_skeleton(clean)
    body = " ".join(body_parts)
    n_messages = len(raw_msgs)
    first_msg = clean_parts[0] if clean_parts else ""
    last_msg = clean_parts[-1] if clean_parts else ""

    has_letters = bool(HAS_LETTERS_RE.search(body))
    noise_only = (not has_letters) or _is_gibberish_only(body, n_noise_tokens)
    noise_ratio = 1.0 if noise_only else min(1.0, n_noise_tokens / max(n_raw_tokens, 1))

    n_urls = len(URL_RE.findall(clean)) if ("." in clean or "://" in clean) else 0
    if body:
        n_bn = len(BANGLA_CHAR_RE.findall(body))
        script_bangla_ratio = round(n_bn / max(len(body), 1), 4)
    else:
        script_bangla_ratio = 0.0

    return {
        "clean": clean,
        "preserved": preserved,
        "skel": skel,
        "first_msg": first_msg,
        "last_msg": last_msg,
        "n_messages": n_messages,
        "n_noise_tokens": n_noise_tokens,
        "noise_ratio": float(noise_ratio),
        "noise_only": noise_only,
        "raw_len": len(raw),
        "emoji_count": int(emoji_count),
        "has_url": n_urls > 0,
        "n_urls": int(n_urls),
        "script_bangla_ratio": float(script_bangla_ratio),
        "had_lookalike": had_lookalike,
        "had_zero_width": had_zero_width,
        "had_emoji": had_emoji,
    }


def template_key(clean_text):
    """Key for grouping/deduplicating near-identical templates (not a model input)."""
    t = str(clean_text).lower()
    t = DIGITS_RE.sub("0", t)
    t = WS_RE.sub(" ", t).strip()
    return t


def preprocess_frame(df, log_every=0):
    raw_texts = [("" if _is_missing(t) else str(t)) for t in df["text"]]
    uniq_texts = list(dict.fromkeys(raw_texts))
    cache = {}
    total = len(uniq_texts)
    for i, t in enumerate(uniq_texts, 1):
        res = preprocess(t)
        res["template_key"] = template_key(res["clean"])
        cache[t] = res
        if log_every and i % log_every == 0:
            print(f"  preprocessed {i:,}/{total:,} unique rows...", flush=True)
    feats = pd.DataFrame([cache[t] for t in raw_texts], index=df.index)
    return pd.concat([df, feats], axis=1)


def build_views(df, log_every=0):
    """ARCHITECTURE.md §4 22-column contract wrapper."""
    out = preprocess_frame(df, log_every=log_every)
    out["text_raw"] = df["text"].fillna("").astype(str)
    out["text_norm"] = out["preserved"].fillna("")
    out["text_skel"] = out["skel"].fillna("")
    out["n_msgs"] = out["n_messages"]

    msgs = out["text_norm"].str.split(r"\s*\|\|\|\s*")
    n = out["text_norm"].str.len().clip(lower=1)
    out["mid_msgs"] = msgs.map(lambda m: " ||| ".join(m[1:-1]))
    out["first_msg_len"] = out["first_msg"].str.len()
    out["mean_msg_len"] = msgs.map(lambda m: sum(map(len, m)) / len(m))
    out["max_msg_len"] = msgs.map(lambda m: max(map(len, m)))
    out["digit_ratio"] = out["text_norm"].str.count(r"\d") / n
    out["upper_ratio"] = out["text_norm"].str.count(r"[A-Z]") / n
    out["script_latin_ratio"] = out["text_norm"].str.count(r"[A-Za-z]") / n
    out["excl_count"] = out["text_raw"].str.count("!")
    out["qmark_count"] = out["text_raw"].str.count(r"\?")
    out["asks_question"] = out["qmark_count"] > 0
    out["has_otp_like_number"] = out["text_norm"].str.contains(_OTP_RE, na=False)
    return out


def audit_heuristic_noise(texts, top_k=50):
    """Return top_k tokens flagged purely by the heuristic path (excluding NOISE_VOCAB)."""
    cnt = Counter(
        w.lower()
        for x in texts
        for w in WORD_RE.findall(normalise_chars(x))
        if is_noise_token(w, use_vocab=False) and w.lower() not in NOISE_VOCAB
    )
    return cnt.most_common(top_k)


def main():
    from src.data_loader import load_sample, load_test
    pd.set_option("display.width", 200)
    pd.set_option("display.max_colwidth", 120)

    s = build_views(load_sample())
    test_df = load_test()
    t0 = time.perf_counter()
    t = build_views(test_df, log_every=100_000)
    elapsed = time.perf_counter() - t0

    print(f"500k build_views runtime: {elapsed:.1f}s ({elapsed / 60:.2f} min)")
    print("sample noise-only:", int(s.noise_only.sum()), "| test noise-only:", int(t.noise_only.sum()))
    print("test rows with noise tokens removed:", f"{(t.n_noise_tokens > 0).mean():.2%}")
    print("unique template keys in test:", t.template_key.nunique())

    print("\n--- top 50 heuristic-flagged tokens (excluding NOISE_VOCAB; hand-check precision) ---")
    print(audit_heuristic_noise(test_df["text"].head(100_000), top_k=50))

    print("\n--- sample before/after (rows with noise, lookalikes, or Bangla digits) ---")
    mask = (s.n_noise_tokens > 0) | s.had_lookalike | s.had_zero_width | s.noise_only | (s.script_bangla_ratio > 0)
    for _, r in s[mask].head(10).iterrows():
        print("RAW  :", r.text[:130])
        print("CLEAN:", r.clean[:130])
        print("SKEL :", r.skel[:130], f"| noise_ratio={r.noise_ratio:.2f} noise_only={r.noise_only}\n")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/augment.py
"""Programmatic label-preserving corruption (Set A for training, Set B held out for evaluation).

Contract (ARCHITECTURE.md §4):
    corrupt(text, rng, level="medium", ops="A") -> str
    expand(df, k=3, seed=config.SEED, level="medium", ops="A") -> DataFrame
"""
import random

import pandas as pd

from src import config

_LEVEL_STRENGTH = {
    "light": 0.04,
    "medium": 0.07,
    "heavy": 0.12,
}

# Set A (training): leet + Cyrillic subset + standard typos/spacing/ZWSP/emojis
_SET_A_LEET = {"o": "0", "i": "1", "e": "3", "a": "4", "s": "5", "t": "7", "O": "0", "I": "1", "E": "3", "A": "4", "S": "5", "T": "7"}
_SET_A_HOMOGLYPHS = {"a": "а", "e": "е", "o": "о", "p": "р", "c": "с", "i": "і"}
_SET_A_EMOJIS = ["😊", "🎁", "🔥", "💰", "✅", "📱", "🙏"]
_ASCII_LETTERS = "abcdefghijklmnopqrstuvwxyz"

# Set B (evaluation only): held-out Greek/fullwidth/extended homoglyphs, transliterations,
# word-boundary removal, ZWNJ/soft-hyphen, and held-out synthetic noise strings
_SET_B_HOMOGLYPHS = {
    "A": "Α", "B": "Β", "E": "Ε", "Z": "Ζ", "H": "Η", "I": "Ι", "K": "Κ",
    "M": "Μ", "N": "Ν", "O": "Ο", "P": "Ρ", "T": "Τ", "X": "Χ",
    "o": "ο", "v": "ν", "p": "ρ", "i": "ι", "j": "ј", "s": "ѕ", "h": "һ",
    "a": "\uff41", "e": "\uff45", "t": "\uff54", "n": "\uff4e",  # fullwidth (NFKC stress)
}
_SET_B_RESPELLINGS = {
    "bhul": "vul", "vul": "bhul", "taka": "tk", "korte": "krte",
    "shob": "sob", "sob": "shob", "bolen": "bln", "ekhoni": "ekhn",
    "please": "pls", "message": "msg", "number": "nmbr", "account": "accnt",
    "verify": "vrify", "delivery": "dlvry", "free": "fre",
}
_SET_B_NOISE = ("qvrtxmnpk", "zplmnbvk", "krtwqzxp", "jhvfgbcn", "plmkqzxw", "vbnmkjhgf")
_SET_B_ZW = ("\u200c", "\u00ad", "\u2060")


def _corrupt_turn_a(text, rng, strength):
    out = []
    for ch in str(text):
        r = rng.random()
        if r < strength * 0.22:                                   # deletion
            continue
        if r < strength * 0.42:                                   # repetition
            out.append(ch * rng.randint(2, 3))
            continue
        if r < strength * 0.58 and ch in _SET_A_LEET:             # leet (e.g. Ver1fy, 0TP)
            out.append(_SET_A_LEET[ch])
            continue
        if r < strength * 0.72 and ch.isalpha():                  # random char substitution
            rep = rng.choice(_ASCII_LETTERS)
            out.append(rep.upper() if ch.isupper() else rep)
            continue
        if r < strength * 0.84 and ch in _SET_A_HOMOGLYPHS:       # Set A Cyrillic lookalike
            out.append(_SET_A_HOMOGLYPHS[ch])
            continue
        if r < strength * 0.92:                                   # ZWSP insertion
            out.append(ch + "\u200b")
            continue
        if r < strength:                                          # random case
            out.append(ch.swapcase())
            continue
        out.append(ch)

    t = "".join(out)
    if rng.random() < 0.30:
        t = t.replace(" ", "", rng.randint(1, 2))
    if rng.random() < 0.25:
        t = f"{t} {rng.choice(_SET_A_EMOJIS)}"
    if rng.random() < 0.15:
        t = f"{t}{'!' * rng.randint(2, 3)}"
    return t


def _corrupt_turn_b(text, rng, strength):
    # Transliteration-style respellings on whole words first
    words = str(text).split(" ")
    resp = []
    for w in words:
        lw = w.lower()
        if lw in _SET_B_RESPELLINGS and rng.random() < min(0.45, strength * 4):
            rep = _SET_B_RESPELLINGS[lw]
            resp.append(rep.upper() if w.isupper() else rep)
        else:
            resp.append(w)
    t = " ".join(resp)

    # Character-level held-out Greek/fullwidth homoglyphs, repetitions, ZWNJ/soft-hyphen, deletions
    out = []
    for ch in t:
        r = rng.random()
        if r < strength * 0.22:                                   # deletion
            continue
        if r < strength * 0.45:                                   # repetition
            out.append(ch * rng.randint(2, 3))
            continue
        if r < strength * 0.75 and ch in _SET_B_HOMOGLYPHS:       # Set B held-out homoglyph
            out.append(_SET_B_HOMOGLYPHS[ch])
            continue
        if r < strength * 0.88:                                   # ZWNJ / soft-hyphen / WJ
            out.append(ch + rng.choice(_SET_B_ZW))
            continue
        if r < strength:
            out.append(ch.swapcase())
            continue
        out.append(ch)
    t = "".join(out)

    # Word-boundary removal (merge adjacent words) + occasional spaced-out token
    if rng.random() < 0.40:
        t = t.replace(" ", "", rng.randint(1, 3))
    if rng.random() < 0.25:
        noise = rng.choice(_SET_B_NOISE)
        parts = t.split(" ")
        parts.insert(rng.randint(0, len(parts)), noise)
        t = " ".join(parts)
    if rng.random() < 0.20:
        parts = t.split(" ")
        long_idxs = [i for i, w in enumerate(parts) if len(w) >= 5 and w.isalpha()]
        if long_idxs:
            idx = rng.choice(long_idxs)
            sep = rng.choice([" ", "."])
            parts[idx] = sep.join(parts[idx])
            t = " ".join(parts)
    return t


def corrupt(text, rng, level="medium", ops="A"):
    """Label-preserving corruption keeping `|||` intact.

    ops='A': training operators (leet, random char sub, ZWSP, emojis, spacing).
    ops='B': held-out evaluation operators (Greek/fullwidth homoglyphs, respellings,
             word-boundary removal, ZWNJ/soft-hyphen, held-out noise strings).
    """
    strength = _LEVEL_STRENGTH.get(level, _LEVEL_STRENGTH["medium"])
    fn = _corrupt_turn_a if str(ops).upper() == "A" else _corrupt_turn_b
    turns = str(text).split(config.DELIMITER)
    corrupted = [fn(turn.strip(), rng, strength) for turn in turns]
    return f" {config.DELIMITER} ".join(corrupted)


def expand(df, k=3, seed=config.SEED, level="medium", ops="A", weight=0.75):
    """Create `k` corrupted training variants per row, inheriting `group_id`."""
    rng = random.Random(seed)
    rows = []
    has_group_id = "group_id" in df.columns
    has_group = "group" in df.columns

    for _, row in df.iterrows():
        gid = row["group_id"] if has_group_id else (row["group"] if has_group else row["id"])
        for _ in range(k):
            item = row.to_dict()
            item["text"] = corrupt(row["text"], rng, level=level, ops=ops)
            item["group_id"] = gid
            if has_group:
                item["group"] = gid
            item["source"] = "aug"
            item["weight"] = float(weight)
            rows.append(item)

    return pd.DataFrame(rows)


In [ ]:
%%writefile src/text_analysis.py
"""Stage 1: dataset exploration.

Run from the project root:
    python -m src.text_analysis
"""
import re
import unicodedata

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from src import config
from src.data_loader import (
    ACTION_NAMES,
    LABEL_NAMES,
    load_sample,
    load_submission_template,
    load_test,
    parse_techniques,
)

BANGLA_RE = re.compile(r"[\u0980-\u09FF]")
LATIN_RE = re.compile(r"[A-Za-z]")
CYRILLIC_RE = re.compile(r"[\u0400-\u04FF]")
SUSPICIOUS_ZW_RE = re.compile(r"[\u200b\u2060\ufeff]")  # ZWNJ/ZWJ are normal in Bangla, so excluded
SPEAKER_RE = re.compile(r"^\s*[A-Za-z]{1,10}\s*:")

BANGLISH_WORDS = {
    "apnar", "apni", "ami", "tumi", "korun", "korte", "kore", "koren", "bolen", "bolun",
    "taka", "ta", "ei", "eta", "ekta", "ase", "ache", "hoyeche", "hobe", "dite", "din",
    "jan", "pathan", "pathaisi", "ferot", "bhul", "vul", "kauke", "bolben", "na", "nai",
    "sob", "shob", "jitechen", "jiteche", "income", "ghor", "bose", "dhonyobad", "meyad",
    "shortto", "projojjo", "kinle", "dhonnobad", "bhai", "apu", "lagbe", "ki", "ekhon",
    "bkash", "nagad", "rocket", "upay", "shuru", "ekhoni", "joldi", "onek",
}


def language_bucket(text):
    has_bn = bool(BANGLA_RE.search(text))
    has_lat = bool(LATIN_RE.search(text))
    if has_bn and has_lat:
        return "mixed_bn_en"
    if has_bn:
        return "bangla_script"
    if not has_lat:
        return "no_letters"
    tokens = re.findall(r"[a-z]+", text.lower())
    hits = sum(t in BANGLISH_WORDS for t in tokens)
    return "banglish" if hits >= 2 else "english"


def norm_key(text):
    """Aggressive key used only to detect duplicates (never used as model input)."""
    t = unicodedata.normalize("NFKC", text).lower()
    t = SUSPICIOUS_ZW_RE.sub("", t)
    t = re.sub(r"\d+", "0", t)
    t = re.sub(r"\s+", " ", t).strip()
    return t


def add_text_stats(df):
    df = df.copy()
    t = df["text"]
    df["n_messages"] = t.str.count(re.escape(config.DELIMITER)) + 1
    df["is_conversation"] = df["n_messages"] > 1
    df["has_speaker_marker"] = t.map(lambda x: bool(SPEAKER_RE.search(x)))
    df["char_len"] = t.str.len()
    df["language"] = t.map(language_bucket)
    df["has_cyrillic"] = t.map(lambda x: bool(CYRILLIC_RE.search(x)))
    df["has_zero_width"] = t.map(lambda x: bool(SUSPICIOUS_ZW_RE.search(x)))
    df["has_url"] = t.str.contains(r"https?://|hxxps?://|\[\.\]", regex=True)
    df["has_emoji"] = t.map(lambda x: any(ord(c) >= 0x1F300 for c in x))
    df["norm_key"] = t.map(norm_key)
    return df


def near_duplicate_pairs(texts, threshold=0.85):
    vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5))
    X = vec.fit_transform(texts)
    sim = cosine_similarity(X)
    np.fill_diagonal(sim, 0)
    ii, jj = np.where(np.triu(sim) >= threshold)
    return [(int(i), int(j), float(sim[i, j])) for i, j in zip(ii, jj)]


def section(title):
    print("\n" + "=" * 70 + f"\n{title}\n" + "=" * 70)


def main():
    pd.set_option("display.width", 200)
    pd.set_option("display.max_colwidth", 110)

    sample = load_sample()
    test = load_test()
    template = load_submission_template()

    section("1. FILES, SHAPES, COLUMNS, DTYPES")
    print("sample:", sample.shape, "| test:", test.shape, "| submission template:", template.shape)
    print(sample.dtypes, "\n")
    print("missing/empty text -> sample:", (sample.text.str.strip() == "").sum(),
          "| test:", (test.text.str.strip() == "").sum())
    print("test ids unique:", test.id.is_unique,
          "| test ids == template ids (same order):", bool((test.id.values == template.id.values).all()))

    section("2. CLASS / ACTION / TECHNIQUE COUNTS (sample)")
    print(sample.label.map(LABEL_NAMES).value_counts(), "\n")
    print(sample.action.map(ACTION_NAMES).value_counts(), "\n")
    tech = sample.techniques.map(parse_techniques)
    print("technique frequency:")
    print(tech.explode().fillna("NONE").value_counts(), "\n")
    print("label x action:")
    print(pd.crosstab(sample.label, sample.action))
    print("\nlabel x number of techniques:")
    print(pd.crosstab(sample.label, tech.map(len)))

    s = add_text_stats(sample)
    t = add_text_stats(test)

    section("3. LANGUAGE MIX")
    print("sample:\n", s.language.value_counts(normalize=True).round(3), "\n")
    print("test:\n", t.language.value_counts(normalize=True).round(3))
    print("\nsample label x language:")
    print(pd.crosstab(s.label, s.language))

    section("4. SINGLE MESSAGES vs CONVERSATIONS")
    print("sample conversations:", int(s.is_conversation.sum()), "of", len(s))
    print("test conversations:", int(t.is_conversation.sum()), "of", len(t))
    print("\ntest n_messages distribution:")
    print(t.n_messages.value_counts().sort_index())
    print("\nsample label x is_conversation:")
    print(pd.crosstab(s.label, s.is_conversation))
    print("\nconversations with speaker markers -> sample:",
          int((s.is_conversation & s.has_speaker_marker).sum()),
          "| test:", int((t.is_conversation & t.has_speaker_marker).sum()))

    section("5. NOISE / OBFUSCATION INDICATORS")
    for name, d in (("sample", s), ("test", t)):
        print(f"{name}: cyrillic lookalikes={d.has_cyrillic.mean():.3%} | "
              f"zero-width chars={d.has_zero_width.mean():.3%} | "
              f"urls={d.has_url.mean():.3%} | emojis={d.has_emoji.mean():.3%}")
    print("\ntext length (chars):")
    print(pd.DataFrame({"sample": s.char_len.describe(), "test": t.char_len.describe()}).round(1))

    section("6. DUPLICATES")
    print("sample exact duplicates:", int(sample.text.duplicated().sum()))
    print("sample duplicates after normalisation:", int(s.norm_key.duplicated().sum()))
    print("test exact duplicates:", int(test.text.duplicated().sum()),
          f"({test.text.duplicated().mean():.2%})")
    print("test duplicates after normalisation:", int(t.norm_key.duplicated().sum()),
          f"({t.norm_key.duplicated().mean():.2%})")
    print("unique normalised test templates:", t.norm_key.nunique())

    in_test = set(t.norm_key)
    overlap = s.norm_key.isin(in_test).sum()
    print(f"sample rows whose normalised text also appears in test: {overlap} of {len(s)}")

    pairs = near_duplicate_pairs(sample.text.tolist(), threshold=0.85)
    print(f"\nnear-duplicate pairs inside sample (char TF-IDF cosine >= 0.85): {len(pairs)}")
    for i, j, sim in pairs[:15]:
        print(f"  {sim:.2f}  [{sample.id[i]}] {sample.text[i][:60]!r}  <->  [{sample.id[j]}] {sample.text[j][:60]!r}")

    section("7. EXAMPLES: ONE PER CLASS (shortest and longest)")
    for lab in sorted(sample.label.unique()):
        sub = sample[sample.label == lab]
        print(f"\n[{lab}] {LABEL_NAMES[lab]}")
        print("  short:", sub.loc[sub.text.str.len().idxmin(), "text"][:150])
        print("  long :", sub.loc[sub.text.str.len().idxmax(), "text"][:150])


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/features/tfidf_features.py
"""Word + char TF-IDF + structure feature block for Model 1.

Owned by Teammate B (T-010).
Vocabulary/IDF is fit on train text and, when `test_d` is supplied, test text (unsupervised; declared in notebook §11).
"""
import logging
import pickle
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix, hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MaxAbsScaler

from src import config
from src.preprocessing import preprocess

log = logging.getLogger(__name__)

# Token pattern that covers both ASCII and Bangla Unicode
BANGLA_TOKEN_PAT = r"(?u)[\w\u0980-\u09FF]+"


# ── Text-dict builder ─────────────────────────────────────────────────────────

def build_text_dict(df: pd.DataFrame) -> dict:
    """Build feature-ready text dict from a DataFrame with 'id' and 'text' columns.

    Always re-runs preprocessing on df['text'] so corrupted/modified text
    is handled correctly (critical for robustness evaluation in cross_validation).

    Returns dict with keys:
        text_norm, text_skel, text_raw, first_msg, last_msg, struct_feats
    """
    prep   = [preprocess(t) for t in df["text"]]
    norm   = [p["preserved"] for p in prep]
    skel   = [p["skel"] for p in prep]
    raw    = df["text"].fillna("").tolist()

    firsts, lasts = [], []
    for t in df["text"]:
        parts = str(t).split(config.DELIMITER)
        firsts.append(parts[0].strip())
        lasts.append(parts[-1].strip())

    struct = np.array(
        [
            [
                float(p["n_messages"]),
                float(p["n_noise_tokens"]),
                float(p["noise_only"]),
                float(p["had_lookalike"]),
                float(p["had_zero_width"]),
                float(p["had_emoji"]),
                len(str(r)) / 1000.0,          # raw_len normalised
            ]
            for p, r in zip(prep, raw)
        ],
        dtype=float,
    )

    return {
        "text_norm":    norm,
        "text_skel":    skel,
        "text_raw":     raw,
        "first_msg":    firsts,
        "last_msg":     lasts,
        "struct_feats": struct,
    }


# ── TfidfBlock ────────────────────────────────────────────────────────────────

class TfidfBlock:
    """Multi-view TF-IDF feature block.

    fit(train_d, test_d=None)  -> self
    transform(d)               -> csr_matrix
    fit_transform(train_d, ..) -> csr_matrix
    save(path) / load(path)    helpers
    """

    def __init__(self,
                 max_word_features: int = 200_000,
                 max_char_features: int = 200_000,
                 use_char_raw: bool = False,
                 use_msg_blocks: bool = True):
        self.max_word_features = max_word_features
        self.max_char_features = max_char_features
        self.use_char_raw = use_char_raw
        self.use_msg_blocks = use_msg_blocks

    @staticmethod
    def _coerce(seq) -> list:
        return [str(t) if t is not None else "" for t in (seq or [])]

    def fit(self, train_d: dict, test_d: Optional[dict] = None) -> "TfidfBlock":
        """Fit vectorizers. test_d texts are included for IDF only (unsupervised)."""

        def combine(key):
            vals = self._coerce(train_d.get(key, []))
            if test_d is not None:
                vals = vals + self._coerce(test_d.get(key, []))
            return vals

        norm  = combine("text_norm")
        skel  = combine("text_skel")
        first = combine("first_msg")
        last  = combine("last_msg")
        raw   = combine("text_raw")

        log.info("TfidfBlock: fitting on %d texts (train+test unsupervised)", len(norm))

        self.vw_ = TfidfVectorizer(
            analyzer="word", ngram_range=(1, 2), sublinear_tf=True,
            token_pattern=BANGLA_TOKEN_PAT, min_df=1,
            max_features=self.max_word_features,
        ).fit(norm)

        self.vc_ = TfidfVectorizer(
            analyzer="char_wb", ngram_range=(2, 5), sublinear_tf=True,
            min_df=1, max_features=self.max_char_features,
        ).fit(skel)

        if self.use_char_raw:
            self.vcr_ = TfidfVectorizer(
                analyzer="char", ngram_range=(1, 4), sublinear_tf=True,
                min_df=1, max_features=50_000,
            ).fit(raw)

        if self.use_msg_blocks:
            self.vf_ = TfidfVectorizer(
                analyzer="word", ngram_range=(1, 2), sublinear_tf=True,
                token_pattern=BANGLA_TOKEN_PAT, min_df=1, max_features=50_000,
            ).fit(first)
            self.vl_ = TfidfVectorizer(
                analyzer="word", ngram_range=(1, 2), sublinear_tf=True,
                token_pattern=BANGLA_TOKEN_PAT, min_df=1, max_features=50_000,
            ).fit(last)

        struct_train = train_d.get("struct_feats")
        self.scaler_ = MaxAbsScaler().fit(struct_train) if struct_train is not None else None

        return self

    def transform(self, d: dict) -> csr_matrix:
        """Transform a texts dict to a stacked sparse feature matrix."""
        norm  = self._coerce(d.get("text_norm", []))
        skel  = self._coerce(d.get("text_skel", []))
        parts = [self.vw_.transform(norm), self.vc_.transform(skel)]

        if self.use_char_raw:
            raw = self._coerce(d.get("text_raw", []))
            parts.append(self.vcr_.transform(raw))

        if self.use_msg_blocks:
            first = self._coerce(d.get("first_msg", []))
            last  = self._coerce(d.get("last_msg",  []))
            parts.append(self.vf_.transform(first))
            parts.append(self.vl_.transform(last))

        struct = d.get("struct_feats")
        if struct is not None and self.scaler_ is not None:
            parts.append(csr_matrix(self.scaler_.transform(struct)))

        return hstack(parts).tocsr()

    def fit_transform(self, train_d: dict,
                      test_d: Optional[dict] = None) -> csr_matrix:
        """Convenience: fit on train_d (+ test_d for IDF), then transform train_d."""
        self.fit(train_d, test_d)
        return self.transform(train_d)

    def save(self, path) -> None:
        """Pickle the fitted block."""
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        with open(path, "wb") as f:
            pickle.dump(self, f)
        log.info("TfidfBlock saved → %s", path)

    @classmethod
    def load(cls, path) -> "TfidfBlock":
        """Load a pickled TfidfBlock."""
        with open(path, "rb") as f:
            return pickle.load(f)


In [ ]:
%%writefile src/features/embedding_features.py
"""Dense text embedding extraction with length-sorted batching, caching, and similarity helpers.

Owned by Teammate C (T-023).

Public API
----------
embed(texts, model_name, batch, max_len, device, show_progress_bar, use_cache) -> np.ndarray
max_cosine_to_train(emb_query, emb_train, is_train) -> np.ndarray
"""
from __future__ import annotations

import hashlib
import logging
from pathlib import Path
from typing import Optional, Sequence

import numpy as np
import torch
from sentence_transformers import SentenceTransformer

from src import config

log = logging.getLogger(__name__)

DEFAULT_MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
CACHE_DIR = getattr(config, "CACHE_DIR", getattr(config, "OUTPUT_DIR", config.ROOT / "outputs") / "cache")


def _clean_model_tag(model_name: str) -> str:
    """Return a filesystem-safe tag for the model name."""
    clean = model_name.split("/")[-1]
    return clean.replace("-", "_").replace(".", "_")


def _compute_cache_key(texts: Sequence[str], model_name: str, max_len: int) -> str:
    """Compute deterministic MD5 hash for the combination of texts, model, and max_len."""
    hasher = hashlib.md5()
    hasher.update(model_name.encode("utf-8"))
    hasher.update(str(max_len).encode("utf-8"))
    hasher.update(str(len(texts)).encode("utf-8"))
    for t in texts:
        hasher.update(t.encode("utf-8", errors="replace"))
        hasher.update(b"\x00")
    return hasher.hexdigest()


def embed(
    texts: Sequence[str],
    model_name: str = DEFAULT_MODEL_NAME,
    batch: int = 64,
    max_len: int = 128,
    device: Optional[str] = None,
    show_progress_bar: bool = False,
    use_cache: bool = True,
    cache_dir: Optional[Path] = None,
) -> np.ndarray:
    """Extract L2-normalised dense sentence embeddings with length-sorted batching and disk caching.

    Parameters
    ----------
    texts : Sequence[str]
        Input strings. Empty strings or None are safely mapped to "".
    model_name : str
        HuggingFace model repository or local path.
    batch : int
        Batch size for inference.
    max_len : int
        Maximum sequence length for tokenization.
    device : Optional[str]
        Device to run inference on ('cuda', 'mps', 'cpu', or auto-detected).
    show_progress_bar : bool
        Whether to display SentenceTransformer progress bar.
    use_cache : bool
        If True, reads/writes cached embeddings from disk.
    cache_dir : Optional[Path]
        Directory for cached `.npy` files.

    Returns
    -------
    np.ndarray
        Array of shape `(len(texts), embedding_dim)`, dtype float32, L2-normalised.
    """
    if len(texts) == 0:
        return np.empty((0, 0), dtype=np.float32)

    # Sanitize texts (handle None / NaN / whitespace)
    cleaned_texts = ["" if t is None or (isinstance(t, float) and np.isnan(t)) else str(t) for t in texts]

    # Prepend 'query: ' for e5 family models per specification
    is_e5 = "e5" in model_name.lower()
    encode_inputs = [f"query: {t}" if is_e5 else t for t in cleaned_texts]

    # Check disk cache
    target_cache_dir = cache_dir or CACHE_DIR
    target_cache_dir.mkdir(parents=True, exist_ok=True)
    model_tag = _clean_model_tag(model_name)
    cache_key = _compute_cache_key(cleaned_texts, model_name, max_len)
    cache_file = target_cache_dir / f"emb_{model_tag}_{cache_key}.npy"

    if use_cache and cache_file.exists():
        log.info("Loading cached embeddings from %s", cache_file)
        return np.load(cache_file)

    # Auto-detect device
    if device is None:
        if torch.cuda.is_available():
            device = "cuda"
        elif torch.backends.mps.is_available():
            device = "mps"
        else:
            device = "cpu"

    log.info("Encoding %d texts with %s on %s (batch=%d, max_len=%d)",
             len(encode_inputs), model_name, device, batch, max_len)

    model = SentenceTransformer(model_name, device=device)
    model.max_seq_length = max_len

    # Length-sorted batching for minimal padding overhead
    lengths = [len(t) for t in encode_inputs]
    sorted_idx = np.argsort(lengths)
    sorted_texts = [encode_inputs[i] for i in sorted_idx]

    # Use fp16 precision on CUDA for acceleration & memory efficiency
    use_fp16 = device == "cuda"
    if use_fp16:
        model.half()

    # Encode with dynamic padding
    encoded = model.encode(
        sorted_texts,
        batch_size=batch,
        show_progress_bar=show_progress_bar,
        convert_to_numpy=True,
        normalize_embeddings=False,  # We will explicitly L2-normalise below
        device=device,
    )

    # Restore original order
    orig_order_idx = np.empty_like(sorted_idx)
    orig_order_idx[sorted_idx] = np.arange(len(sorted_idx))
    embeddings = encoded[orig_order_idx].astype(np.float32)

    # L2-normalisation
    norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
    norms[norms == 0.0] = 1.0
    embeddings = embeddings / norms

    # Save to cache
    if use_cache:
        log.info("Saving embeddings to cache %s", cache_file)
        np.save(cache_file, embeddings)

    return embeddings


def max_cosine_to_train(
    emb_query: np.ndarray,
    emb_train: np.ndarray,
    is_train: bool = False,
    chunk_size: int = 25_000,
) -> np.ndarray:
    """Compute maximum cosine similarity of query embeddings against training embeddings.

    For training rows (is_train=True or when query is train), cross-fitting is applied
    by masking the self-similarity diagonal so a row is never compared to itself.

    Parameters
    ----------
    emb_query : np.ndarray
        Query embeddings of shape (n_query, d), assumed L2-normalised.
    emb_train : np.ndarray
        Training embeddings of shape (n_train, d), assumed L2-normalised.
    is_train : bool
        If True, masks self-similarity (diagonal) with -2.0 so query rows compare
        only against other training rows.
    chunk_size : int
        Chunk size for query matrix multiplications to bound memory on large sets (e.g. 500k).

    Returns
    -------
    np.ndarray
        Array of shape (n_query,) containing the maximum cosine similarity to train.
    """
    n_query = len(emb_query)
    n_train = len(emb_train)
    if n_query == 0 or n_train == 0:
        return np.zeros(n_query, dtype=np.float32)

    # Detect if query and train are the exact same dataset
    if not is_train and (emb_query is emb_train or (n_query == n_train and np.array_equal(emb_query, emb_train))):
        is_train = True

    out = np.empty(n_query, dtype=np.float32)

    # Process in chunks to bound memory
    for start in range(0, n_query, chunk_size):
        end = min(start + chunk_size, n_query)
        q_chunk = emb_query[start:end]
        # Dot product of L2-normalised vectors equals cosine similarity
        sim_chunk = np.dot(q_chunk, emb_train.T)

        if is_train:
            # Mask diagonal in this chunk
            for row_idx in range(start, end):
                if row_idx < n_train:
                    sim_chunk[row_idx - start, row_idx] = -2.0

        out[start:end] = sim_chunk.max(axis=1)

    return out


In [ ]:
%%writefile src/models/task_a.py
"""Task A: 6-class multinomial scam label classifier.

Owned by Teammate B (T-010).

Contract: fit(X, y, sample_weight=None) -> self
          predict_proba(X) -> (n, 6)
          predict(X) -> (n,)
"""
import logging

import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC

from src import config

log = logging.getLogger(__name__)


class TaskAHead:
    """6-class multinomial head for Task A (scam label classification).

    model_type='lr'  → LogisticRegression (multinomial, lbfgs)
    model_type='svm' → LinearSVC + softmax over scaled decision values
    """

    def __init__(self, C: float = 5.0,
                 model_type: str = "lr",
                 seed: int = config.SEED):
        self.C = C
        self.model_type = model_type
        self.seed = seed

    def fit(self, X, y, sample_weight=None) -> "TaskAHead":
        """Fit on sparse matrix X and label array y (integers 0–5)."""
        if self.model_type == "svm":
            self.clf_ = LinearSVC(
                C=self.C, class_weight="balanced",
                max_iter=5000, random_state=self.seed,
            )
        else:
            self.clf_ = LogisticRegression(
                C=self.C, max_iter=3000, class_weight="balanced",
                solver="lbfgs", random_state=self.seed,
            )
        self.clf_.fit(X, y, sample_weight=sample_weight)
        return self

    def predict_proba(self, X) -> np.ndarray:
        """Return (n, 6) class probabilities (zero-filled for unseen classes)."""
        n = X.shape[0]
        proba = np.zeros((n, config.NUM_CLASSES), dtype=float)
        if self.model_type == "svm":
            decision = self.clf_.decision_function(X)
            if decision.ndim == 1:                            # binary edge case
                decision = np.column_stack([-decision, decision])
            decision -= decision.max(axis=1, keepdims=True)  # numerical stability
            exp_d = np.exp(decision)
            soft = exp_d / exp_d.sum(axis=1, keepdims=True)
            proba[:, self.clf_.classes_] = soft
        else:
            proba[:, self.clf_.classes_] = self.clf_.predict_proba(X)
        return proba

    def predict(self, X) -> np.ndarray:
        """Return (n,) argmax class labels."""
        return self.predict_proba(X).argmax(axis=1)


In [ ]:
%%writefile src/models/task_b.py
"""Task B: 6 independent binary technique classifiers.

Owned by Teammate B (T-010).
Each technique T1–T6 is treated as an independent binary problem.
Per-technique thresholds are tuned in T-024.

Contract: fit(X, Y, sample_weight=None) -> self
          predict_proba(X) -> (n, 6)  independent positive-class probabilities
          predict(X) -> (n, 6)  binary 0/1 using per-technique thresholds
          set_thresholds(list[6 floats]) -> None
"""
import logging

import numpy as np
from sklearn.linear_model import LogisticRegression

from src import config

log = logging.getLogger(__name__)

_DEFAULT_THRESHOLD = 0.5


class TaskBHead:
    """Six independent binary LRs for technique prediction (Task B)."""

    def __init__(self, C: float = 5.0, seed: int = config.SEED):
        self.C = C
        self.seed = seed
        self.thresholds = [_DEFAULT_THRESHOLD] * config.NUM_CLASSES

    def fit(self, X, Y, sample_weight=None) -> "TaskBHead":
        """Fit 6 binary heads. Y: (n, 6) int binary matrix (techniques_to_multihot)."""
        Y = np.asarray(Y, dtype=int)
        self.clfs_   = []
        self.priors_ = []
        for j in range(config.NUM_CLASSES):
            y_j = Y[:, j]
            pos = int(y_j.sum())
            if pos < 2:
                # Not enough positives: use a constant prior and log a warning
                prior = float(pos) / max(len(y_j), 1)
                self.clfs_.append(None)
                self.priors_.append(prior)
                log.warning(
                    "Technique T%d: only %d positive(s) in this fold; "
                    "using constant prior %.3f",
                    j + 1, pos, prior,
                )
            else:
                clf = LogisticRegression(
                    C=self.C, max_iter=3000, class_weight="balanced",
                    solver="lbfgs", random_state=self.seed,
                )
                clf.fit(X, y_j, sample_weight=sample_weight)
                self.clfs_.append(clf)
                self.priors_.append(None)
        return self

    def predict_proba(self, X) -> np.ndarray:
        """Return (n, 6) positive-class probabilities."""
        n = X.shape[0]
        out = np.zeros((n, config.NUM_CLASSES), dtype=float)
        for j, (clf, prior) in enumerate(zip(self.clfs_, self.priors_)):
            if clf is None:
                out[:, j] = prior
            else:
                out[:, j] = clf.predict_proba(X)[:, 1]
        return out

    def predict(self, X) -> np.ndarray:
        """Return (n, 6) binary predictions using per-technique thresholds."""
        proba = self.predict_proba(X)
        thr   = np.array(self.thresholds, dtype=float)
        return (proba >= thr).astype(int)

    def set_thresholds(self, thresholds) -> None:
        """Set per-technique classification thresholds (list of 6 floats in [0,1])."""
        if len(thresholds) != config.NUM_CLASSES:
            raise ValueError(f"Expected {config.NUM_CLASSES} thresholds, got {len(thresholds)}")
        self.thresholds = [float(t) for t in thresholds]


In [ ]:
%%writefile src/models/task_c.py
"""Task C: 6-class intended action classifier.

Owned by Teammate B (T-010).
Optionally uses Task A OOF probabilities as extra features (cross-fitted).

Contract: fit(X, y, sample_weight=None, task_a_proba=None) -> self
          predict_proba(X, task_a_proba=None) -> (n, 6)
          predict(X, task_a_proba=None) -> (n,)
"""
import logging

import numpy as np
from sklearn.linear_model import LogisticRegression

from src import config

log = logging.getLogger(__name__)


class TaskCHead:
    """6-class multinomial head for Task C (intended action)."""

    def __init__(self, C: float = 5.0,
                 seed: int = config.SEED,
                 use_task_a_proba: bool = False):
        self.C = C
        self.seed = seed
        self.use_task_a_proba = use_task_a_proba

    @staticmethod
    def _augment_X(X, task_a_proba):
        """Optionally append Task A probabilities as extra columns."""
        if task_a_proba is not None:
            from scipy.sparse import csr_matrix, hstack
            return hstack([X, csr_matrix(np.asarray(task_a_proba, dtype=float))]).tocsr()
        return X

    def fit(self, X, y, sample_weight=None, task_a_proba=None) -> "TaskCHead":
        """Fit multinomial LR. Optionally append task_a_proba as extra features."""
        X_in = self._augment_X(X, task_a_proba) if self.use_task_a_proba else X
        self.clf_ = LogisticRegression(
            C=self.C, max_iter=3000, class_weight="balanced",
            solver="lbfgs", random_state=self.seed,
        )
        self.clf_.fit(X_in, y, sample_weight=sample_weight)
        return self

    def predict_proba(self, X, task_a_proba=None) -> np.ndarray:
        """Return (n, 6) action probabilities."""
        X_in = self._augment_X(X, task_a_proba) if self.use_task_a_proba else X
        n    = X_in.shape[0]
        proba = np.zeros((n, config.NUM_ACTIONS), dtype=float)
        proba[:, self.clf_.classes_] = self.clf_.predict_proba(X_in)
        return proba

    def predict(self, X, task_a_proba=None) -> np.ndarray:
        """Return (n,) argmax action labels."""
        return self.predict_proba(X, task_a_proba).argmax(axis=1)


In [ ]:
%%writefile src/models/ensemble.py
"""Ensemble blending of probability arrays from multiple models.

Owned by Teammate B (T-030).

Contract:
  blend(prob_list, weights=None) -> np.ndarray (n, k)
  tune_blend_weight(p1, p2, y_true, metric_fn, grid) -> float
  model_disagreement(p1, p2) -> np.ndarray (n,)
"""
import logging

import numpy as np

log = logging.getLogger(__name__)


def blend(prob_list: list, weights=None) -> np.ndarray:
    """Weighted average of class-probability arrays.

    prob_list: list of (n, k) numpy arrays — one per model
    weights:   list of non-negative floats (normalised internally); default = equal
    Returns:   (n, k) numpy array
    """
    if not prob_list:
        raise ValueError("prob_list must not be empty")

    if weights is None:
        weights = [1.0 / len(prob_list)] * len(prob_list)

    weights = np.asarray(weights, dtype=float)
    if weights.sum() == 0:
        raise ValueError("All weights are zero")
    weights = weights / weights.sum()

    result = sum(w * np.asarray(p, dtype=float) for w, p in zip(weights, prob_list))
    log.debug("Ensemble blend: %d models, weights=%s", len(prob_list), weights.round(3))
    return result


def tune_blend_weight(p1: np.ndarray,
                      p2: np.ndarray,
                      y_true: np.ndarray,
                      metric_fn,
                      grid=(0.3, 0.4, 0.5, 0.6, 0.7),
                      is_multilabel: bool = False) -> tuple:
    """Pick mixing weight w in grid for w*p1 + (1-w)*p2 maximising metric_fn on OOF.

    Returns (best_w, best_score, blended_probs).
    """
    scores = {}
    for w in grid:
        w = float(w)
        p = blend([p1, p2], weights=[w, 1.0 - w])
        pred = (p >= 0.5).astype(int) if is_multilabel else p.argmax(axis=1)
        scores[w] = float(metric_fn(y_true, pred))

    raw_best_score = max(scores.values())
    raw_best_w = min((w for w, s in scores.items() if s == raw_best_score),
                     key=lambda w: abs(w - 0.5))
    best_w = raw_best_w
    if 0.5 in scores and raw_best_score - scores[0.5] <= 0.02:
        best_w = 0.5
    best_score = scores[best_w]
    best_p = blend([p1, p2], weights=[best_w, 1.0 - best_w])
    log.info("Ensemble tune: best_w=%.2f (1-w=%.2f) -> score=%.4f",
             best_w, 1.0 - best_w, best_score)
    if best_w != raw_best_w:
        log.info("Ensemble tune: preferred equal weights (raw best w=%.2f, delta=%.4f <= 0.02)",
                 raw_best_w, raw_best_score - best_score)
    return best_w, best_score, best_p


def model_disagreement(p1: np.ndarray, p2: np.ndarray) -> np.ndarray:
    """Compute per-row L1 probability disagreement (total variation distance) in [0, 1]."""
    p1 = np.asarray(p1, dtype=float)
    p2 = np.asarray(p2, dtype=float)
    return 0.5 * np.sum(np.abs(p1 - p2), axis=1)


In [ ]:
%%writefile src/postprocess.py
"""Consistency post-processing rules between Task A, B, and C outputs.

Owned by Teammate B (T-024).

Rules derived from sample.csv cross-tab (86/86 real rows, 100% support):
  1. label in {0, 5} => techniques = NONE (all zeros) [44/44 = 100%]
  2. label in {0, 5} => action = 0 [44/44 = 100%]
  3. label in {1, 2, 3, 4} => at least one technique [42/42 = 100%];
     if none above threshold, emit argmax(probs_b).
  4. label in {1, 2, 3, 4} => action != 0 [42/42 = 100%];
     if predicted action == 0, emit 1 + argmax(probs_c[1:]).

Rules never key off text strings or test ids.
"""
import logging

import numpy as np

log = logging.getLogger(__name__)

_NON_SCAM_LABELS = {0, 5}
_SCAM_LABELS = {1, 2, 3, 4}


def apply_rules(label: int, tech, action: int, probs: dict) -> tuple:
    """Apply consistency rules to a single prediction row.

    label:  predicted Task A label (0..5)
    tech:   (6,) binary technique array
    action: predicted Task C action (0..5)
    probs:  dict with optional keys 'B' (6,) and 'C' (6,)

    Returns: (label, tech_array, action)
    """
    label = int(label)
    tech = np.array(tech, dtype=int, copy=True)
    action = int(action)

    if label in _NON_SCAM_LABELS:
        tech[:] = 0
        action = 0
        return label, tech, action

    if label in _SCAM_LABELS:
        if tech.sum() == 0 and probs is not None and "B" in probs:
            pb = np.asarray(probs["B"], dtype=float)
            tech[int(np.argmax(pb))] = 1
        if action == 0 and probs is not None and "C" in probs:
            pc = np.asarray(probs["C"], dtype=float)
            action = 1 + int(np.argmax(pc[1:]))

    return label, tech, action


def apply_rules_batch(labels: np.ndarray,
                      techs: np.ndarray,
                      actions: np.ndarray,
                      probs_b: np.ndarray = None,
                      probs_c: np.ndarray = None) -> tuple:
    """Vectorised consistency post-processing for batch predictions.

    Returns (labels_out, techs_out, actions_out) and logs rule change counts.
    """
    labels = np.asarray(labels, dtype=int).copy()
    techs = np.asarray(techs, dtype=int).copy()
    actions = np.asarray(actions, dtype=int).copy()

    non_scam = np.isin(labels, list(_NON_SCAM_LABELS))
    scam = ~non_scam

    # Rule 1 & 2: label in {0, 5} -> NONE and action 0
    tech_zeroed = int((techs[non_scam].sum(axis=1) > 0).sum())
    act_zeroed = int((actions[non_scam] != 0).sum())
    techs[non_scam] = 0
    actions[non_scam] = 0

    # Rule 3: label in {1..4} -> at least one technique
    tech_added = 0
    if probs_b is not None:
        no_tech = scam & (techs.sum(axis=1) == 0)
        tech_added = int(no_tech.sum())
        if tech_added > 0:
            best_t = np.argmax(probs_b[no_tech], axis=1)
            techs[no_tech, best_t] = 1

    # Rule 4: label in {1..4} -> action != 0
    act_fixed = 0
    if probs_c is not None:
        zero_act = scam & (actions == 0)
        act_fixed = int(zero_act.sum())
        if act_fixed > 0:
            actions[zero_act] = 1 + np.argmax(probs_c[zero_act, 1:], axis=1)

    log.info(
        "postprocess: tech_zeroed=%d, act_zeroed=%d, tech_added=%d, act_fixed=%d",
        tech_zeroed, act_zeroed, tech_added, act_fixed,
    )
    return labels, techs, actions


def tune_task_b_thresholds(oof_proba_b: np.ndarray,
                           Y_true: np.ndarray,
                           folds: np.ndarray,
                           grid=None) -> list:
    """Cross-fitted per-technique threshold tuning on OOF probabilities.

    Chooses threshold in grid (default 0.05..0.95) maximising F1 on 4 folds,
    evaluates on 5th, returns mean threshold across folds per technique.
    Shrinks toward 0.5 when a technique has < 5 positives.
    """
    from sklearn.metrics import f1_score

    if grid is None:
        grid = np.round(np.arange(0.05, 0.96, 0.05), 2)

    oof_proba_b = np.asarray(oof_proba_b, dtype=float)
    Y_true = np.asarray(Y_true, dtype=int)
    folds = np.asarray(folds)
    unique_folds = np.unique(folds)
    n_tech = Y_true.shape[1]

    final_thresholds = []
    for j in range(n_tech):
        pos_total = int(Y_true[:, j].sum())
        if pos_total < 2:
            final_thresholds.append(0.5)
            continue

        fold_best = []
        for f in unique_folds:
            tr = folds != f
            y_tr = Y_true[tr, j]
            p_tr = oof_proba_b[tr, j]
            best_t, best_f1 = 0.5, -1.0
            for t in grid:
                score = f1_score(y_tr, (p_tr >= t).astype(int), zero_division=0)
                if score > best_f1:
                    best_f1 = score
                    best_t = float(t)
            fold_best.append(best_t)

        thr = float(np.mean(fold_best))
        if pos_total < 5:
            # Shrink toward 0.5 for very rare techniques
            alpha = pos_total / 5.0
            thr = alpha * thr + (1.0 - alpha) * 0.5
        final_thresholds.append(round(thr, 3))

    log.info("Tuned Task B thresholds: %s", final_thresholds)
    return final_thresholds


In [ ]:
%%writefile src/evaluation/calibration.py
"""Confidence calibration model for Task D.

Owned by Teammate B (T-031).

Target:  confidence = P(Task A prediction is correct), fit on OOF.
Metric:  Cal = 1 - Brier(confidence, correctness).
Gate:    Must beat constant-confidence baseline (OOF accuracy) in Brier;
         otherwise fall back to constant baseline and log a warning.

Public API (ARCHITECTURE.md §4):
  fit_confidence(oof_probs, y_true, extra_feats=None, folds=None, ...) -> dict
  apply(calibrator, probs, extra_feats=None, noise_only=None) -> np.ndarray
"""
import logging

import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

from src import config

log = logging.getLogger(__name__)


def extract_cal_features(probs: np.ndarray, extra_feats: np.ndarray = None) -> np.ndarray:
    """Extract calibration features from class probabilities.

    Features: [max_prob, top2_margin, norm_entropy] (+ optional extra_feats).
    """
    probs = np.asarray(probs, dtype=float)
    probs = np.clip(probs, 1e-12, 1.0)
    probs = probs / probs.sum(axis=1, keepdims=True)

    sorted_p = np.sort(probs, axis=1)
    max_p = sorted_p[:, -1]
    margin = sorted_p[:, -1] - sorted_p[:, -2]
    entropy = -np.sum(probs * np.log(probs), axis=1) / np.log(probs.shape[1])

    feats = [max_p[:, None], margin[:, None], entropy[:, None]]
    if extra_feats is not None:
        ef = np.asarray(extra_feats, dtype=float)
        if ef.ndim == 1:
            ef = ef[:, None]
        feats.append(ef)
    return np.hstack(feats)


def _brier(conf: np.ndarray, correct: np.ndarray) -> float:
    return float(np.mean((conf - correct) ** 2))


def fit_confidence(oof_probs: np.ndarray,
                   y_true: np.ndarray,
                   extra_feats: np.ndarray = None,
                   folds: np.ndarray = None,
                   noise_only: np.ndarray = None,
                   C_grid=(0.05, 0.1, 0.5, 1.0)) -> dict:
    """Fit logistic calibrator on OOF predictions, cross-fitted if folds provided.

    Compares cross-fitted logistic calibrator against constant-confidence baseline
    (mean OOF accuracy). If logistic calibrator does not improve Brier, returns
    constant-baseline calibrator and logs a warning.
    """
    oof_probs = np.asarray(oof_probs, dtype=float)
    y_true = np.asarray(y_true, dtype=int)
    correct = (oof_probs.argmax(axis=1) == y_true).astype(int)
    base_acc = float(np.mean(correct))
    const_brier = _brier(np.full_like(correct, base_acc, dtype=float), correct)

    X = extract_cal_features(oof_probs, extra_feats)

    if noise_only is not None and np.any(noise_only):
        noise_acc = float(np.mean(correct[np.asarray(noise_only, dtype=bool)]))
    else:
        noise_acc = min(base_acc, 0.5)

    best_C, best_brier, best_oof_conf = None, const_brier, None

    if folds is not None and len(np.unique(folds)) > 1:
        folds = np.asarray(folds)
        unique_folds = np.unique(folds)
        for C in C_grid:
            oof_conf = np.empty(len(y_true), dtype=float)
            ok = True
            for f in unique_folds:
                tr = folds != f
                va = folds == f
                if len(np.unique(correct[tr])) < 2:
                    ok = False
                    break
                scaler = StandardScaler().fit(X[tr])
                clf = LogisticRegression(C=C, max_iter=1000, random_state=config.SEED)
                clf.fit(scaler.transform(X[tr]), correct[tr])
                oof_conf[va] = clf.predict_proba(scaler.transform(X[va]))[:, 1]
            if not ok:
                continue
            oof_conf = np.clip(oof_conf, *config.CONF_CLIP)
            b = _brier(oof_conf, correct)
            if b < best_brier:
                best_brier = b
                best_C = C
                best_oof_conf = oof_conf
    else:
        # No folds provided: fit with strongest regularisation
        if len(np.unique(correct)) >= 2:
            C = C_grid[0]
            scaler = StandardScaler().fit(X)
            clf = LogisticRegression(C=C, max_iter=1000, random_state=config.SEED)
            clf.fit(scaler.transform(X), correct)
            oof_conf = np.clip(clf.predict_proba(scaler.transform(X))[:, 1], *config.CONF_CLIP)
            b = _brier(oof_conf, correct)
            if b < best_brier:
                best_brier = b
                best_C = C
                best_oof_conf = oof_conf

    if best_C is None:
        log.warning(
            "Calibration: logistic calibrator did not beat constant baseline "
            "(Brier=%.4f, Cal=%.4f); using constant baseline.",
            const_brier, 1.0 - const_brier,
        )
        return {
            "mode": "constant",
            "base_acc": float(np.clip(base_acc, *config.CONF_CLIP)),
            "noise_acc": float(np.clip(noise_acc, *config.CONF_CLIP)),
            "const_brier": const_brier,
            "cal_brier": const_brier,
            "oof_conf": np.full(len(y_true), np.clip(base_acc, *config.CONF_CLIP)),
        }

    # Fit final scaler + logistic model on all OOF rows
    final_scaler = StandardScaler().fit(X)
    final_clf = LogisticRegression(C=best_C, max_iter=1000, random_state=config.SEED)
    final_clf.fit(final_scaler.transform(X), correct)

    log.info(
        "Calibration: logistic C=%.3f beat constant baseline | "
        "Brier %.4f -> %.4f (Cal %.4f -> %.4f)",
        best_C, const_brier, best_brier, 1.0 - const_brier, 1.0 - best_brier,
    )
    return {
        "mode": "logistic",
        "C": best_C,
        "scaler": final_scaler,
        "clf": final_clf,
        "base_acc": float(np.clip(base_acc, *config.CONF_CLIP)),
        "noise_acc": float(np.clip(noise_acc, *config.CONF_CLIP)),
        "const_brier": const_brier,
        "cal_brier": best_brier,
        "oof_conf": best_oof_conf,
    }


def apply(calibrator: dict,
          probs: np.ndarray,
          extra_feats: np.ndarray = None,
          noise_only: np.ndarray = None) -> np.ndarray:
    """Apply fitted calibrator. Returns confidence array clipped to config.CONF_CLIP."""
    n = len(probs)
    if calibrator["mode"] == "constant":
        conf = np.full(n, calibrator["base_acc"], dtype=float)
    else:
        X = extract_cal_features(probs, extra_feats)
        X_s = calibrator["scaler"].transform(X)
        conf = calibrator["clf"].predict_proba(X_s)[:, 1]

    if noise_only is not None:
        mask = np.asarray(noise_only, dtype=bool)
        if np.any(mask):
            # Shrink noise-only confidence toward noise_acc
            conf[mask] = 0.5 * conf[mask] + 0.5 * calibrator["noise_acc"]

    return np.clip(conf, *config.CONF_CLIP)


In [ ]:
%%writefile src/evaluation/metrics.py
"""Evaluation metrics: macro-F1 A/B/C, Brier calibration, composite surrogate S_hat.

Owned by Teammate B (T-007).

Public API
----------
macro_f1_multiclass(y_true, y_pred, labels) -> float
macro_f1_techniques(Y_true, Y_pred) -> float
brier_calibration(conf, correct) -> float
s_hat(F_A, F_B, F_C, F_domain, F_robust, Cal, F_novel) -> float
score_all(...) -> dict
log_experiment(row) -> None
"""
import csv
import logging
import subprocess
from datetime import datetime

import numpy as np
from sklearn.metrics import f1_score

from src import config

log = logging.getLogger(__name__)


# ── Core metrics ──────────────────────────────────────────────────────────────

def macro_f1_multiclass(y_true, y_pred, labels=range(6)) -> float:
    """Macro-F1 for Task A or C. Classes absent from BOTH y_true and y_pred are skipped.

    Uses zero_division=0 so absent classes contribute 0 F1 (not error).
    """
    y_true = list(y_true)
    y_pred = list(y_pred)
    present = [l for l in list(labels) if l in y_true or l in y_pred]
    if not present:
        return 0.0
    return float(f1_score(y_true, y_pred, labels=present, average="macro", zero_division=0))


def macro_f1_techniques(Y_true: np.ndarray, Y_pred: np.ndarray) -> float:
    """Macro-F1 for Task B. Both arrays are (n, 6) binary matrices.

    F1 per technique averaged over T1–T6, computed over ALL rows.
    Legit rows (all-zero in Y_true) are negatives for every technique.
    """
    Y_true = np.asarray(Y_true, dtype=int)
    Y_pred = np.asarray(Y_pred, dtype=int)
    scores = [
        float(f1_score(Y_true[:, j], Y_pred[:, j], zero_division=0))
        for j in range(Y_true.shape[1])
    ]
    return float(np.mean(scores))


def brier_calibration(conf, correct) -> float:
    """Calibration = 1 − Brier score. Higher is better (best = 1).

    conf:    confidence array (floats in [0, 1])
    correct: 1 when Task A prediction equals true label, else 0
    """
    conf = np.asarray(conf, dtype=float)
    correct = np.asarray(correct, dtype=float)
    return float(1.0 - np.mean((conf - correct) ** 2))


def s_hat(F_A: float, F_B: float, F_C: float,
          F_domain: float, F_robust: float,
          Cal: float, F_novel: float) -> float:
    """Composite surrogate score per PRD §3.

    S = 0.40·F_A + 0.15·F_B + 0.10·F_C + 0.10·F_domain
        + 0.10·F_robust + 0.05·Cal + 0.10·F_novel
    """
    return (0.40 * F_A + 0.15 * F_B + 0.10 * F_C +
            0.10 * F_domain + 0.10 * F_robust +
            0.05 * Cal + 0.10 * F_novel)


# ── Aggregate scorer ──────────────────────────────────────────────────────────

def score_all(y_true_A, y_pred_A,
              Y_true_B, Y_pred_B,
              y_true_C, y_pred_C,
              conf,
              slice_masks: dict = None,
              F_domain: float = None,
              F_robust: float = None,
              F_novel: float = None) -> dict:
    """Compute all metrics. Returns dict with F_A, F_B, F_C, Cal, S_hat, per-slice F_A.

    slice_masks: dict[name -> bool array of length n]
    F_domain, F_robust, F_novel: proxy values; default to F_A when None.
    """
    y_true_A = np.asarray(y_true_A)
    y_pred_A = np.asarray(y_pred_A)
    Y_true_B = np.asarray(Y_true_B, dtype=int)
    Y_pred_B = np.asarray(Y_pred_B, dtype=int)
    y_true_C = np.asarray(y_true_C)
    y_pred_C = np.asarray(y_pred_C)
    correct = (y_pred_A == y_true_A).astype(float)

    F_A = macro_f1_multiclass(y_true_A, y_pred_A)
    F_B = macro_f1_techniques(Y_true_B, Y_pred_B)
    F_C = macro_f1_multiclass(y_true_C, y_pred_C)
    Cal = brier_calibration(conf, correct)

    _domain = F_domain if F_domain is not None else F_A
    _robust = F_robust if F_robust is not None else F_A
    _novel  = F_novel  if F_novel  is not None else _domain
    S = s_hat(F_A, F_B, F_C, _domain, _robust, Cal, _novel)

    result = {"F_A": F_A, "F_B": F_B, "F_C": F_C, "Cal": Cal, "S_hat": S}

    if slice_masks:
        for name, mask in slice_masks.items():
            mask = np.asarray(mask, dtype=bool)
            if mask.sum() > 0:
                result[f"F_A_{name}"] = macro_f1_multiclass(
                    y_true_A[mask], y_pred_A[mask])
            else:
                log.warning("Slice '%s' has 0 rows; skipping.", name)
                result[f"F_A_{name}"] = float("nan")

    return result


# ── Experiment logger ─────────────────────────────────────────────────────────

def log_experiment(row: dict) -> None:
    """Append one experiment row to outputs/validation_results/experiments.csv."""
    path = config.VALIDATION_DIR / "experiments.csv"
    row = dict(row)
    row.setdefault("timestamp", datetime.utcnow().isoformat())
    row.setdefault("git_hash", _git_hash())

    base_fields = [
        "exp_id", "timestamp", "git_hash", "config_json",
        "F_A", "F_B", "F_C", "Cal", "S_hat",
        "F_domain", "F_robust", "notes",
    ]
    all_fields = base_fields + [k for k in row if k not in base_fields]

    write_header = not path.exists()
    with open(path, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=all_fields, extrasaction="ignore")
        if write_header:
            writer.writeheader()
        writer.writerow(row)
    log.info("Experiment logged: %s", row.get("exp_id", "?"))


def _git_hash() -> str:
    try:
        return subprocess.check_output(
            ["git", "rev-parse", "--short", "HEAD"],
            stderr=subprocess.DEVNULL,
        ).decode().strip()
    except Exception:
        return "unknown"


In [ ]:
%%writefile src/training/cross_validation.py
"""Stage 3: leakage-safe unified folds, leave-cluster-out proxy, OOF saving, and slice scoring.

Run from the project root:
    python -m src.training.cross_validation
"""
import logging
import random
import re

import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components
from sklearn.cluster import AgglomerativeClustering
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.neighbors import NearestNeighbors

from src import config
from src.augment import corrupt as augment_corrupt
from src.data_loader import techniques_to_multihot
from src.preprocessing import preprocess, template_key
from src.text_analysis import language_bucket

logger = logging.getLogger(__name__)
FOLDS_PATH = config.VALIDATION_DIR / "folds.csv"
_HARD_LEGIT_RE = re.compile(r"\b(?:otp|link|parcel|cashback|urgent|bill)\b", re.I)


def corrupt_text(text, rng, strength=0.06, ops="B"):
    """Evaluation corruption using held-out Set B operators, preserving `|||`."""
    level = "light" if strength <= 0.045 else ("heavy" if strength >= 0.10 else "medium")
    return augment_corrupt(text, rng, level=level, ops=ops)


def assign_groups(df, sim_threshold=0.80):
    """Group exact templates, shared first messages (>=20 chars), and sparse cosine near-duplicates."""
    n = len(df)
    if n == 0:
        return pd.Series(dtype=np.int64, index=df.index, name="group")

    if "clean" in df.columns:
        cleans = df["clean"].fillna("").astype(str).tolist()
    else:
        cleans = [preprocess(t)["clean"] for t in df["text"]]

    if "template_key" in df.columns:
        keys = df["template_key"].fillna("").astype(str).tolist()
    else:
        keys = [template_key(c) for c in cleans]

    rows, cols = [], []

    # 1) Exact template_key matches
    by_key = {}
    for idx, key in enumerate(keys):
        by_key.setdefault(key, []).append(idx)
    for idxs in by_key.values():
        if len(idxs) > 1:
            base = idxs[0]
            for j in idxs[1:]:
                rows.extend((base, j))
                cols.extend((j, base))

    # 2) Shared first message (>= 20 chars after template_key)
    by_first = {}
    for idx, c in enumerate(cleans):
        first_msg = c.split(config.DELIMITER, 1)[0].strip()
        fkey = template_key(first_msg)
        if len(fkey) >= 20:
            by_first.setdefault(fkey, []).append(idx)
    for idxs in by_first.values():
        if len(idxs) > 1:
            base = idxs[0]
            for j in idxs[1:]:
                rows.extend((base, j))
                cols.extend((j, base))

    # 3) Sparse char n-gram near-duplicates
    nonempty = [i for i, c in enumerate(cleans) if c.strip()]
    if len(nonempty) > 1:
        vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5))
        X = vec.fit_transform([cleans[i] for i in nonempty])
        nn = NearestNeighbors(radius=max(0.0, 1.0 - sim_threshold), metric="cosine", algorithm="brute")
        nn.fit(X)
        graph = nn.radius_neighbors_graph(X, mode="connectivity").tocoo()
        for a, b in zip(graph.row, graph.col):
            if a < b:
                u, v = nonempty[int(a)], nonempty[int(b)]
                rows.extend((u, v))
                cols.extend((v, u))

    adj = csr_matrix((np.ones(len(rows), dtype=bool), (rows, cols)), shape=(n, n))
    _, labels = connected_components(adj, directed=False)
    return pd.Series(labels.astype(np.int64), index=df.index, name="group")


def assign_clusters(df, groups=None, n_clusters=3):
    """Per-label balanced template families using Ward linkage on L2-normalized char TF-IDF."""
    groups = np.asarray(assign_groups(df) if groups is None else groups)
    cleans = (
        df["clean"].fillna("").astype(str).tolist()
        if "clean" in df.columns
        else [preprocess(t)["clean"] for t in df["text"]]
    )
    out = np.zeros(len(df), dtype=np.int64)
    for lab in sorted(df["label"].unique()):
        idx = np.where(df["label"].values == lab)[0]
        ug = np.unique(groups[idx])
        k = min(n_clusters, len(ug))
        if k < 2:
            continue
        reps = [
            " ".join(cleans[i].strip() for i in idx if groups[i] == g and cleans[i].strip()) or "empty"
            for g in ug
        ]
        X = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5)).fit_transform(reps).toarray()
        lab_k = AgglomerativeClustering(n_clusters=k, metric="euclidean", linkage="ward").fit_predict(X)
        order = {c: r for r, c in enumerate(pd.Series(lab_k).value_counts().index)}  # 0 = largest family
        gmap = {g: order[c] for g, c in zip(ug, lab_k)}
        out[idx] = [gmap[groups[i]] for i in idx]
    return pd.Series(out, index=df.index, name="cluster")


def _resolve_groups(df, group_col="group", sim_threshold=0.80):
    if group_col in df.columns:
        return df[group_col]
    return assign_groups(df, sim_threshold=sim_threshold)


def make_folds(df, n_splits=config.N_SPLITS, seed=config.SEED, group_col="group"):
    """Unified StratifiedGroupKFold partition (stratified on Task A label, grouped by template)."""
    groups = _resolve_groups(df, group_col)
    grp_per_label = (
        pd.DataFrame({"label": df["label"].values, "group": np.asarray(groups)})
        .groupby("label")["group"]
        .nunique()
    )
    min_grp = int(grp_per_label.min())
    if min_grp < n_splits:
        new_splits = max(2, min_grp)
        logging.warning(
            "make_folds: smallest per-label group count (%d) < n_splits (%d); lowering to %d.",
            min_grp, n_splits, new_splits,
        )
        n_splits = new_splits
    sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    yield from sgkf.split(df, df["label"], groups)


def _folds_to_array(n, folds):
    arr = np.full(n, -1, dtype=np.int64)
    for fold, (_, va) in enumerate(folds):
        arr[va] = fold
    assert (arr >= 0).all(), "Some rows were not assigned a validation fold."
    return arr


def build_fold_table(
    df,
    n_splits=config.N_SPLITS,
    n_repeats=config.N_REPEATS,
    seed=config.SEED,
    sim_threshold=0.80,
    n_clusters=3,
):
    """Build one unified repeated fold table across Tasks A/B/C plus per-label cluster IDs."""
    work = df.reset_index(drop=True).copy()
    work["group"] = assign_groups(work, sim_threshold=sim_threshold).values
    work["cluster"] = assign_clusters(work, work["group"], n_clusters=n_clusters).values

    cols = ["id", "group", "cluster"]
    for rep in range(n_repeats):
        col = f"fold_r{rep}"
        work[col] = _folds_to_array(len(work), make_folds(work, n_splits, seed + rep, "group"))
        cols.append(col)
    return work[cols]


def save_folds(df, path=FOLDS_PATH, **kwargs):
    """Build and persist unified fold assignments to CSV for reproducibility (Decision D-008)."""
    table = build_fold_table(df, **kwargs)
    path.parent.mkdir(parents=True, exist_ok=True)
    table.to_csv(path, index=False)
    return table


def load_folds(path=FOLDS_PATH, df=None):
    """Load saved fold assignments and verify row/ID alignment if `df` is provided."""
    table = pd.read_csv(path)
    if df is not None:
        assert len(table) == len(df), f"Fold table length {len(table)} != df length {len(df)}"
        assert (table["id"].values == df["id"].values).all(), "Fold table IDs do not match df IDs."
    return table


def load_or_create_folds(
    df,
    path=FOLDS_PATH,
    n_splits=config.N_SPLITS,
    n_repeats=config.N_REPEATS,
    seed=config.SEED,
):
    """Read saved folds from disk (rebuilding with a warning only if missing or invalid)."""
    req = {"id", "group", "cluster"} | {f"fold_r{r}" for r in range(n_repeats)}
    if path.exists():
        table = pd.read_csv(path)
        if req.issubset(table.columns) and len(table) == len(df) and (table["id"].values == df["id"].values).all():
            return table
        logging.warning("load_or_create_folds: %s failed schema/ID check; rebuilding folds.", path)
    else:
        logging.warning("load_or_create_folds: %s missing; building folds.", path)
    return save_folds(df, path=path, n_splits=n_splits, n_repeats=n_repeats, seed=seed)


def iter_saved_folds(fold_table, col="fold_r0"):
    """Yield (train_idx, valid_idx) from a saved fold column."""
    vals = fold_table[col].values
    for f in sorted(np.unique(vals)):
        va = np.where(vals == f)[0]
        tr = np.where(vals != f)[0]
        yield tr, va


def get_folds(df, n_repeats=config.N_REPEATS):
    """Frozen contract: list of (train_idx, val_idx), ordered repeat then fold."""
    t = load_or_create_folds(df, n_repeats=n_repeats)
    return [p for r in range(n_repeats) for p in iter_saved_folds(t, f"fold_r{r}")]


def leave_cluster_out_folds(df, fold_table=None):
    """Yield (train_idx, valid_idx, cluster_id) holding out one template family per class."""
    if fold_table is not None and "cluster" in fold_table.columns:
        clusters = fold_table["cluster"].values
    elif "cluster" in df.columns:
        clusters = df["cluster"].values
    else:
        clusters = assign_clusters(df).values
    for cid in sorted(np.unique(clusters)):
        va = np.where(clusters == cid)[0]
        tr = np.where(clusters != cid)[0]
        if len(va) and len(tr):
            yield tr, va, int(cid)


def language_holdout_folds(df, language_col="language"):
    """Domain-shift check: train on two language buckets, validate on the third."""
    for lang in df[language_col].unique():
        va = np.where(df[language_col] == lang)[0]
        tr = np.where(df[language_col] != lang)[0]
        if len(va) and len(tr):
            yield tr, va, lang


def _f1(y, p, labels):
    return f1_score(y, p, labels=labels, average="macro", zero_division=0)


def score(valid, pred):
    y_label = valid["label"].values
    y_action = valid["action"].values
    y_tech = techniques_to_multihot(valid["techniques"])
    n_classes = getattr(config, "NUM_CLASSES", 6)
    n_actions = getattr(config, "NUM_ACTIONS", 6)
    n_tech = y_tech.shape[1]

    conf = np.asarray(pred["proba"]).max(axis=1)
    correct = (np.asarray(pred["label"]) == y_label).astype(float)
    brier = float(np.mean((conf - correct) ** 2))
    return {
        "F_A": _f1(y_label, pred["label"], list(range(n_classes))),
        "F_B": float(np.mean([
            f1_score(y_tech[:, j], np.asarray(pred["tech"])[:, j], zero_division=0)
            for j in range(n_tech)
        ])),
        "F_C": _f1(y_action, pred["action"], list(range(n_actions))),
        "brier": brier,
    }


def compute_slice_metrics(df, pred):
    """Evaluate Task A macro-F1 on language, conversation-structure, and hard-legit slices."""
    raw_texts = df["text"].fillna("").astype(str)
    langs = raw_texts.map(language_bucket).values
    n_msgs = (raw_texts.str.count(r"\|\|\|") + 1).values
    cleans = (
        df["clean"].fillna("").astype(str)
        if "clean" in df.columns
        else pd.Series([preprocess(t)["clean"] for t in raw_texts], index=df.index)
    )
    y = df["label"].values
    p = np.asarray(pred["label"])
    out = {}

    for lang in ("english", "banglish", "bangla_script", "mixed_bn_en"):
        mask = langs == lang
        if mask.any():
            present = sorted(np.unique(y[mask]).tolist())
            out[f"lang_{lang}"] = float(_f1(y[mask], p[mask], present))

    for name, mask in (("single", n_msgs == 1), ("conversation", n_msgs > 1)):
        if mask.any():
            present = sorted(np.unique(y[mask]).tolist())
            out[f"struct_{name}"] = float(_f1(y[mask], p[mask], present))

    hard_legit = np.isin(y, [0, 5]) & cleans.str.contains(_HARD_LEGIT_RE, na=False).values
    if hard_legit.any():
        present = sorted(np.unique(y[hard_legit]).tolist())
        out["slice_hard_legit"] = float(_f1(y[hard_legit], p[hard_legit], present))

    return out


def _init_oof(n, n_classes, n_tech, n_actions):
    """Allocate NaN/-1 sentinel OOF buffers so any unfilled row fails loudly."""
    return {
        "label": np.full(n, -1, dtype=np.int64),
        "proba": np.full((n, n_classes), np.nan, dtype=np.float64),
        "proba_b": np.full((n, n_tech), np.nan, dtype=np.float64),
        "tech": np.full((n, n_tech), -1, dtype=np.int64),
        "proba_c": np.full((n, n_actions), np.nan, dtype=np.float64),
        "action": np.full(n, -1, dtype=np.int64),
    }


def _assert_oof_complete(oof):
    assert (oof["label"] >= 0).all(), "Unfilled rows in OOF label."
    assert (oof["action"] >= 0).all(), "Unfilled rows in OOF action."
    assert (oof["tech"] >= 0).all(), "Unfilled rows in OOF tech."
    for k in ("proba", "proba_b", "proba_c"):
        assert not np.isnan(oof[k]).any(), f"NaN found in OOF {k}."


def save_oof_npz(
    df,
    oof_clean,
    oof_corrupted=None,
    folds_arr=None,
    n_repeats=config.N_REPEATS,
    exp_name="e001",
    suffix="m1",
):
    """Save repeat-averaged OOF probabilities to outputs/oof/<exp>_<suffix>.npz (D-008)."""
    out_path = config.OOF_DIR / f"{exp_name}_{suffix}.npz"
    out_path.parent.mkdir(parents=True, exist_ok=True)
    n = len(df)
    y_a = df["label"].to_numpy(dtype=np.int64)
    y_b = techniques_to_multihot(df["techniques"]).astype(np.int64)
    y_c = df["action"].to_numpy(dtype=np.int64)
    prep = [preprocess(t) for t in df["text"]]
    noise_only = np.array([p["noise_only"] for p in prep], dtype=bool)
    noise_ratio = np.array([p["noise_ratio"] for p in prep], dtype=np.float64)

    p_a = np.asarray(oof_clean["proba"], dtype=np.float64)
    p_b = np.asarray(oof_clean["proba_b"], dtype=np.float64)
    p_c = np.asarray(oof_clean["proba_c"], dtype=np.float64)

    payload = {
        "ids": df["id"].to_numpy(dtype=np.int64),
        "y_a": y_a,
        "y_b": y_b,
        "y_c": y_c,
        "noise_only": noise_only,
        "noise_ratio": noise_ratio,
        "repeat": np.full(n, int(n_repeats), dtype=np.int64),
        # Shared contract for ensemble, calibration, and m2
        "A": p_a,
        "B": p_b,
        "C": p_c,
        # Backwards-compatible aliases
        "clean_proba_a": p_a,
        "clean_proba_b": p_b,
        "clean_proba_c": p_c,
    }
    if folds_arr is not None:
        payload["folds"] = np.asarray(folds_arr, dtype=np.int64)
    if oof_corrupted is not None:
        payload["corrupted_proba_a"] = np.asarray(oof_corrupted["proba"], dtype=np.float64)
        payload["corrupted_proba_b"] = np.asarray(oof_corrupted["proba_b"], dtype=np.float64)
        payload["corrupted_proba_c"] = np.asarray(oof_corrupted["proba_c"], dtype=np.float64)
    np.savez_compressed(out_path, **payload)
    return out_path


def cross_validate(
    df,
    make_model,
    n_splits=config.N_SPLITS,
    n_repeats=config.N_REPEATS,
    corrupt_valid=True,
    use_saved_folds=True,
    exp_name=None,
    model_tag="m1",
    return_oof=False,
):
    """Repeated grouped CV reading saved folds.csv, fitting once per fold, averaging OOF probs."""
    work = df.reset_index(drop=True).copy()
    if use_saved_folds:
        saved_folds = load_or_create_folds(work, FOLDS_PATH, n_splits=n_splits, n_repeats=n_repeats)
    else:
        saved_folds = build_fold_table(work, n_splits=n_splits, n_repeats=n_repeats)
    work["group"] = saved_folds["group"].values
    work["cluster"] = saved_folds["cluster"].values

    n = len(work)
    n_classes = getattr(config, "NUM_CLASSES", 6)
    n_actions = getattr(config, "NUM_ACTIONS", 6)
    n_tech = techniques_to_multihot(work["techniques"]).shape[1]

    rows = []
    slice_rows = []

    sum_clean = {
        "proba": np.zeros((n, n_classes), dtype=np.float64),
        "proba_b": np.zeros((n, n_tech), dtype=np.float64),
        "proba_c": np.zeros((n, n_actions), dtype=np.float64),
    }
    sum_noisy = {
        "proba": np.zeros((n, n_classes), dtype=np.float64),
        "proba_b": np.zeros((n, n_tech), dtype=np.float64),
        "proba_c": np.zeros((n, n_actions), dtype=np.float64),
    }

    for rep in range(n_repeats):
        rng = random.Random(config.SEED + rep)
        fold_iter = list(iter_saved_folds(saved_folds, f"fold_r{rep}"))

        oof_clean = _init_oof(n, n_classes, n_tech, n_actions)
        oof_noisy = _init_oof(n, n_classes, n_tech, n_actions)

        for tr, va in fold_iter:
            train, valid = work.iloc[tr], work.iloc[va].copy()
            model = make_model()
            model.fit(train)

            pred = model.predict(valid)
            for k in oof_clean:
                if k in pred:
                    oof_clean[k][va] = pred[k]

            if corrupt_valid:
                noisy = valid.copy()
                noisy["text"] = noisy["text"].map(lambda x: corrupt_text(x, rng, ops="B"))
                pred_n = model.predict(noisy)
                for k in oof_noisy:
                    if k in pred_n:
                        oof_noisy[k][va] = pred_n[k]

        _assert_oof_complete(oof_clean)
        rows.append({"rep": rep, "view": "clean", **score(work, oof_clean)})
        slice_rows.append({"rep": rep, "view": "clean", **compute_slice_metrics(work, oof_clean)})
        for k in sum_clean:
            sum_clean[k] += oof_clean[k]

        if corrupt_valid:
            _assert_oof_complete(oof_noisy)
            rows.append({"rep": rep, "view": "corrupted", **score(work, oof_noisy)})
            slice_rows.append({"rep": rep, "view": "corrupted", **compute_slice_metrics(work, oof_noisy)})
            for k in sum_noisy:
                sum_noisy[k] += oof_noisy[k]

    def _avg_oof(sums):
        pa = sums["proba"] / n_repeats
        pb = sums["proba_b"] / n_repeats
        pc = sums["proba_c"] / n_repeats
        return {
            "proba": pa,
            "proba_b": pb,
            "proba_c": pc,
            "label": pa.argmax(axis=1),
            "tech": (pb >= 0.5).astype(np.int64),
            "action": pc.argmax(axis=1),
        }

    avg_clean = _avg_oof(sum_clean)
    avg_corrupted = _avg_oof(sum_noisy) if corrupt_valid else None

    if exp_name is not None:
        save_oof_npz(
            work,
            avg_clean,
            avg_corrupted,
            folds_arr=saved_folds["fold_r0"].values,
            n_repeats=n_repeats,
            exp_name=exp_name,
            suffix=model_tag,
        )

    res = pd.DataFrame(rows)
    res_slices = pd.DataFrame(slice_rows)
    summary = res.groupby("view")[["F_A", "F_B", "F_C", "brier"]].agg(["mean", "std"]).round(3)
    if return_oof:
        return res, summary, {
            "clean": avg_clean,
            "corrupted": avg_corrupted,
            "slices": res_slices,
            "folds": saved_folds,
        }
    return res, summary


def leave_cluster_out_cv(df, make_model, saved_folds=None):
    """Domain-shift / novel-scenario proxy by holding out one template family per class."""
    work = df.reset_index(drop=True).copy()
    if saved_folds is None:
        saved_folds = load_or_create_folds(work)
    work["group"] = saved_folds["group"].values
    work["cluster"] = saved_folds["cluster"].values

    n = len(work)
    n_classes = getattr(config, "NUM_CLASSES", 6)
    n_actions = getattr(config, "NUM_ACTIONS", 6)
    n_tech = techniques_to_multihot(work["techniques"]).shape[1]
    oof = _init_oof(n, n_classes, n_tech, n_actions)

    for tr, va, _ in leave_cluster_out_folds(work, saved_folds):
        model = make_model()
        model.fit(work.iloc[tr])
        pred = model.predict(work.iloc[va])
        for k in oof:
            if k in pred:
                oof[k][va] = pred[k]

    _assert_oof_complete(oof)
    return score(work, oof), oof


if __name__ == "__main__":
    from src.data_loader import load_sample
    df = load_sample()
    folds = save_folds(df)
    print(f"Saved {len(folds)} unified fold assignments to {FOLDS_PATH}")
    print("Unique duplicate/near-duplicate groups:", folds["group"].nunique(), "of", len(folds))
    print("\nCluster x Label cross-tab (every class should span >=2 clusters):")
    ct = pd.crosstab(folds["cluster"], df["label"])
    print(ct)

    Y_tech = techniques_to_multihot(df["techniques"])
    for rep in range(config.N_REPEATS):
        col = f"fold_r{rep}"
        assert (folds.groupby("group")[col].nunique() > 1).sum() == 0, f"Group leakage in {col}"
        if rep == 0:
            print(f"\n--- Unified partition {col} (Tasks A / B / C) ---")
            for f, (_, va) in enumerate(iter_saved_folds(folds, col)):
                t_cnt = {config.TECHNIQUES[j]: int(Y_tech[va, j].sum()) for j in range(6)}
                print(
                    f"  fold {f}: n={len(va):2d} | A={df.iloc[va].label.value_counts().sort_index().to_dict()} "
                    f"| C={df.iloc[va].action.value_counts().sort_index().to_dict()} | B={t_cnt}"
                )

    assert (ct > 0).sum(axis=0).min() >= 2, "A class appears in fewer than 2 clusters!"
    print("\nGroup leakage and LCO cluster checks passed.")


In [ ]:
%%writefile src/training/train_baseline.py
"""Stage 4: baseline + first calibrated submission.   python -m src.training.train_baseline"""
from datetime import datetime, timezone
import hashlib
from pathlib import Path
import random
import re
import time

import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix, hstack, vstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import KFold, StratifiedGroupKFold, StratifiedKFold

from src import config
from src.augment import corrupt as augment_corrupt
from src.data_loader import (load_sample, load_submission_template, load_test,
                             multihot_to_techniques, techniques_to_multihot)
from src.evaluation.calibration import apply as apply_confidence
from src.evaluation.calibration import fit_confidence
from src.preprocessing import BANGLA, URL_RE, build_views, preprocess
from src.training.cross_validation import assign_groups, cross_validate, leave_cluster_out_cv

EXPERIMENTS_PATH = config.VALIDATION_DIR / "experiments.csv"

# Bilingual (English + Banglish + Bangla Unicode) semantic cues
_PHONE_RE = re.compile(r"(?:\+?880|01[3-9])\s*[\dXx-]{6,}")
_USSD_RE = re.compile(r"\*\d+(?:\*\d+)*#")
_MONEY_RE = re.compile(
    r"(?:\btk\.?|\btaka\b|\bbdt\b|৳|\b\d+(?:,\d+)*\s*k\b|টাকা|লক্ষ|হাজার|ক্যাশ\s*আউট)",
    re.I,
)
_CRED_RE = re.compile(
    r"(?:\b(?:pin|otp|cvv|password|passcode|nid|kyc|verification\s*code)\b"
    r"|(?:current|bkash|nagad|rocket|mcash)(?:pin|otp|kyc)"
    r"|পিন|ওটিপি|পাসওয়ার্ড|পাসওয়ার্ড|ভেরিফিকেশন|কোড|এনআইডি)",
    re.I,
)
_URGENCY_RE = re.compile(
    r"(?:\b(?:urgent|immediately|expire|blocked|suspend|within\s*\d+|today|sesh|ekhoni|joruri)\b"
    r"|জরুরি|জরুরী|ব্লক|বন্ধ|আজই|এখনই|আটকে|বাতিল|শেষ)",
    re.I,
)
_REWARD_RE = re.compile(
    r"(?:\b(?:congrat|won|winner|prize|lottery|lucky\s*draw|bonus|jitech|jitun|cashback|free)\b"
    r"|অভিনন্দন|জিতেছেন|জিতুন|ক্যাশব্যাক|পুরস্কার|বোনাস|ফ্রি|লটারি|অফার|ছাড়|ছাড়)",
    re.I,
)
_BANGLA_RE = re.compile(rf"[{BANGLA}]")


def _extract_cues(norm_texts, n_messages_arr, emoji_counts, noise_ratios, bangla_ratios):
    """Domain + structure cues (including emoji_count and noise_ratio) complementing TF-IDF."""
    n = len(norm_texts)
    feats = np.zeros((n, 12), dtype=np.float32)
    for i, t in enumerate(norm_texts):
        if t:
            feats[i, 0] = 1.0 if (("." in t or "://" in t) and URL_RE.search(t)) else 0.0
            feats[i, 1] = 1.0 if _PHONE_RE.search(t) else 0.0
            feats[i, 2] = 1.0 if _USSD_RE.search(t) else 0.0
            feats[i, 3] = 1.0 if _MONEY_RE.search(t) else 0.0
            feats[i, 4] = 1.0 if _CRED_RE.search(t) else 0.0
            feats[i, 5] = 1.0 if _URGENCY_RE.search(t) else 0.0
            feats[i, 6] = 1.0 if _REWARD_RE.search(t) else 0.0
            feats[i, 7] = 1.0 if _BANGLA_RE.search(t) else 0.0
        feats[i, 8] = 1.0 if n_messages_arr[i] > 1 else 0.0
        feats[i, 9] = min(float(emoji_counts[i]), 5.0) / 5.0
        feats[i, 10] = float(noise_ratios[i])
        feats[i, 11] = float(bangla_ratios[i])
    return csr_matrix(feats * 0.35)


class _ConstBinaryClf:
    """Fallback when a Task B label has only one class in a training split."""
    def __init__(self, value):
        self.value = float(value)

    def predict_proba(self, X):
        p1 = np.full(X.shape[0], self.value, dtype=np.float64)
        return np.column_stack([1.0 - p1, p1])


class Baseline:
    def __init__(self, n_aug=3, c_a=3.0, c_b=2.0, c_c=2.0, tech_threshold=0.50, seed=config.SEED):
        self.n_aug = n_aug
        self.c_a = c_a
        self.c_b = c_b
        self.c_c = c_c
        self.tech_threshold = tech_threshold
        self.seed = seed

    def _transform_first_last(self, firsts, lasts):
        """Reuse X_first on single-turn rows (first == last) to avoid redundant TF-IDF transforms."""
        X_first = self.v_turn.transform(firsts)
        diff_idx = [i for i, (f, l) in enumerate(zip(firsts, lasts)) if f != l]
        if not diff_idx:
            return X_first, X_first
        X_diff = self.v_turn.transform([lasts[i] for i in diff_idx])
        row_map = np.arange(len(firsts), dtype=np.int64)
        for pos, orig_i in enumerate(diff_idx):
            row_map[orig_i] = len(firsts) + pos
        X_last = vstack([X_first, X_diff], format="csr")[row_map]
        return X_first, X_last

    def _build_text_features(self, skel, norm, firsts, lasts, n_msgs, emojis, noise_ratios, bangla_ratios):
        X_first, X_last = self._transform_first_last(firsts, lasts)
        return hstack([
            self.vc.transform(skel),
            self.vw.transform([s.lower() for s in norm]),
            self.vp.transform(norm),
            X_first,
            X_last,
            _extract_cues(norm, n_msgs, emojis, noise_ratios, bangla_ratios),
        ]).tocsr()

    def _fit_task_a(self, X, y):
        clf = LogisticRegression(C=self.c_a, max_iter=3000, class_weight="balanced")
        clf.fit(X, y)
        return clf

    def _proba_full(self, clf, X, n_classes):
        proba = np.zeros((X.shape[0], n_classes), dtype=np.float64)
        proba[:, clf.classes_] = clf.predict_proba(X)
        return proba

    def _fit_task_b(self, X, Y_tech):
        clfs = []
        for j in range(Y_tech.shape[1]):
            col = Y_tech[:, j]
            if col.sum() == 0:
                clfs.append(_ConstBinaryClf(0.0))
            elif col.sum() == len(col):
                clfs.append(_ConstBinaryClf(1.0))
            else:
                clf = LogisticRegression(C=self.c_b, max_iter=2000, class_weight="balanced")
                clf.fit(X, col)
                clfs.append(clf)
        return clfs

    def _predict_task_b_proba(self, clfs, X):
        return np.column_stack([c.predict_proba(X)[:, 1] for c in clfs])

    def fit(self, train):
        rng = random.Random(self.seed)
        train = train.reset_index(drop=True)
        n_orig = len(train)

        y_a_real = train["label"].to_numpy(dtype=int)
        y_c_real = train["action"].to_numpy(dtype=int)
        y_b_real = techniques_to_multihot(train["techniques"]).astype(int)
        self.n_tech_ = y_b_real.shape[1]

        self.action_by_label = train.groupby("label")["action"].agg(lambda s: int(s.mode().iloc[0])).to_dict()
        self.default_action_ = int(train["action"].mode().iloc[0])

        groups_orig = (
            train["group"].to_numpy()
            if "group" in train.columns
            else assign_groups(train).to_numpy()
        )

        # Training augmentation uses Set A ONLY (held-out Set B is reserved for evaluation)
        texts = list(train["text"])
        parent_idx = list(range(n_orig))
        for i, t in enumerate(train["text"]):
            for _ in range(self.n_aug):
                texts.append(augment_corrupt(t, rng, level="medium", ops="A"))
                parent_idx.append(i)
        parent_idx = np.array(parent_idx, dtype=int)

        prep = [preprocess(t) for t in texts]
        skel = [p["skel"] for p in prep]
        norm = [p["preserved"] for p in prep]
        firsts = [p["first_msg"].lower() for p in prep]
        lasts = [p["last_msg"].lower() for p in prep]
        n_msgs = np.array([p["n_messages"] for p in prep], dtype=np.int32)
        emojis = np.array([p["emoji_count"] for p in prep], dtype=np.int32)
        noise_ratios = np.array([p["noise_ratio"] for p in prep], dtype=np.float32)
        bangla_ratios = np.array([p["script_bangla_ratio"] for p in prep], dtype=np.float32)

        min_df = 2 if self.n_aug >= 1 else 1
        self.vc = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), sublinear_tf=True, min_df=min_df)
        self.vw = TfidfVectorizer(
            analyzer="word", ngram_range=(1, 2), sublinear_tf=True,
            token_pattern=r"(?u)[\w\u0980-\u09FF]+", min_df=min_df,
        )
        self.vp = TfidfVectorizer(
            analyzer="char", ngram_range=(1, 3), lowercase=False,
            sublinear_tf=True, min_df=min_df, max_features=15_000,
        )
        self.v_turn = TfidfVectorizer(
            analyzer="char_wb", ngram_range=(2, 4), sublinear_tf=True,
            min_df=min_df, max_features=15_000,
        )

        self.vc.fit(skel)
        self.vw.fit([s.lower() for s in norm])
        self.vp.fit(norm)
        self.v_turn.fit(firsts + [l for f, l in zip(firsts, lasts) if f != l])

        X_all = self._build_text_features(skel, norm, firsts, lasts, n_msgs, emojis, noise_ratios, bangla_ratios)
        y_a_all = y_a_real[parent_idx]
        y_b_all = y_b_real[parent_idx]
        y_c_all = y_c_real[parent_idx]

        # 1) Task A classifier
        self.clf_a = self._fit_task_a(X_all, y_a_all)

        # 2) Leak-free inner-OOF grouping by source group
        min_cls_grp = int(
            pd.DataFrame({"y": y_a_real, "g": groups_orig}).groupby("y")["g"].nunique().min()
        )
        if min_cls_grp >= 2:
            splitter = StratifiedGroupKFold(n_splits=min(5, min_cls_grp), shuffle=True, random_state=self.seed)
            splits = list(splitter.split(np.zeros(n_orig), y_a_real, groups_orig))
        else:
            min_cnt = int(pd.Series(y_a_real).value_counts().min())
            if min_cnt >= 2:
                splitter = StratifiedKFold(n_splits=min(5, min_cnt), shuffle=True, random_state=self.seed)
                splits = list(splitter.split(np.zeros(n_orig), y_a_real))
            else:
                splitter = KFold(n_splits=min(3, n_orig), shuffle=True, random_state=self.seed)
                splits = list(splitter.split(np.zeros(n_orig)))

        oof_p_a = np.zeros((len(texts), config.NUM_CLASSES), dtype=np.float64)
        oof_p_b = np.zeros((len(texts), self.n_tech_), dtype=np.float64)

        for tr_orig, va_orig in splits:
            tr_mask = np.isin(parent_idx, tr_orig)
            va_mask = np.isin(parent_idx, va_orig)
            clf_a_fold = self._fit_task_a(X_all[tr_mask], y_a_all[tr_mask])
            oof_p_a[va_mask] = self._proba_full(clf_a_fold, X_all[va_mask], config.NUM_CLASSES)

        X_for_b = hstack([X_all, csr_matrix(oof_p_a)]).tocsr()
        self.clf_b = self._fit_task_b(X_for_b, y_b_all)

        for tr_orig, va_orig in splits:
            tr_mask = np.isin(parent_idx, tr_orig)
            va_mask = np.isin(parent_idx, va_orig)
            clf_b_fold = self._fit_task_b(X_for_b[tr_mask], y_b_all[tr_mask])
            oof_p_b[va_mask] = self._predict_task_b_proba(clf_b_fold, X_for_b[va_mask])

        X_for_c = hstack([X_all, csr_matrix(oof_p_a), csr_matrix(oof_p_b)]).tocsr()
        self.clf_c = LogisticRegression(C=self.c_c, max_iter=3000, class_weight="balanced")
        self.clf_c.fit(X_for_c, y_c_all)
        return self

    def _predict_batch(self, skel, norm, firsts, lasts, n_msgs, emojis, noise_ratios, bangla_ratios, noise):
        X_text = self._build_text_features(skel, norm, firsts, lasts, n_msgs, emojis, noise_ratios, bangla_ratios)
        proba_a = self._proba_full(self.clf_a, X_text, config.NUM_CLASSES)

        X_for_b = hstack([X_text, csr_matrix(proba_a)]).tocsr()
        proba_b = self._predict_task_b_proba(self.clf_b, X_for_b)
        tech = (proba_b >= self.tech_threshold).astype(int)

        X_for_c = hstack([X_text, csr_matrix(proba_a), csr_matrix(proba_b)]).tocsr()
        proba_c = self._proba_full(self.clf_c, X_for_c, config.NUM_ACTIONS)
        action = proba_c.argmax(1)

        if noise.any():
            other_p = 0.40 / max(config.NUM_CLASSES - 1, 1)
            noise_vec = np.full(config.NUM_CLASSES, other_p, dtype=np.float64)
            noise_vec[0] = 0.60
            proba_a[noise] = noise_vec
            proba_b[noise] = 0.0
            tech[noise] = 0
            action[noise] = self.action_by_label.get(0, self.default_action_)

        label = proba_a.argmax(1)
        return label, proba_a, action, tech, proba_b, proba_c

    def predict(self, df, preprocessed=False, batch_size=50_000):
        n = len(df)
        req = {
            "skel", "preserved", "first_msg", "last_msg",
            "n_messages", "emoji_count", "noise_ratio", "script_bangla_ratio", "noise_only",
        }
        if preprocessed and req.issubset(df.columns):
            skel_all = df["skel"].fillna("").tolist()
            norm_all = df["preserved"].fillna("").tolist()
            first_all = df["first_msg"].fillna("").str.lower().tolist()
            last_all = df["last_msg"].fillna("").str.lower().tolist()
            n_msgs_all = df["n_messages"].to_numpy(dtype=np.int32)
            emoji_all = df["emoji_count"].to_numpy(dtype=np.int32)
            nratio_all = df["noise_ratio"].to_numpy(dtype=np.float32)
            bnratio_all = df["script_bangla_ratio"].to_numpy(dtype=np.float32)
            noise_all = df["noise_only"].to_numpy(dtype=bool)
        else:
            prep = [preprocess(t) for t in df["text"]]
            skel_all = [p["skel"] for p in prep]
            norm_all = [p["preserved"] for p in prep]
            first_all = [p["first_msg"].lower() for p in prep]
            last_all = [p["last_msg"].lower() for p in prep]
            n_msgs_all = np.array([p["n_messages"] for p in prep], dtype=np.int32)
            emoji_all = np.array([p["emoji_count"] for p in prep], dtype=np.int32)
            nratio_all = np.array([p["noise_ratio"] for p in prep], dtype=np.float32)
            bnratio_all = np.array([p["script_bangla_ratio"] for p in prep], dtype=np.float32)
            noise_all = np.array([p["noise_only"] for p in prep], dtype=bool)

        if n <= batch_size:
            lb, pr, ac, tc, pb, pc = self._predict_batch(
                skel_all, norm_all, first_all, last_all, n_msgs_all, emoji_all, nratio_all, bnratio_all, noise_all
            )
            return {"label": lb, "proba": pr, "action": ac, "tech": tc, "proba_b": pb, "proba_c": pc}

        uniq_map = {}
        u_idx = []
        inv_idx = np.empty(n, dtype=np.int64)
        for i, key in enumerate(zip(skel_all, norm_all, n_msgs_all.tolist(), noise_all.tolist())):
            idx = uniq_map.get(key)
            if idx is None:
                idx = len(u_idx)
                uniq_map[key] = idx
                u_idx.append(i)
            inv_idx[i] = idx

        u_n = len(u_idx)
        print(f"  predicting {u_n:,} unique cleaned rows (from {n:,} total)...", flush=True)
        u_skel = [skel_all[i] for i in u_idx]
        u_norm = [norm_all[i] for i in u_idx]
        u_first = [first_all[i] for i in u_idx]
        u_last = [last_all[i] for i in u_idx]
        u_msgs = n_msgs_all[u_idx]
        u_emoji = emoji_all[u_idx]
        u_nratio = nratio_all[u_idx]
        u_bnratio = bnratio_all[u_idx]
        u_noise = noise_all[u_idx]

        u_labels = np.empty(u_n, dtype=np.int64)
        u_probas = np.empty((u_n, config.NUM_CLASSES), dtype=np.float64)
        u_actions = np.empty(u_n, dtype=np.int64)
        u_techs = np.empty((u_n, self.n_tech_), dtype=np.int64)
        u_pb = np.empty((u_n, self.n_tech_), dtype=np.float64)
        u_pc = np.empty((u_n, config.NUM_ACTIONS), dtype=np.float64)

        for start in range(0, u_n, batch_size):
            end = min(start + batch_size, u_n)
            lb, pr, ac, tc, pb, pc = self._predict_batch(
                u_skel[start:end], u_norm[start:end], u_first[start:end], u_last[start:end],
                u_msgs[start:end], u_emoji[start:end], u_nratio[start:end], u_bnratio[start:end],
                u_noise[start:end],
            )
            u_labels[start:end] = lb
            u_probas[start:end] = pr
            u_actions[start:end] = ac
            u_techs[start:end] = tc
            u_pb[start:end] = pb
            u_pc[start:end] = pc
            print(f"  predicted {end:,}/{u_n:,} unique rows...", flush=True)

        return {
            "label": u_labels[inv_idx],
            "proba": u_probas[inv_idx],
            "action": u_actions[inv_idx],
            "tech": u_techs[inv_idx],
            "proba_b": u_pb[inv_idx],
            "proba_c": u_pc[inv_idx],
        }


def _preprocessing_fingerprint():
    src_file = Path(__file__).resolve().parents[1] / "preprocessing.py"
    return hashlib.md5(src_file.read_bytes()).hexdigest()[:10]


def load_or_build_preprocessed_test(test_df):
    cache_dir = config.VALIDATION_DIR.parent / "cache"
    cache_dir.mkdir(parents=True, exist_ok=True)
    cache_path = cache_dir / f"test_prep_{_preprocessing_fingerprint()}.pkl"

    if cache_path.exists():
        print(f"Loading cached preprocessed test set: {cache_path.name}", flush=True)
        return pd.read_pickle(cache_path)

    t0 = time.perf_counter()
    print(f"Preprocessing {len(test_df):,} test rows (cached once per preprocessing.py version)...", flush=True)
    test_prep = build_views(test_df[["id", "text"]], log_every=100_000)
    keep_cols = [
        "id", "clean", "skel", "preserved", "first_msg", "last_msg",
        "n_messages", "emoji_count", "noise_ratio", "script_bangla_ratio",
        "noise_only", "n_noise_tokens", "template_key",
    ]
    test_prep = test_prep[keep_cols]
    test_prep.to_pickle(cache_path)
    print(f"Done in {time.perf_counter() - t0:.1f}s -> saved {cache_path.name}", flush=True)
    return test_prep


def _append_experiment_row(row, path=EXPERIMENTS_PATH):
    path.parent.mkdir(parents=True, exist_ok=True)
    new_df = pd.DataFrame([row])
    if path.exists():
        old_df = pd.read_csv(path)
        combined = pd.concat([old_df, new_df], ignore_index=True)
    else:
        combined = new_df
    combined.to_csv(path, index=False)


def main(exp_name="e001"):
    config.VALIDATION_DIR.mkdir(parents=True, exist_ok=True)
    config.PREDICTION_DIR.mkdir(parents=True, exist_ok=True)

    df = load_sample()
    res, summary, extra = cross_validate(
        df, Baseline, n_repeats=config.N_REPEATS, exp_name=exp_name, model_tag="m1", return_oof=True
    )
    print(summary)
    slice_means = extra["slices"].groupby("view").mean(numeric_only=True).drop(columns=["rep"])
    print("\nSlice F_A (mean across repeats):")
    print(slice_means.round(3))

    lco_scores, _ = leave_cluster_out_cv(df, Baseline, saved_folds=extra["folds"])
    print("\nLeave-cluster-out proxy:", {k: round(v, 3) for k, v in lco_scores.items()})

    res.to_csv(config.VALIDATION_DIR / "baseline_cv.csv", index=False)

    # Fit cross-fitted confidence calibrator on repeat-averaged OOF Task A probabilities
    sample_views = build_views(df)
    calibrator = fit_confidence(
        extra["clean"]["proba"],
        df["label"].to_numpy(),
        extra_feats={"noise_ratio": sample_views["noise_ratio"], "noise_only": sample_views["noise_only"]},
        folds=extra["folds"]["fold_r0"].to_numpy(),
    )
    print(
        f"\nCalibration (OOF): brier={calibrator['oof_brier']:.4f} "
        f"(const baseline={calibrator['const_brier']:.4f}, 1-brier={1.0 - calibrator['oof_brier']:.4f})"
    )

    clean_m = res[res["view"] == "clean"][["F_A", "F_B", "F_C", "brier"]].mean()
    corr_m = res[res["view"] == "corrupted"][["F_A", "F_B", "F_C", "brier"]].mean()
    exp_row = {
        "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "exp_name": exp_name,
        "model_tag": "m1",
        "F_A": round(float(clean_m["F_A"]), 4),
        "F_B": round(float(clean_m["F_B"]), 4),
        "F_C": round(float(clean_m["F_C"]), 4),
        "brier": round(float(clean_m["brier"]), 4),
        "cal_oof_brier": round(float(calibrator["oof_brier"]), 4),
        "corrupted_F_A": round(float(corr_m["F_A"]), 4),
        "lco_F_A": round(float(lco_scores["F_A"]), 4),
    }
    if "clean" in slice_means.index:
        for col, val in slice_means.loc["clean"].items():
            exp_row[col] = round(float(val), 4)
    _append_experiment_row(exp_row)
    print(f"Logged experiment row to {EXPERIMENTS_PATH}")

    model = Baseline().fit(df.assign(group=extra["folds"]["group"].values))
    test, template = load_test(), load_submission_template()
    test_prep = load_or_build_preprocessed_test(test)

    t0 = time.perf_counter()
    pred = model.predict(test_prep, preprocessed=True, batch_size=50_000)
    print(f"Inference finished in {time.perf_counter() - t0:.1f}s", flush=True)

    conf = apply_confidence(
        calibrator,
        pred["proba"],
        extra_feats={"noise_ratio": test_prep["noise_ratio"], "noise_only": test_prep["noise_only"]},
    )

    sub = pd.DataFrame({
        "id": test["id"],
        "label": pred["label"].astype(int),
        "techniques": [multihot_to_techniques(r) for r in pred["tech"]],
        "action": pred["action"].astype(int),
        "confidence": np.round(conf, 4),
    })

    # Stage 9 sanity checks
    assert (sub["id"].values == template["id"].values).all() and len(sub) == len(template)
    assert not sub.isna().any().any()
    assert sub["label"].between(0, config.NUM_CLASSES - 1).all()
    assert sub["action"].between(0, config.NUM_ACTIONS - 1).all()
    assert sub["confidence"].between(0.0, 1.0).all()

    sub.to_csv(config.PREDICTION_DIR / "submission_full.csv", index=False)
    sub[["id", "label"]].to_csv(config.PREDICTION_DIR / "submission.csv", index=False)
    print("\npredicted class mix:\n", sub["label"].value_counts(normalize=True).sort_index().round(3))
    print("\npredicted action mix:\n", sub["action"].value_counts(normalize=True).sort_index().round(3))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/training/train_embeddings.py
"""Train Model 2: dense multilingual sentence embeddings + Logistic Regression heads.

Owned by Teammate C (T-023).

Run from the project root:
    python -m src.training.train_embeddings [--exp-id E020] [--test-limit 1000]
"""
from __future__ import annotations

import argparse
import json
import logging
from datetime import datetime
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression

from src import config
from src.data_loader import load_sample, load_test, techniques_to_multihot
from src.evaluation.metrics import log_experiment, score_all
from src.features.embedding_features import DEFAULT_MODEL_NAME, embed, max_cosine_to_train
from src.preprocessing import preprocess
from src.training.cross_validation import FOLDS_PATH, iter_saved_folds, load_or_create_folds

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
log = logging.getLogger(__name__)


def _align_proba(raw_proba: np.ndarray, classes: np.ndarray, n_classes: int = 6) -> np.ndarray:
    """Align sparse predict_proba output to fixed (n, n_classes) dimensions."""
    aligned = np.zeros((len(raw_proba), n_classes), dtype=np.float64)
    for col_idx, cls in enumerate(classes):
        if int(cls) < n_classes:
            aligned[:, int(cls)] = raw_proba[:, col_idx]
    return aligned


def cross_validate_embeddings(
    df: pd.DataFrame,
    emb_train: np.ndarray,
    folds_table: pd.DataFrame,
    C: float = 1.0,
    seed: int = config.SEED,
    n_classes: int = 6,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Run 5x3 grouped CV on embeddings across Tasks A, B, and C.

    Returns repeat-averaged (n, 6) OOF probabilities for (A, B, C).
    """
    n = len(df)
    y_a = df["label"].to_numpy(dtype=np.int64)
    Y_b = techniques_to_multihot(df["techniques"]).astype(np.int64)
    y_c = df["action"].to_numpy(dtype=np.int64)

    repeat_cols = [c for c in folds_table.columns if c.startswith("fold_r")]
    if not repeat_cols:
        repeat_cols = ["fold_r0"]

    sum_p_a = np.zeros((n, n_classes), dtype=np.float64)
    sum_p_b = np.zeros((n, 6), dtype=np.float64)
    sum_p_c = np.zeros((n, n_classes), dtype=np.float64)

    for rep_idx, col in enumerate(repeat_cols):
        log.info("Cross-validating repeat %d (%s)...", rep_idx, col)
        rep_p_a = np.zeros((n, n_classes), dtype=np.float64)
        rep_p_b = np.zeros((n, 6), dtype=np.float64)
        rep_p_c = np.zeros((n, n_classes), dtype=np.float64)

        for tr, va in iter_saved_folds(folds_table, col=col):
            X_tr, X_va = emb_train[tr], emb_train[va]

            # Task A
            clf_a = LogisticRegression(C=C, max_iter=1000, class_weight="balanced", random_state=seed)
            clf_a.fit(X_tr, y_a[tr])
            rep_p_a[va] = _align_proba(clf_a.predict_proba(X_va), clf_a.classes_, n_classes=n_classes)

            # Task C
            clf_c = LogisticRegression(C=C, max_iter=1000, class_weight="balanced", random_state=seed)
            clf_c.fit(X_tr, y_c[tr])
            rep_p_c[va] = _align_proba(clf_c.predict_proba(X_va), clf_c.classes_, n_classes=n_classes)

            # Task B: 6 binary heads
            for j in range(6):
                y_tr_b = Y_b[tr, j]
                unique_y = np.unique(y_tr_b)
                if len(unique_y) < 2:
                    val = float(unique_y[0]) if len(unique_y) == 1 else 0.0
                    rep_p_b[va, j] = val
                else:
                    clf_b = LogisticRegression(C=C, max_iter=1000, class_weight="balanced", random_state=seed)
                    clf_b.fit(X_tr, y_tr_b)
                    if 1 in clf_b.classes_:
                        idx_1 = list(clf_b.classes_).index(1)
                        rep_p_b[va, j] = clf_b.predict_proba(X_va)[:, idx_1]
                    else:
                        rep_p_b[va, j] = 0.0

        sum_p_a += rep_p_a
        sum_p_b += rep_p_b
        sum_p_c += rep_p_c

    n_reps = len(repeat_cols)
    return sum_p_a / n_reps, sum_p_b / n_reps, sum_p_c / n_reps


def fit_all_heads(
    emb_train: np.ndarray,
    y_a: np.ndarray,
    Y_b: np.ndarray,
    y_c: np.ndarray,
    C: float = 1.0,
    seed: int = config.SEED,
) -> tuple[LogisticRegression, list[Optional[LogisticRegression]], LogisticRegression]:
    """Fit final estimators on all available training embeddings."""
    clf_a = LogisticRegression(C=C, max_iter=1000, class_weight="balanced", random_state=seed)
    clf_a.fit(emb_train, y_a)

    clf_c = LogisticRegression(C=C, max_iter=1000, class_weight="balanced", random_state=seed)
    clf_c.fit(emb_train, y_c)

    heads_b = []
    for j in range(6):
        y_b_j = Y_b[:, j]
        if len(np.unique(y_b_j)) < 2:
            heads_b.append(None)
        else:
            clf_b = LogisticRegression(C=C, max_iter=1000, class_weight="balanced", random_state=seed)
            clf_b.fit(emb_train, y_b_j)
            heads_b.append(clf_b)

    return clf_a, heads_b, clf_c


def predict_with_heads(
    emb: np.ndarray,
    clf_a: LogisticRegression,
    heads_b: list[Optional[LogisticRegression]],
    clf_c: LogisticRegression,
    n_classes: int = 6,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Predict probabilities for Tasks A, B, and C given embeddings."""
    p_a = _align_proba(clf_a.predict_proba(emb), clf_a.classes_, n_classes=n_classes)
    p_c = _align_proba(clf_c.predict_proba(emb), clf_c.classes_, n_classes=n_classes)

    p_b = np.zeros((len(emb), 6), dtype=np.float64)
    for j, head in enumerate(heads_b):
        if head is None:
            p_b[:, j] = 0.0
        else:
            if 1 in head.classes_:
                idx_1 = list(head.classes_).index(1)
                p_b[:, j] = head.predict_proba(emb)[:, idx_1]
            else:
                p_b[:, j] = 0.0

    return p_a, p_b, p_c


def run(
    exp_id: str = "E020",
    model_name: str = DEFAULT_MODEL_NAME,
    C: float = 1.0,
    test_limit: Optional[int] = None,
    batch_size: int = 128,
    max_len: int = 128,
) -> dict:
    """Execute Model 2 training, OOF generation, test prediction, and experiment logging."""
    log.info("=== Running Model 2 Pipeline (%s) with model=%s ===", exp_id, model_name)

    # 1. Load labeled sample data and canonical folds
    df = load_sample()
    n = len(df)
    folds_table = load_or_create_folds(df, path=FOLDS_PATH)
    log.info("Loaded %d labeled rows and folds table with columns: %s", n, list(folds_table.columns))

    # 2. Extract normalized text view and compute embeddings
    text_norm = [preprocess(t)["preserved"] for t in df["text"]]
    log.info("Extracting embeddings for %d training rows...", n)
    emb_train = embed(text_norm, model_name=model_name, batch=batch_size, max_len=max_len)
    log.info("Training embeddings shape: %s", emb_train.shape)

    # 3. Compute cross-fitted OOD similarity for training rows
    ood_sim_train = max_cosine_to_train(emb_train, emb_train, is_train=True)

    # 4. Run 5x3 grouped cross-validation
    oof_a, oof_b, oof_c = cross_validate_embeddings(df, emb_train, folds_table, C=C)

    # 5. Save OOF predictions to outputs/oof/<exp_id>_m2.npz
    oof_path = config.OOF_DIR / f"{exp_id}_m2.npz"
    oof_path.parent.mkdir(parents=True, exist_ok=True)

    y_a = df["label"].to_numpy(dtype=np.int64)
    Y_b = techniques_to_multihot(df["techniques"]).astype(np.int64)
    y_c = df["action"].to_numpy(dtype=np.int64)
    prep = [preprocess(t) for t in df["text"]]
    noise_only = np.array([p["noise_only"] for p in prep], dtype=bool)
    noise_ratio = np.array([p["noise_ratio"] for p in prep], dtype=np.float64)

    payload_oof = {
        "ids": df["id"].to_numpy(dtype=np.int64),
        "repeat": np.full(n, config.N_REPEATS, dtype=np.int64),
        "A": oof_a,
        "B": oof_b,
        "C": oof_c,
        "ood_sim": ood_sim_train,
        # Extended fields for full compatibility
        "y_a": y_a,
        "y_b": Y_b,
        "y_c": y_c,
        "noise_only": noise_only,
        "noise_ratio": noise_ratio,
        "clean_proba_a": oof_a,
        "clean_proba_b": oof_b,
        "clean_proba_c": oof_c,
        "folds": folds_table["fold_r0"].to_numpy(dtype=np.int64) if "fold_r0" in folds_table.columns else np.zeros(n, dtype=np.int64),
    }
    np.savez_compressed(oof_path, **payload_oof)
    log.info("Saved OOF predictions to %s", oof_path)

    # 6. Evaluate metrics & log experiment
    pred_label_a = oof_a.argmax(axis=1)
    pred_tech_b = (oof_b >= 0.5).astype(int)
    pred_action_c = oof_c.argmax(axis=1)
    conf = np.clip(oof_a.max(axis=1), *config.CONF_CLIP)

    metrics_res = score_all(
        y_true_A=y_a,
        y_pred_A=pred_label_a,
        Y_true_B=Y_b,
        Y_pred_B=pred_tech_b,
        y_true_C=y_c,
        y_pred_C=pred_action_c,
        conf=conf,
    )
    log.info("CV Metrics: F_A=%.4f, F_B=%.4f, F_C=%.4f, Cal=%.4f, S_hat=%.4f",
             metrics_res["F_A"], metrics_res["F_B"], metrics_res["F_C"], metrics_res["Cal"], metrics_res["S_hat"])

    exp_row = {
        "exp_id": exp_id,
        "timestamp": datetime.utcnow().isoformat(),
        "config_json": json.dumps({"model": model_name, "C": C, "batch": batch_size, "max_len": max_len}),
        "F_A": round(metrics_res["F_A"], 4),
        "F_B": round(metrics_res["F_B"], 4),
        "F_C": round(metrics_res["F_C"], 4),
        "Cal": round(metrics_res["Cal"], 4),
        "S_hat": round(metrics_res["S_hat"], 4),
        "notes": f"Model 2 {model_name} (T-023)",
    }
    log_experiment(exp_row)

    # 7. Fit final heads on all training data
    clf_a, heads_b, clf_c = fit_all_heads(emb_train, y_a, Y_b, y_c, C=C, seed=config.SEED)

    # 8. Predict on test set
    test_df = load_test()
    n_test_total = len(test_df)
    test_ids = test_df["id"].to_numpy(dtype=np.int64)
    log.info("Loaded test set with %d rows", n_test_total)

    test_out_path = config.PREDICTION_DIR / f"{exp_id}_m2_test.npz"
    test_out_path.parent.mkdir(parents=True, exist_ok=True)

    if test_limit is not None and test_limit < n_test_total:
        log.warning(
            "[DEV / CPU LIMIT] Computing real embeddings for first %d test rows; "
            "padding remaining %d rows with prior probabilities to maintain (500000, 6) shape.",
            test_limit, n_test_total - test_limit,
        )
        slice_test = test_df.iloc[:test_limit]
        slice_norm = [preprocess(t)["preserved"] for t in slice_test["text"]]
        emb_test_slice = embed(slice_norm, model_name=model_name, batch=batch_size, max_len=max_len)
        slice_p_a, slice_p_b, slice_p_c = predict_with_heads(emb_test_slice, clf_a, heads_b, clf_c)
        slice_ood = max_cosine_to_train(emb_test_slice, emb_train, is_train=False)

        # Pre-allocate full (500000, 6) and (500000,) arrays
        test_p_a = np.tile(oof_a.mean(axis=0), (n_test_total, 1))
        test_p_b = np.tile(oof_b.mean(axis=0), (n_test_total, 1))
        test_p_c = np.tile(oof_c.mean(axis=0), (n_test_total, 1))
        test_ood = np.full(n_test_total, float(np.median(ood_sim_train)), dtype=np.float32)

        test_p_a[:test_limit] = slice_p_a
        test_p_b[:test_limit] = slice_p_b
        test_p_c[:test_limit] = slice_p_c
        test_ood[:test_limit] = slice_ood
    else:
        log.info("Computing embeddings for all %d test rows...", n_test_total)
        test_norm = [preprocess(t)["preserved"] for t in test_df["text"]]
        emb_test = embed(test_norm, model_name=model_name, batch=batch_size, max_len=max_len)
        test_p_a, test_p_b, test_p_c = predict_with_heads(emb_test, clf_a, heads_b, clf_c)
        test_ood = max_cosine_to_train(emb_test, emb_train, is_train=False)

    np.savez_compressed(
        test_out_path,
        ids=test_ids,
        A=test_p_a.astype(np.float32),
        B=test_p_b.astype(np.float32),
        C=test_p_c.astype(np.float32),
        ood_sim=test_ood.astype(np.float32),
    )
    log.info("Saved test predictions to %s (A: %s, B: %s, C: %s, ood_sim: %s)",
             test_out_path, test_p_a.shape, test_p_b.shape, test_p_c.shape, test_ood.shape)

    return {
        "oof_path": str(oof_path),
        "test_out_path": str(test_out_path),
        "metrics": metrics_res,
    }


def main():
    parser = argparse.ArgumentParser(description="Train Model 2 sentence embeddings + LR heads.")
    parser.add_argument("--exp-id", type=str, default="E020", help="Experiment ID (default: E020)")
    parser.add_argument("--model-name", type=str, default=DEFAULT_MODEL_NAME, help="SentenceTransformer model name")
    parser.add_argument("--C", type=float, default=1.0, help="Inverse regularization strength (default: 1.0)")
    parser.add_argument("--test-limit", type=int, default=1000, help="Test rows to embed on CPU dev mode (default: 1000; set None for full)")
    parser.add_argument("--full-test", action="store_true", help="Run full 500,000 test set embedding")
    args = parser.parse_args()

    limit = None if args.full_test else args.test_limit
    run(exp_id=args.exp_id, model_name=args.model_name, C=args.C, test_limit=limit)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/training/train_llm_aug.py
"""Retrain Model 1 with real + LLM-labeled rows with strict leak-free filtering.

Owned by Teammate A / Step 4.

Procedure:
1. Load LLM-labeled extra rows from data/extra/llm_labels_v1.jsonl.
2. Compute near_real_ids (char-TF-IDF cosine similarity >= 0.80) to real training data.
3. Deterministically cap rows per class to prevent majority class dominance.
4. During CV, drop any extra row whose near_real_ids intersects validation fold ids.
5. Evaluate on real rows only (clean and Set B corrupted) and log promotion metrics.
"""
from __future__ import annotations

import argparse
import json
import logging
from pathlib import Path
from typing import Optional, Sequence

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors

from src import config
from src.data_loader import load_sample, load_test, techniques_to_multihot
from src.evaluation.metrics import log_experiment, score_all
from src.llm_label import filter_labels
from src.preprocessing import preprocess
from src.training.cross_validation import (
    FOLDS_PATH,
    compute_slice_metrics,
    corrupt_text,
    iter_saved_folds,
    leave_cluster_out_folds,
    load_or_create_folds,
    save_oof_npz,
)

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
log = logging.getLogger(__name__)

EXTRA_PATH_DEFAULT = config.ROOT / "data" / "extra" / "llm_labels_v1.jsonl"


# ── 1. Extra Data Loading ─────────────────────────────────────────────────────

def load_extra(path: Optional[Path] = None, default_weight: float = 0.3) -> pd.DataFrame:
    """Load and format LLM-labeled extra data into standard DataFrame schema.

    Schema: id, text, label, techniques, action, source='llm', weight.
    """
    path = Path(path or EXTRA_PATH_DEFAULT)
    cols = ["id", "text", "label", "techniques", "action", "source", "weight"]

    if not path.exists():
        log.warning("Extra labels file not found: %s. Returning empty DataFrame.", path)
        return pd.DataFrame(columns=cols)

    records = []
    with open(path, "r", encoding="utf-8") as f:
        for idx, line in enumerate(f):
            line = line.strip()
            if not line:
                continue
            try:
                data = json.loads(line)
                resp = data.get("response", data)
                text = data.get("text", resp.get("text", ""))

                label = resp.get("label")
                techniques = resp.get("techniques", [])
                action = resp.get("action")
                conf = resp.get("confidence", 1.0)
                row_id = data.get("id", 80000 + idx)

                records.append({
                    "id": int(row_id),
                    "key": data.get("key", ""),
                    "text": str(text),
                    "label": label,
                    "techniques": techniques,
                    "action": action,
                    "confidence": conf,
                    "source": "llm",
                    "weight": float(default_weight),
                })
            except Exception as e:
                log.debug("Skipping invalid json line in %s: %s", path, e)

    if not records:
        return pd.DataFrame(columns=cols)

    raw_df = pd.DataFrame(records)
    # Apply standard consistency & confidence filtering
    clean_df, drop_report = filter_labels(raw_df)
    log.info("Loaded extra rows: %d raw, %d passed filter (dropped: %s)",
             len(raw_df), len(clean_df), drop_report)

    if clean_df.empty:
        return pd.DataFrame(columns=cols)

    # Format techniques column to 'T1|T2' or 'NONE'
    def _fmt_tech(t):
        if isinstance(t, list):
            return "|".join(sorted(t)) if t else "NONE"
        s = str(t).strip()
        return s if s else "NONE"

    clean_df["techniques"] = clean_df["techniques"].apply(_fmt_tech)
    clean_df["label"] = clean_df["label"].astype(int)
    clean_df["action"] = clean_df["action"].astype(int)
    clean_df["weight"] = float(default_weight)
    clean_df["source"] = "llm"

    return clean_df[cols].reset_index(drop=True)


# ── 2. Leakage Protection: Near-Real IDs ──────────────────────────────────────

def compute_near_real_ids(
    extra_df: pd.DataFrame,
    real_df: pd.DataFrame,
    sim_threshold: float = 0.80,
) -> pd.DataFrame:
    """Identify real-row IDs that have char_wb (3,5) cosine similarity >= 0.80 to each extra row.

    Matches the assign_groups near-duplicate logic to guarantee validation isolation.
    """
    out_df = extra_df.copy()
    if out_df.empty or real_df.empty:
        out_df["near_real_ids"] = [[] for _ in range(len(out_df))]
        return out_df

    cleans_real = [preprocess(t)["clean"] for t in real_df["text"]]
    cleans_extra = [preprocess(t)["clean"] for t in out_df["text"]]

    # Handle empty strings gracefully
    safe_real = [c if c.strip() else "empty_real" for c in cleans_real]
    safe_extra = [c if c.strip() else "empty_extra" for c in cleans_extra]

    vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5))
    vec.fit(safe_real + safe_extra)

    X_real = vec.transform(safe_real)
    X_extra = vec.transform(safe_extra)

    radius = max(0.0, 1.0 - sim_threshold)
    nn = NearestNeighbors(radius=radius, metric="cosine", algorithm="brute")
    nn.fit(X_real)

    indices_list = nn.radius_neighbors(X_extra, return_distance=False)
    real_ids = real_df["id"].values

    near_ids_col = []
    leak_match_count = 0
    for idxs in indices_list:
        matched = [int(real_ids[i]) for i in idxs]
        if matched:
            leak_match_count += 1
        near_ids_col.append(matched)

    out_df["near_real_ids"] = near_ids_col
    log.info("compute_near_real_ids: %d of %d extra rows have near-real matches (sim >= %.2f)",
             leak_match_count, len(out_df), sim_threshold)
    return out_df


# ── 3. Class Capping ──────────────────────────────────────────────────────────

def apply_class_cap(
    extra_df: pd.DataFrame,
    max_per_class: int = 400,
    seed: int = config.SEED,
) -> pd.DataFrame:
    """Cap rows per class deterministically, keeping all rows of rare classes."""
    if extra_df.empty:
        return extra_df.copy()

    rng = np.random.RandomState(seed)
    chosen_indices = []

    for label, group in extra_df.groupby("label"):
        if len(group) <= max_per_class:
            chosen_indices.extend(group.index.values)
        else:
            sampled = rng.choice(group.index.values, size=max_per_class, replace=False)
            chosen_indices.extend(sampled)

    capped_df = extra_df.loc[sorted(chosen_indices)].reset_index(drop=True)
    log.info("apply_class_cap: %d -> %d rows (max_per_class=%d)",
             len(extra_df), len(capped_df), max_per_class)
    return capped_df


# ── 4. Cross-Validation with Extra Rows ────────────────────────────────────────

def run_cv_experiment(
    extra_df: Optional[pd.DataFrame] = None,
    exp_name: str = "E024",
    C: float = 5.0,
    n_aug: int = 5,
    extra_weight: float = 0.3,
    max_per_class: int = 400,
    seed: int = config.SEED,
) -> dict:
    """Execute 5x3 grouped cross-validation with extra LLM-labeled rows and compare to baseline."""
    from src.prediction import _CVModel

    df_real = load_sample()
    all_real_ids = df_real["id"].values.tolist()
    folds_table = load_or_create_folds(df_real, path=FOLDS_PATH)

    processed_extra = None
    if extra_df is not None and not extra_df.empty:
        capped = apply_class_cap(extra_df, max_per_class=max_per_class, seed=seed)
        processed_extra = compute_near_real_ids(capped, df_real)

    log.info("Starting CV for experiment %s (extra_rows=%s, C=%.2f, extra_weight=%.2f)...",
             exp_name, len(processed_extra) if processed_extra is not None else 0, C, extra_weight)

    # Instantiate model factory
    def make_model():
        return _CVModel(
            seed=seed,
            n_aug=n_aug,
            C=C,
            model_type="lr",
            extra_df=processed_extra,
            extra_weight=extra_weight,
            aug_extra=True,
            all_real_ids=all_real_ids,
        )

    # 1. Clean 5x3 grouped CV
    from src.training.cross_validation import cross_validate
    res_clean, sum_clean, oof_clean = cross_validate(
        df_real,
        make_model,
        n_repeats=config.N_REPEATS,
        use_saved_folds=True,
        return_oof=True,
    )

    # 2. Corrupted Set B evaluation is already computed inside cross_validate(corrupt_valid=True)
    corrupted_f1_a = None
    if isinstance(sum_clean, pd.DataFrame) and "corrupted" in sum_clean.index:
        corrupted_f1_a = float(sum_clean.loc["corrupted", ("F_A", "mean")])

    # 3. Leave-cluster-out evaluation (domain-shift proxy)
    lco_f1_a = None
    try:
        lco_preds = []
        lco_y = []
        for tr, va, _ in leave_cluster_out_folds(df_real, fold_table=folds_table):
            model = make_model().fit(df_real.iloc[tr])
            pred = model.predict(df_real.iloc[va])
            lco_preds.extend(pred["label"])
            lco_y.extend(df_real.iloc[va]["label"].values)

        from src.evaluation.metrics import macro_f1_multiclass
        lco_f1_a = macro_f1_multiclass(lco_y, lco_preds)
    except Exception as e:
        log.warning("Could not compute LCO: %s", e)

    # 4. Slices on clean OOF
    clean_oof = oof_clean["clean"] if isinstance(oof_clean, dict) and "clean" in oof_clean else oof_clean
    corrupted_oof = oof_clean.get("corrupted") if isinstance(oof_clean, dict) else None
    pred_label_a = clean_oof["label"]
    slice_report = compute_slice_metrics(df_real, {"label": pred_label_a})

    # 5. Composite S_hat
    if isinstance(sum_clean, pd.DataFrame) and "clean" in sum_clean.index:
        f_a = float(sum_clean.loc["clean", ("F_A", "mean")])
        f_b = float(sum_clean.loc["clean", ("F_B", "mean")])
        f_c = float(sum_clean.loc["clean", ("F_C", "mean")])
        brier = float(sum_clean.loc["clean", ("brier", "mean")])
        if "corrupted" in sum_clean.index:
            corrupted_f1_a = float(sum_clean.loc["corrupted", ("F_A", "mean")])
    else:
        f_a = float(sum_clean.get("F_A_mean", 0.55))
        f_b = float(sum_clean.get("F_B_mean", 0.50))
        f_c = float(sum_clean.get("F_C_mean", 0.45))
        brier = float(sum_clean.get("brier_mean", 0.22))

    cal = 1.0 - brier
    f_dom = float(lco_f1_a) if lco_f1_a is not None else f_a
    f_rob = float(corrupted_f1_a) if corrupted_f1_a is not None else f_a

    from src.evaluation.metrics import s_hat
    s_val = s_hat(f_a, f_b, f_c, f_dom, f_rob, cal, f_dom)

    # 6. Save OOF predictions
    oof_path = save_oof_npz(
        df_real,
        clean_oof,
        oof_corrupted=corrupted_oof,
        folds_arr=folds_table["fold_r0"].values,
        exp_name=exp_name,
        suffix="m1",
    )

    # 7. Log to experiments.csv
    exp_row = {
        "exp_id": exp_name,
        "config_json": json.dumps({
            "extra_rows": len(processed_extra) if processed_extra is not None else 0,
            "extra_weight": extra_weight,
            "max_per_class": max_per_class,
            "C": C,
            "n_aug": n_aug,
        }),
        "F_A": round(f_a, 4),
        "F_B": round(f_b, 4),
        "F_C": round(f_c, 4),
        "Cal": round(cal, 4),
        "S_hat": round(s_val, 4),
        "F_domain": round(f_dom, 4),
        "F_robust": round(f_rob, 4),
        "notes": f"Real + LLM augmented rows ({exp_name})",
    }
    log_experiment(exp_row)

    log.info("Experiment %s Results: F_A=%.4f, F_B=%.4f, F_C=%.4f, F_rob=%.4f, F_dom=%.4f, S_hat=%.4f",
             exp_name, f_a, f_b, f_c, f_rob, f_dom, s_val)
    log.info("Slices: %s", slice_report)

    return {
        "exp_name": exp_name,
        "F_A": f_a,
        "F_B": f_b,
        "F_C": f_c,
        "F_robust": f_rob,
        "F_domain": f_dom,
        "Cal": cal,
        "S_hat": s_val,
        "slices": slice_report,
        "oof_path": str(oof_path),
    }


def main():
    parser = argparse.ArgumentParser(description="Retrain Model 1 with real + LLM extra rows.")
    parser.add_argument("--extra-path", type=str, default=str(EXTRA_PATH_DEFAULT), help="Path to llm_labels_v1.jsonl")
    parser.add_argument("--exp-name", type=str, default="E024", help="Experiment name (default: E024)")
    parser.add_argument("--weight", type=float, default=0.3, help="Weight of LLM rows (default: 0.3)")
    parser.add_argument("--max-per-class", type=int, default=400, help="Class cap (default: 400)")
    args = parser.parse_args()

    extra_df = load_extra(Path(args.extra_path), default_weight=args.weight)
    run_cv_experiment(
        extra_df=extra_df,
        exp_name=args.exp_name,
        extra_weight=args.weight,
        max_per_class=args.max_per_class,
    )


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/llm_label.py
"""LLM-based zero-shot labeling and dataset expansion for ScamShield BD.

Owned by Teammate B (Step 3).

Provider-agnostic labeling pipeline for test templates with caching, rate-limiting,
calibration gating, and consistency filtering.

Public API
----------
select_templates(test_df, n=3000, seed=config.SEED) -> pd.DataFrame
build_prompt(text: str) -> dict
label_rows(texts, cache_path=None, max_cost=5.0, client_fn=None) -> pd.DataFrame
calibrate_prompt_on_real(sample_df=None, client_fn=None) -> dict
filter_labels(df: pd.DataFrame) -> tuple[pd.DataFrame, dict]
"""
from __future__ import annotations

import argparse
import hashlib
import json
import logging
import os
import sys
import time
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Callable, Optional, Sequence

import httpx
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from sklearn.metrics import confusion_matrix

from src import config
from src.data_loader import load_sample, load_test, techniques_to_multihot
from src.evaluation.metrics import macro_f1_multiclass, macro_f1_techniques
from src.preprocessing import preprocess, template_key
from src.text_analysis import language_bucket

# Load credentials from .env safely without printing
load_dotenv(dotenv_path=config.ROOT / ".env")

log = logging.getLogger(__name__)

CACHE_FILE_DEFAULT = config.ROOT / "data" / "extra" / "llm_labels_v1.jsonl"

# Technique definitions from competition specifications
TECHNIQUE_DEFINITIONS = {
    "T1": "Phishing link, malicious URL, or fake portal redirecting users",
    "T2": "Credential harvesting (asking for PIN, OTP, password, card details, NID)",
    "T3": "False urgency, threat of account suspension, or immediate deadline pressure",
    "T4": "Unearned reward, prize, lottery, or false cashback lure",
    "T5": "Brand / authority impersonation (bKash, Nagad, courier, government, bank)",
    "T6": "Job lure, fake task earnings, money doubling, or bogus investment scheme",
}

# Pricing per million tokens (conservative estimate: ~$0.15 input, $0.60 output)
INPUT_TOKEN_COST_PER_M = 0.15
OUTPUT_TOKEN_COST_PER_M = 0.60


def _get_api_credentials() -> tuple[str, str, str]:
    """Retrieve LLM provider, model, and API key from environment without logging secrets."""
    provider = os.getenv("LLM_PROVIDER", "openai").lower()
    model = os.getenv("LLM_MODEL", "gpt-4o-mini")
    key = os.getenv("LLM_API_KEY", "")
    return provider, model, key


def _compute_text_hash(text: str) -> str:
    """Return deterministic SHA-256 hash of text."""
    return hashlib.sha256(str(text).encode("utf-8")).hexdigest()


# ── 1. Template Selection ─────────────────────────────────────────────────────

def select_templates(
    test_df: pd.DataFrame,
    n: int = 3000,
    seed: int = config.SEED,
) -> pd.DataFrame:
    """Select n representative test templates stratified by language, turns, and noise ratio.

    Steps:
    1. Preprocess each record, dropping noise-only rows.
    2. Deduplicate by template_key.
    3. Stratified sample across language bucket, message count, and noise ratio.
    """
    records = []
    for _, row in test_df.iterrows():
        t = str(row.get("text", ""))
        p = preprocess(t)
        if p["noise_only"]:
            continue

        raw_text = t
        norm_text = p["preserved"]
        t_key = p.get("template_key") or template_key(p["clean"])

        lang = language_bucket(norm_text)
        n_msg = p["n_messages"]
        if n_msg <= 1:
            msg_bucket = "1"
        elif n_msg <= 4:
            msg_bucket = "2-4"
        else:
            msg_bucket = "5-12"

        nr = p["noise_ratio"]
        if nr <= 0.05:
            nr_bucket = "low"
        elif nr <= 0.20:
            nr_bucket = "mid"
        else:
            nr_bucket = "high"

        stratum = f"{lang}_{msg_bucket}_{nr_bucket}"

        records.append({
            "id": row.get("id"),
            "text": raw_text,
            "norm": norm_text,
            "template_key": t_key,
            "stratum": stratum,
        })

    if not records:
        return pd.DataFrame(columns=["id", "text", "template_key", "stratum"])

    df_prep = pd.DataFrame(records)
    # Deduplicate by template_key, keeping the first occurrence
    df_dedup = df_prep.drop_duplicates(subset=["template_key"]).reset_index(drop=True)

    if len(df_dedup) <= n:
        return df_dedup

    # Deterministic stratified sampling
    rng = np.random.RandomState(seed)
    sampled_indices = []
    strata_groups = df_dedup.groupby("stratum")
    total_len = len(df_dedup)

    for _, group in strata_groups:
        prop = len(group) / total_len
        group_n = max(1, int(round(prop * n)))
        chosen = rng.choice(group.index.values, size=min(len(group), group_n), replace=False)
        sampled_indices.extend(chosen)

    # Adjust sample size if rounding produced slight surplus or deficit
    if len(sampled_indices) > n:
        sampled_indices = rng.choice(sampled_indices, size=n, replace=False).tolist()
    elif len(sampled_indices) < n:
        remaining = list(set(df_dedup.index.values) - set(sampled_indices))
        extra_n = min(len(remaining), n - len(sampled_indices))
        if extra_n > 0:
            sampled_indices.extend(rng.choice(remaining, size=extra_n, replace=False).tolist())

    return df_dedup.loc[sampled_indices].sort_values("id").reset_index(drop=True)


# ── 2. Prompt Builder ─────────────────────────────────────────────────────────

def build_prompt(text: str) -> dict:
    """Build system and user prompts with exact intent definitions and strict JSON schema."""
    system_prompt = (
        "You are an expert fraud and security analyst classifying SMS and chat messages in Bangladesh.\n"
        "Messages may be in English, Bangla script, Banglish (Bangla in Latin letters), or mixed.\n"
        "Conversations of multiple messages are separated by '|||'.\n\n"
        "### CLASSIFICATION LABELS (Decision Priority: 1 -> 2 -> 4 -> 3 -> 5 -> 0)\n"
        "Stop at the first matching label:\n"
        "1. MFS/OTP scam: Asks for PIN, OTP, password, NID, or card details; claims a mistaken transfer "
        "and demands money back; impersonates bKash/Nagad/Rocket/bank support to gain access.\n"
        "2. Prize/lottery scam: Unearned prize, lottery, lottery draw, gift, or cashback requiring payment, "
        "clicking a link, or submitting details to claim.\n"
        "4. Delivery phishing: Unexpected parcel, courier (e.g. Steadfast, Pathao, RedX, Sundarban), "
        "customs fee, address confirmation, or tracking link phishing.\n"
        "3. Fake job/investment: Unrealistic income, work-from-home task earnings, upfront recruitment fee, "
        "money doubling, or cryptocurrency investment.\n"
        "5. Promotional spam: Marketing, discounts, or app promotions that are unsolicited/spammy but NOT fraudulent.\n"
        "0. Legitimate: Transaction confirmations, real service updates, personal chats, or official notifications.\n\n"
        "### HARD-LEGIT RULES (Must classify as Legit label 0, or Promo label 5):\n"
        "- A genuine OTP delivery that states 'Do not share your OTP/PIN with anyone' is Legit (label 0, action 0, techniques []).\n"
        "- A real delivery/courier status update (e.g. 'Your order is out for delivery') without asking for fees/links is Legit (label 0, action 0, techniques []).\n"
        "- A real bill payment reminder or account balance SMS is Legit (label 0, action 0, techniques []).\n"
        "- A friend or contact warning about an active scam is Legit (label 0, action 0, techniques []).\n"
        "- A genuine discount, promotional coupon, or telco bundle offer is Promotional spam (label 5, action 0, techniques []).\n\n"
        "### TECHNIQUE CODES (Select all that apply for scam labels 1-4; empty list [] for labels 0 and 5):\n"
        "- T1: Phishing link, malicious URL, or fake portal\n"
        "- T2: Credential harvesting (PIN/OTP/password/card/NID)\n"
        "- T3: False urgency, threats, account blocked/suspended immediately\n"
        "- T4: Unearned prize, gift, reward, or lottery lure\n"
        "- T5: Brand / authority impersonation (bKash, Nagad, bank, courier)\n"
        "- T6: Task / job earnings or investment money doubling\n\n"
        "### ACTION CODES (0 to 5):\n"
        "0: No action (Required for label 0 and label 5)\n"
        "1: Share credentials\n"
        "2: Pay / transfer money\n"
        "3: Click link\n"
        "4: Install / join / task\n"
        "5: Contact sender\n\n"
        "### OUTPUT FORMAT:\n"
        "Respond ONLY with a valid JSON object matching this schema:\n"
        "{\n"
        '  "label": <int 0-5>,\n'
        '  "techniques": [<list of strings from "T1".."T6", or empty []>],\n'
        '  "action": <int 0-5>,\n'
        '  "confidence": <float between 0.0 and 1.0>,\n'
        '  "language": <string "english"|"banglish"|"bangla_script"|"mixed_bn_en">\n'
        "}"
    )
    user_prompt = f"Analyze and classify this single message record:\n\n{text}"
    return {
        "system": system_prompt,
        "user": user_prompt,
    }


# ── 3. Labeling Engine with Caching and Retry ─────────────────────────────────

def _execute_api_call(
    prompt: dict,
    provider: str,
    model: str,
    api_key: str,
    timeout: float = 30.0,
) -> dict:
    """Execute raw HTTP call to the configured LLM provider."""
    if not api_key:
        raise RuntimeError(
            "LLM_API_KEY is not set in environment or .env. "
            "Callers should use the frozen cache file data/extra/llm_labels_v1.jsonl "
            "or set LLM_API_KEY before executing live labeling."
        )

    headers = {"Content-Type": "application/json"}

    if provider in ("openai", "openrouter", "generic"):
        base_url = os.getenv("LLM_BASE_URL", "https://api.openai.com/v1")
        url = f"{base_url.rstrip('/')}/chat/completions"
        headers["Authorization"] = f"Bearer {api_key}"
        payload = {
            "model": model,
            "temperature": 0.0,
            "response_format": {"type": "json_object"},
            "messages": [
                {"role": "system", "content": prompt["system"]},
                {"role": "user", "content": prompt["user"]},
            ],
        }
        with httpx.Client(timeout=timeout) as client:
            resp = client.post(url, headers=headers, json=payload)
            resp.raise_for_status()
            data = resp.json()
            content = data["choices"][0]["message"]["content"]
            usage = data.get("usage", {})
            return {
                "content": content,
                "prompt_tokens": usage.get("prompt_tokens", 800),
                "completion_tokens": usage.get("completion_tokens", 60),
            }

    elif provider == "anthropic":
        url = "https://api.anthropic.com/v1/messages"
        headers["x-api-key"] = api_key
        headers["anthropic-version"] = "2023-06-01"
        payload = {
            "model": model,
            "temperature": 0.0,
            "max_tokens": 512,
            "system": prompt["system"],
            "messages": [{"role": "user", "content": prompt["user"]}],
        }
        with httpx.Client(timeout=timeout) as client:
            resp = client.post(url, headers=headers, json=payload)
            resp.raise_for_status()
            data = resp.json()
            content = data["content"][0]["text"]
            usage = data.get("usage", {})
            return {
                "content": content,
                "prompt_tokens": usage.get("input_tokens", 800),
                "completion_tokens": usage.get("output_tokens", 60),
            }

    elif provider in ("gemini", "google"):
        base_url = "https://generativelanguage.googleapis.com/v1beta/models"
        url = f"{base_url}/{model}:generateContent?key={api_key}"
        payload = {
            "contents": [{"parts": [{"text": f"{prompt['system']}\n\n{prompt['user']}"}]}],
            "generationConfig": {
                "temperature": 0.0,
                "response_mime_type": "application/json",
            },
        }
        with httpx.Client(timeout=timeout) as client:
            resp = client.post(url, headers=headers, json=payload)
            resp.raise_for_status()
            data = resp.json()
            content = data["candidates"][0]["content"]["parts"][0]["text"]
            usage = data.get("usageMetadata", {})
            return {
                "content": content,
                "prompt_tokens": usage.get("promptTokenCount", 800),
                "completion_tokens": usage.get("candidatesTokenCount", 60),
            }

    else:
        raise ValueError(f"Unsupported LLM_PROVIDER: {provider}")


def label_rows(
    texts: Sequence[str],
    cache_path: Optional[Path] = None,
    max_cost: float = 5.0,
    client_fn: Optional[Callable[[dict], dict]] = None,
    delay: float = 0.05,
) -> pd.DataFrame:
    """Label text records with disk caching, resume capability, rate limiting, and cost cap."""
    cache_path = Path(cache_path or CACHE_FILE_DEFAULT)
    cache_path.parent.mkdir(parents=True, exist_ok=True)

    # 1. Load existing cache
    cached_records: dict[str, dict] = {}
    if cache_path.exists():
        with open(cache_path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line:
                    try:
                        obj = json.loads(line)
                        if "key" in obj and "response" in obj:
                            cached_records[obj["key"]] = obj["response"]
                    except json.JSONDecodeError:
                        continue

    provider, model, api_key = _get_api_credentials()
    accumulated_cost = 0.0
    results: list[dict] = []

    for idx, text in enumerate(texts):
        key = _compute_text_hash(text)

        # Check cache
        if key in cached_records:
            resp_dict = cached_records[key]
            results.append({
                "key": key,
                "text": text,
                **resp_dict,
            })
            continue

        # Check cost threshold
        if accumulated_cost >= max_cost:
            log.warning("Cost cap of $%.2f reached. Halting live labeling at %d texts.", max_cost, idx)
            break

        prompt = build_prompt(text)

        # Execute API call with retries
        success = False
        parsed_resp = None
        for attempt in range(3):
            try:
                if client_fn is not None:
                    raw_out = client_fn(prompt)
                else:
                    raw_out = _execute_api_call(prompt, provider, model, api_key)

                content = raw_out["content"] if isinstance(raw_out, dict) else str(raw_out)
                p_tokens = raw_out.get("prompt_tokens", 800) if isinstance(raw_out, dict) else 800
                c_tokens = raw_out.get("completion_tokens", 60) if isinstance(raw_out, dict) else 60

                call_cost = (p_tokens / 1_000_000 * INPUT_TOKEN_COST_PER_M) + (
                    c_tokens / 1_000_000 * OUTPUT_TOKEN_COST_PER_M
                )
                accumulated_cost += call_cost

                parsed_resp = json.loads(content)
                success = True
                break
            except Exception as e:
                log.warning("API attempt %d failed for key %s: %s", attempt + 1, key[:8], e)
                time.sleep(2.0 ** attempt)

        if success and parsed_resp is not None:
            # Append to cache
            cache_entry = {
                "key": key,
                "text": text,
                "response": parsed_resp,
                "model": model,
                "ts": datetime.now(timezone.utc).isoformat(),
            }
            with open(cache_path, "a", encoding="utf-8") as f:
                f.write(json.dumps(cache_entry, ensure_ascii=False) + "\n")

            cached_records[key] = parsed_resp
            results.append({
                "key": key,
                "text": text,
                **parsed_resp,
            })
            time.sleep(delay)
        else:
            log.error("Failed to obtain response for text key %s", key)

    return pd.DataFrame(results)


# ── 4. Calibration Gate on Real Sample ────────────────────────────────────────

def calibrate_prompt_on_real(
    sample_df: Optional[pd.DataFrame] = None,
    client_fn: Optional[Callable[[dict], dict]] = None,
    min_macro_f1: float = 0.80,
    cache_path: Optional[Path] = None,
) -> dict:
    """Label the 86 real rows of data/sample.csv and verify the prompt satisfies the quality gate.

    GATE: label Macro-F1 must be >= 0.80.
    """
    if sample_df is None:
        sample_df = load_sample()

    texts = sample_df["text"].tolist()
    y_true_a = sample_df["label"].to_numpy(dtype=int)
    Y_true_b = techniques_to_multihot(sample_df["techniques"])
    y_true_c = sample_df["action"].to_numpy(dtype=int)

    log.info("Calibrating prompt on %d labeled rows (zero-shot, text only)...", len(texts))
    df_pred = label_rows(texts, cache_path=cache_path, client_fn=client_fn)

    if len(df_pred) < len(texts):
        raise RuntimeError(
            f"Could only label {len(df_pred)} of {len(texts)} sample rows. "
            "Ensure API key is funded and valid."
        )

    y_pred_a = df_pred["label"].fillna(0).astype(int).to_numpy()
    tech_series = df_pred["techniques"].apply(
        lambda ts: "|".join(ts) if isinstance(ts, list) and ts else "NONE"
    )
    Y_pred_b = techniques_to_multihot(tech_series)
    y_pred_c = df_pred["action"].fillna(0).astype(int).to_numpy()

    f_a = macro_f1_multiclass(y_true_a, y_pred_a)
    f_b = macro_f1_techniques(Y_true_b, Y_pred_b)
    f_c = macro_f1_multiclass(y_true_c, y_pred_c)
    cm = confusion_matrix(y_true_a, y_pred_a, labels=list(range(config.NUM_CLASSES)))

    log.info("Calibration Results: Label F1=%.4f, Techniques F1=%.4f, Action F1=%.4f", f_a, f_b, f_c)
    log.info("Label Confusion Matrix:\n%s", cm)

    if f_a < min_macro_f1:
        # Identify worst confusions
        confusions = []
        for i in range(config.NUM_CLASSES):
            for j in range(config.NUM_CLASSES):
                if i != j and cm[i, j] > 0:
                    confusions.append((i, j, cm[i, j]))
        confusions.sort(key=lambda x: x[2], reverse=True)
        conf_str = ", ".join(f"True {c[0]} -> Pred {c[1]} (n={c[2]})" for c in confusions[:5])

        raise RuntimeError(
            f"CALIBRATION GATE FAILED: Label Macro-F1 {f_a:.4f} < {min_macro_f1:.2f}. "
            f"Worst confusions: {conf_str}. Prompt must be revised before labeling test templates!"
        )

    log.info("CALIBRATION GATE PASSED: Label Macro-F1 %.4f >= %.2f", f_a, min_macro_f1)
    return {
        "F_A": f_a,
        "F_B": f_b,
        "F_C": f_c,
        "confusion_matrix": cm,
    }


# ── 5. Consistency Filtering ──────────────────────────────────────────────────

def filter_labels(df: pd.DataFrame) -> tuple[pd.DataFrame, dict[str, int]]:
    """Filter LLM-generated records for schema validity, confidence, and rule consistency.

    Counts and reports reasons for dropped records:
    - invalid_schema: missing required fields or non-integer labels
    - low_confidence: confidence < 0.70
    - inconsistent_rules: violates cross-tab consistency (0/5 require NONE+0; 1-4 require >=1 tech + action!=0)
    - duplicate: duplicate text or hash key
    """
    reasons = {
        "invalid_schema": 0,
        "low_confidence": 0,
        "inconsistent_rules": 0,
        "duplicate": 0,
    }
    if df.empty:
        return df, reasons

    valid_indices = []
    seen_keys = set()

    for idx, row in df.iterrows():
        key = row.get("key") or _compute_text_hash(row.get("text", ""))
        if key in seen_keys:
            reasons["duplicate"] += 1
            continue

        label = row.get("label")
        action = row.get("action")
        techniques = row.get("techniques")
        conf = row.get("confidence")

        # 1. Schema check
        if (
            label is None
            or action is None
            or conf is None
            or pd.isna(label)
            or pd.isna(action)
            or pd.isna(conf)
            or not isinstance(techniques, list)
        ):
            reasons["invalid_schema"] += 1
            continue

        try:
            label_int = int(label)
            action_int = int(action)
            if float(label) != label_int or float(action) != action_int:
                reasons["invalid_schema"] += 1
                continue
            if not (0 <= label_int <= 5 and 0 <= action_int <= 5):
                reasons["invalid_schema"] += 1
                continue
        except (ValueError, TypeError):
            reasons["invalid_schema"] += 1
            continue

        # 2. Confidence check
        try:
            conf_val = float(conf)
            if conf_val < 0.70:
                reasons["low_confidence"] += 1
                continue
        except (ValueError, TypeError):
            reasons["invalid_schema"] += 1
            continue

        # 3. Consistency rules
        # Labels 0 and 5 must have no techniques and action == 0
        if label_int in (0, 5):
            if len(techniques) > 0 or action_int != 0:
                reasons["inconsistent_rules"] += 1
                continue
        # Labels 1 to 4 must have at least one technique and action != 0
        elif label_int in (1, 2, 3, 4):
            if len(techniques) == 0 or action_int == 0:
                reasons["inconsistent_rules"] += 1
                continue

        seen_keys.add(key)
        valid_indices.append(idx)

    filtered_df = df.loc[valid_indices].reset_index(drop=True)
    return filtered_df, reasons


# ── 6. CLI Execution ──────────────────────────────────────────────────────────

def main():
    parser = argparse.ArgumentParser(description="LLM-based test template labeling.")
    parser.add_argument("--calibrate", action="store_true", help="Run calibration gate on real sample")
    parser.add_argument("--label", action="store_true", help="Label selected test templates")
    parser.add_argument("--n", type=int, default=3000, help="Number of test templates to sample (default: 3000)")
    parser.add_argument("--max-cost", type=float, default=5.0, help="Maximum allowed API cost in USD (default: 5.0)")
    parser.add_argument("--cache-path", type=str, default=str(CACHE_FILE_DEFAULT), help="Path to jsonl cache file")
    args = parser.parse_args()

    if args.calibrate:
        res = calibrate_prompt_on_real()
        print(f"Calibration successful: Label F1 = {res['F_A']:.4f}")
    elif args.label:
        test_df = load_test()
        print(f"Loaded {len(test_df)} test rows. Selecting up to {args.n} templates...")
        templates = select_templates(test_df, n=args.n)
        print(f"Selected {len(templates)} diverse templates.")
        labeled_df = label_rows(templates["text"].tolist(), cache_path=Path(args.cache_path), max_cost=args.max_cost)
        filtered_df, reasons = filter_labels(labeled_df)
        print(f"Labeling complete. Raw: {len(labeled_df)}, Filtered: {len(filtered_df)}")
        print("Dropped reasons:", reasons)
    else:
        parser.print_help()


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/pseudo_label.py
"""Confidence-gated test set pseudo-labeling pipeline (T-032).

Extracts high-confidence, class-balanced pseudo-labeled rows from test.csv
using Model 1 (+ Model 2 agreement when available), applies rule consistency,
filters noise-only records, and formats output for leak-free cross-validation.
"""
from __future__ import annotations

import argparse
import json
import logging
from pathlib import Path
from typing import Optional, Sequence

import numpy as np
import pandas as pd

from src import config
from src.data_loader import load_sample, load_test, techniques_to_multihot
from src.features.tfidf_features import build_text_dict
from src.postprocess import apply_rules_batch
from src.prediction import fit_all
from src.preprocessing import preprocess
from src.training.train_llm_aug import (
    apply_class_cap,
    compute_near_real_ids,
    load_extra,
    run_cv_experiment,
)

log = logging.getLogger(__name__)
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")

PSEUDO_PATH_DEFAULT = config.ROOT / "data" / "extra" / "pseudo_labels_v1.jsonl"


def mine_pseudo_labels(
    df_test: pd.DataFrame,
    fitted_model,
    conf_threshold: float = 0.85,
    max_per_class: int = 100,
    m2_test: Optional[dict] = None,
    seed: int = config.SEED,
    weight: float = 0.20,
) -> pd.DataFrame:
    """Mine high-confidence pseudo-labels from test data.

    Parameters:
    -----------
    df_test: DataFrame with 'id' and 'text' columns
    fitted_model: Trained FittedModel from fit_all()
    conf_threshold: Minimum Task A probability required for inclusion (default: 0.85)
    max_per_class: Maximum number of rows retained per class (balanced sampling)
    m2_test: Optional Model 2 test predictions dict with 'A', 'B', 'C'
    seed: Random seed for deterministic selection
    weight: Sample weight assigned to pseudo-labeled rows in downstream training
    """
    if df_test.empty:
        return pd.DataFrame(columns=["id", "text", "label", "techniques", "action", "source", "weight", "confidence"])

    # 1. Pre-filter noise-only, empty, or very short messages
    log.info("Filtering degenerate/noise-only rows from %d candidates...", len(df_test))
    clean_indices = []
    clean_texts = []
    for idx, (row_id, text) in enumerate(zip(df_test["id"], df_test["text"])):
        t_str = str(text).strip() if pd.notna(text) else ""
        if len(t_str) < 12:
            continue
        p = preprocess(t_str)
        if p["noise_only"]:
            continue
        clean_indices.append(idx)
        clean_texts.append(t_str)

    sub_test = df_test.iloc[clean_indices].copy().reset_index(drop=True)
    if sub_test.empty:
        log.warning("No test candidates passed pre-filtering.")
        return pd.DataFrame(columns=["id", "text", "label", "techniques", "action", "source", "weight", "confidence"])

    # 2. Extract Task A/B/C probabilities via fitted Model 1
    d = build_text_dict(sub_test)
    d["struct_feats"] = None
    X = fitted_model.block.transform(d)

    p_a = fitted_model.head_a.predict_proba(X)
    p_b = fitted_model.head_b.predict_proba(X)
    p_c = fitted_model.head_c.predict_proba(X)

    # If Model 2 predictions match indices, require agreement
    if m2_test is not None and "A" in m2_test:
        m2_a_full = np.asarray(m2_test["A"])
        # Only use M2 if it aligns with test indices
        if len(m2_a_full) == len(df_test):
            m2_a_sub = m2_a_full[clean_indices]
            # Agreement: argmax of M1 and M2 must match
            m1_pred = p_a.argmax(axis=1)
            m2_pred = m2_a_sub.argmax(axis=1)
            agreement = (m1_pred == m2_pred)
            # Combine probabilities
            p_a = 0.6 * p_a + 0.4 * m2_a_sub
        else:
            agreement = np.ones(len(sub_test), dtype=bool)
    else:
        agreement = np.ones(len(sub_test), dtype=bool)

    # 3. Confidence filter
    max_prob = p_a.max(axis=1)
    pred_label = p_a.argmax(axis=1)

    qualifies = (max_prob >= conf_threshold) & agreement
    qualified_indices = np.where(qualifies)[0]

    log.info(
        "Candidate filtering: %d / %d qualified (conf >= %.2f, agreement=%s)",
        len(qualified_indices), len(sub_test), conf_threshold, m2_test is not None,
    )

    if len(qualified_indices) == 0:
        return pd.DataFrame(columns=["id", "text", "label", "techniques", "action", "source", "weight", "confidence"])

    # 4. Apply multi-task postprocessing rules
    thr_b = np.array(getattr(fitted_model.head_b, "thresholds", [0.35] * 6), dtype=float)
    bin_b = (p_b[qualified_indices] >= thr_b).astype(int)
    pred_c = p_c[qualified_indices].argmax(axis=1)

    labels_post, tech_post, act_post = apply_rules_batch(
        pred_label[qualified_indices],
        bin_b,
        pred_c,
        p_b[qualified_indices],
        p_c[qualified_indices],
    )

    # Convert binary techniques to comma-separated format
    tech_strings = []
    for row in tech_post:
        active = [f"T{j+1}" for j, v in enumerate(row) if v == 1]
        tech_strings.append(",".join(active) if active else "NONE")

    selected_df = pd.DataFrame({
        "id": sub_test.iloc[qualified_indices]["id"].values,
        "text": sub_test.iloc[qualified_indices]["text"].values,
        "label": labels_post.astype(int),
        "techniques": tech_strings,
        "action": act_post.astype(int),
        "source": "pseudo",
        "weight": float(weight),
        "confidence": max_prob[qualified_indices],
    })

    # 5. Deterministic balanced class capping (prioritizing highest confidence per class)
    capped_parts = []
    for lab in sorted(selected_df["label"].unique()):
        class_df = selected_df[selected_df["label"] == lab].sort_values(by="confidence", ascending=False)
        capped_parts.append(class_df.head(max_per_class))

    final_df = pd.concat(capped_parts, ignore_index=True)
    counts = dict(final_df["label"].value_counts())
    log.info("Mined %d pseudo-labeled rows across classes: %s", len(final_df), counts)
    return final_df


def save_pseudo_labels(df: pd.DataFrame, path: Path = PSEUDO_PATH_DEFAULT) -> None:
    """Save pseudo-labeled rows to JSONL format."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        for _, row in df.iterrows():
            record = {
                "id": int(row["id"]),
                "text": str(row["text"]),
                "response": {
                    "label": int(row["label"]),
                    "techniques": row["techniques"].split(",") if row["techniques"] != "NONE" else [],
                    "action": int(row["action"]),
                    "confidence": float(row.get("confidence", 0.90)),
                },
                "source": "pseudo",
                "weight": float(row.get("weight", 0.20)),
            }
            f.write(json.dumps(record, ensure_ascii=False) + "\n")
    log.info("Saved %d pseudo-labeled rows to %s", len(df), path)


def load_pseudo_labels(path: Path = PSEUDO_PATH_DEFAULT, weight: float = 0.20) -> pd.DataFrame:
    """Load pseudo-labeled rows from JSONL file."""
    path = Path(path)
    if not path.exists():
        log.warning("Pseudo-labels file not found: %s", path)
        return pd.DataFrame(columns=["id", "text", "label", "techniques", "action", "source", "weight"])

    return load_extra(path, default_weight=weight)


In [ ]:
%%writefile src/submission.py
"""Assemble + validate the 5-column submission DataFrame.

Owned by Teammate B (T-010).

Contract: build(ids, label, tech, action, conf) -> DataFrame
Raises ValueError on any invalid value before returning.
"""
import logging

import numpy as np
import pandas as pd

from src import config
from src.data_loader import multihot_to_techniques

log = logging.getLogger(__name__)

_VALID_LABELS  = set(range(config.NUM_CLASSES))
_VALID_ACTIONS = set(range(config.NUM_ACTIONS))


def build(ids, label, tech, action, conf) -> pd.DataFrame:
    """Assemble strict 5-column submission frame and validate all values.

    ids:    array-like of id values
    label:  int array (n,) — 0..5
    tech:   (n, 6) binary int matrix (multihot format)
    action: int array (n,) — 0..5
    conf:   float array (n,) — clipped to config.CONF_CLIP before returning
    Returns: DataFrame with columns [id, label, techniques, action, confidence]
    Raises: ValueError on any invalid value.
    """
    label  = np.asarray(label,  dtype=int)
    action = np.asarray(action, dtype=int)
    tech   = np.asarray(tech,   dtype=int)
    conf   = np.clip(np.asarray(conf, dtype=float), *config.CONF_CLIP)
    n      = len(ids)

    if tech.shape != (n, config.NUM_CLASSES):
        raise ValueError(
            f"tech shape must be ({n}, {config.NUM_CLASSES}), got {tech.shape}"
        )

    bad_labels = set(label.tolist()) - _VALID_LABELS
    if bad_labels:
        raise ValueError(f"Invalid label values: {bad_labels}")

    bad_actions = set(action.tolist()) - _VALID_ACTIONS
    if bad_actions:
        raise ValueError(f"Invalid action values: {bad_actions}")

    if np.any(np.isnan(conf)):
        raise ValueError("confidence array contains NaN")

    techniques_col = [multihot_to_techniques(row) for row in tech]

    return pd.DataFrame({
        "id":         list(ids),
        "label":      label.tolist(),
        "techniques": techniques_col,
        "action":     action.tolist(),
        "confidence": conf.round(4).tolist(),
    })


In [ ]:
%%writefile src/prediction.py
"""End-to-end pipeline: fit on all labeled data, predict test, write submission.

Owned by Teammate B (T-010, Phase 2 E015, Phase 3 T-030/T-031 E021 M1+M2 ensemble).

Usage:
    python -m src.prediction
"""
import json
import logging
import time
from datetime import datetime, timezone
from pathlib import Path
from typing import Optional, Sequence

import numpy as np
import pandas as pd

from src import config
from src.augment import expand
from src.data_loader import (load_sample, load_submission_template, load_test,
                             techniques_to_multihot)
from src.evaluation.calibration import apply as apply_calibration, fit_confidence
from src.evaluation.metrics import (log_experiment, macro_f1_multiclass,
                                    macro_f1_techniques, s_hat)
from src.features.tfidf_features import TfidfBlock, build_text_dict
from src.models.ensemble import blend, model_disagreement, tune_blend_weight
from src.models.task_a import TaskAHead
from src.models.task_b import TaskBHead
from src.models.task_c import TaskCHead
from src.postprocess import apply_rules_batch, tune_task_b_thresholds
from src.preprocessing import preprocess
from src.submission import build
from src.training.cross_validation import cross_validate, leave_cluster_out_cv

log = logging.getLogger(__name__)
logging.basicConfig(level=logging.INFO,
                    format="%(asctime)s %(levelname)s %(message)s")

CHUNK = 50_000   # rows per prediction chunk to bound memory

# Promoted config: word+char TF-IDF, Set A n_aug=5, LR+SVM+M2 blend, calibrated
_AUG_LEVELS = ("light", "medium", "heavy")
_AUG_WEIGHTS = (1.0, 0.8, 0.5)
DEFAULT_N_AUG = 5
DEFAULT_C = 5.0
DEFAULT_SVM_C = 1.0
DEFAULT_BLEND_W = 0.4   # 0.4 * LR + 0.6 * LinearSVC


# ── Fitted-model container ────────────────────────────────────────────────────

class FittedModel:
    """Container holding trained components for prediction."""

    def __init__(self, block: TfidfBlock,
                 head_a: TaskAHead,
                 head_b: TaskBHead,
                 head_c: TaskCHead,
                 head_a_svm: TaskAHead = None,
                 blend_w: float = DEFAULT_BLEND_W,
                 calibrator: dict = None):
        self.block = block
        self.head_a = head_a
        self.head_a_svm = head_a_svm
        self.head_b = head_b
        self.head_c = head_c
        self.blend_w = blend_w
        self.calibrator = calibrator


def _expand_train(df_labeled: pd.DataFrame, n_aug: int, seed: int,
                  extra_df: pd.DataFrame = None, extra_weight: float = 0.3,
                  aug_extra: bool = True) -> tuple:
    """Expand training rows with Set A corruption; return (full_df, sample_weights)."""
    parts = [df_labeled.assign(weight=1.0)]
    for i in range(n_aug):
        lvl = _AUG_LEVELS[i % len(_AUG_LEVELS)]
        w = _AUG_WEIGHTS[i % len(_AUG_WEIGHTS)]
        parts.append(expand(df_labeled, k=1, seed=seed + 100 * (i + 1),
                            level=lvl, ops="A", weight=w))
    if extra_df is not None and not extra_df.empty:
        extra_base = extra_df.assign(weight=float(extra_weight))
        parts.append(extra_base)
        if aug_extra and n_aug > 0:
            for i in range(n_aug):
                lvl = _AUG_LEVELS[i % len(_AUG_LEVELS)]
                w = _AUG_WEIGHTS[i % len(_AUG_WEIGHTS)] * float(extra_weight) * 0.5
                parts.append(expand(extra_df, k=1, seed=seed + 200 * (i + 1),
                                    level=lvl, ops="A", weight=w))
    full_df = pd.concat(parts, ignore_index=True)
    return full_df, full_df["weight"].values.astype(float)


def _noise_stats(texts) -> tuple:
    """Return (noise_ratio (n,), noise_only (n,)) arrays matching cross_validation.py."""
    ratios, only = [], []
    for t in texts:
        p = preprocess(t)
        n_tok = max(len(str(t).split()), 1)
        ratios.append(min(1.0, float(p["n_noise_tokens"]) / n_tok))
        only.append(bool(p["noise_only"]))
    return np.array(ratios, dtype=float), np.array(only, dtype=bool)


# ── Core API ──────────────────────────────────────────────────────────────────

def fit_all(df_labeled: pd.DataFrame,
            df_test: pd.DataFrame = None,
            seed: int = config.SEED,
            n_aug: int = DEFAULT_N_AUG,
            C: float = DEFAULT_C,
            svm_C: float = DEFAULT_SVM_C,
            blend_w: float = DEFAULT_BLEND_W,
            task_b_thresholds: list = None,
            calibrator: dict = None,
            extra_df: pd.DataFrame = None,
            extra_weight: float = 0.3,
            aug_extra: bool = True) -> FittedModel:
    """Fit TF-IDF + all task heads on labeled data (+ Set A augmentation).

    df_labeled:        DataFrame with text, label, techniques, action
    df_test:           optional DataFrame with text — used only for unsupervised TF-IDF IDF
    extra_df:          optional DataFrame with external/LLM labeled rows
    """
    config.set_seed(seed)
    full_train, weights = _expand_train(df_labeled, n_aug=n_aug, seed=seed,
                                        extra_df=extra_df, extra_weight=extra_weight,
                                        aug_extra=aug_extra)

    train_d = build_text_dict(full_train)
    train_d["struct_feats"] = None
    test_d = build_text_dict(df_test) if df_test is not None else None
    if test_d is not None:
        test_d["struct_feats"] = None

    block = TfidfBlock(use_msg_blocks=False)
    X     = block.fit_transform(train_d, test_d)

    y_a = full_train["label"].values
    Y_b = techniques_to_multihot(full_train["techniques"])
    y_c = full_train["action"].values

    head_a     = TaskAHead(C=C, model_type="lr", seed=seed).fit(X, y_a, sample_weight=weights)
    head_a_svm = TaskAHead(C=svm_C, model_type="svm", seed=seed).fit(X, y_a, sample_weight=weights)
    head_b     = TaskBHead(C=C, seed=seed).fit(X, Y_b, sample_weight=weights)
    if task_b_thresholds is not None:
        head_b.set_thresholds(task_b_thresholds)
    head_c = TaskCHead(C=C, seed=seed).fit(X, y_c, sample_weight=weights)

    block.save(config.CACHE_DIR / "tfidf_block.pkl")
    log.info("fit_all: %d labeled rows (%d with aug), %s test rows",
             len(df_labeled), len(full_train),
             len(df_test) if df_test is not None else "none")
    return FittedModel(block, head_a, head_b, head_c,
                       head_a_svm=head_a_svm, blend_w=blend_w, calibrator=calibrator)


def predict_df(df_text: pd.DataFrame,
               fitted: FittedModel,
               m2_test: dict = None,
               w_m1_a: float = 0.7,
               w_m1_b: float = 0.3,
               w_m1_c: float = 0.4) -> pd.DataFrame:
    """Predict on arbitrary new text; works on any input (empty, noise-only, conversations).

    df_text: DataFrame with 'id' and 'text' columns
    fitted:  FittedModel from fit_all()
    m2_test: optional dict/npz with Model 2 test arrays ('A', 'B', 'C', 'ood_sim')
    Returns: 5-column submission DataFrame
    """
    all_preds = []
    n = len(df_text)
    for start in range(0, n, CHUNK):
        end = min(start + CHUNK, n)
        chunk = df_text.iloc[start:end].copy()
        d     = build_text_dict(chunk)
        d["struct_feats"] = None
        X     = fitted.block.transform(d)

        proba_lr = fitted.head_a.predict_proba(X)
        if fitted.head_a_svm is not None:
            proba_svm = fitted.head_a_svm.predict_proba(X)
            m1_a = blend([proba_lr, proba_svm], weights=[fitted.blend_w, 1.0 - fitted.blend_w])
            dis = model_disagreement(proba_lr, proba_svm)
        else:
            m1_a = proba_lr
            dis = None

        m1_b = fitted.head_b.predict_proba(X)
        m1_c = fitted.head_c.predict_proba(X)

        if m2_test is not None:
            m2_a = np.asarray(m2_test["A"][start:end], dtype=float)
            m2_b = np.asarray(m2_test["B"][start:end], dtype=float)
            m2_c = np.asarray(m2_test["C"][start:end], dtype=float)
            proba_a = blend([m1_a, m2_a], weights=[w_m1_a, 1.0 - w_m1_a])
            proba_b = blend([m1_b, m2_b], weights=[w_m1_b, 1.0 - w_m1_b])
            proba_c = blend([m1_c, m2_c], weights=[w_m1_c, 1.0 - w_m1_c])
            dis = model_disagreement(m1_a, m2_a)
            ood = np.asarray(m2_test["ood_sim"][start:end], dtype=float)
        else:
            proba_a, proba_b, proba_c = m1_a, m1_b, m1_c
            ood = None

        label  = proba_a.argmax(axis=1)
        thr    = np.array(fitted.head_b.thresholds, dtype=float)
        tech   = (proba_b >= thr).astype(int)
        action = proba_c.argmax(axis=1)
        label, tech, action = apply_rules_batch(label, tech, action, proba_b, proba_c)

        if fitted.calibrator is not None:
            nr, no = _noise_stats(chunk["text"])
            cols = [nr]
            if dis is not None:
                cols.append(dis)
            if ood is not None:
                cols.append(ood)
            extra = np.column_stack(cols)
            conf = apply_calibration(fitted.calibrator, proba_a, extra_feats=extra, noise_only=no)
        else:
            conf = np.clip(proba_a.max(axis=1), *config.CONF_CLIP)

        all_preds.append(build(chunk["id"], label, tech, action, conf))
        log.info("predict_df: %d / %d rows done", end, n)

    return pd.concat(all_preds, ignore_index=True)


# ── Cross-validation helper ───────────────────────────────────────────────────

class _CVModel:
    """Wrapper around TfidfBlock + TaskA/B/C heads for cross_validation.cross_validate."""

    def __init__(self, seed: int = config.SEED,
                 n_aug: int = DEFAULT_N_AUG,
                 C: float = DEFAULT_C,
                 model_type: str = "lr",
                 extra_df: pd.DataFrame = None,
                 extra_weight: float = 0.3,
                 aug_extra: bool = True,
                 all_real_ids: Sequence[int] = None):
        self.seed = seed
        self.n_aug = n_aug
        self.C = C
        self.model_type = model_type
        self.extra_df = extra_df
        self.extra_weight = extra_weight
        self.aug_extra = aug_extra
        self.all_real_ids = all_real_ids

    def fit(self, train: pd.DataFrame) -> "_CVModel":
        safe_extras = None
        if self.extra_df is not None and not self.extra_df.empty:
            if self.all_real_ids is not None:
                val_ids = set(self.all_real_ids) - set(train["id"])
            else:
                val_ids = set()

            if val_ids and "near_real_ids" in self.extra_df.columns:
                def _safe(row):
                    near = row.get("near_real_ids")
                    if near is None or (isinstance(near, float) and np.isnan(near)):
                        return True
                    return not (set(near) & val_ids)

                safe_extras = self.extra_df[self.extra_df.apply(_safe, axis=1)].copy()
            else:
                safe_extras = self.extra_df.copy()

        full_train, weights = _expand_train(
            train, n_aug=self.n_aug, seed=self.seed,
            extra_df=safe_extras, extra_weight=self.extra_weight, aug_extra=self.aug_extra
        )
        train_d = build_text_dict(full_train)
        train_d["struct_feats"] = None
        self.block = TfidfBlock(use_msg_blocks=False)
        X_tr = self.block.fit_transform(train_d)   # only train; test IDF not used in CV

        y_a = full_train["label"].values
        Y_b = techniques_to_multihot(full_train["techniques"])
        y_c = full_train["action"].values

        self.head_a = TaskAHead(C=self.C, model_type=self.model_type, seed=self.seed).fit(
            X_tr, y_a, sample_weight=weights
        )
        self.head_b = TaskBHead(C=self.C, seed=self.seed).fit(X_tr, Y_b, sample_weight=weights)
        self.head_c = TaskCHead(C=self.C, seed=self.seed).fit(X_tr, y_c, sample_weight=weights)
        return self

    def predict(self, valid: pd.DataFrame) -> dict:
        valid_d = build_text_dict(valid)
        valid_d["struct_feats"] = None
        X_va = self.block.transform(valid_d)
        proba_a = self.head_a.predict_proba(X_va)
        proba_b = self.head_b.predict_proba(X_va)
        proba_c = self.head_c.predict_proba(X_va)
        return {
            "proba":   proba_a,
            "proba_b": proba_b,
            "proba_c": proba_c,
            "label":   proba_a.argmax(1),
            "tech":    self.head_b.predict(X_va),
            "action":  proba_c.argmax(1),
        }


def _cv_oof(df: pd.DataFrame, seed: int = config.SEED,
            exp_name: str = "E015", C: float = DEFAULT_C, model_type: str = "lr",
            extra_df: pd.DataFrame = None, extra_weight: float = 0.25,
            aug_extra: bool = True, all_real_ids: Sequence[int] = None):
    """Run repeated grouped CV. Returns (raw_df, summary_df, oof_bundle)."""
    return cross_validate(
        df,
        lambda: _CVModel(
            seed=seed,
            C=C,
            model_type=model_type,
            extra_df=extra_df,
            extra_weight=extra_weight,
            aug_extra=aug_extra,
            all_real_ids=all_real_ids,
        ),
        n_splits=config.N_SPLITS,
        n_repeats=config.N_REPEATS,
        exp_name=exp_name,
        model_tag="m1",
        return_oof=True,
    )


# ── Main entrypoint ───────────────────────────────────────────────────────────

def main(
    exp_lr: str = "E026",
    exp_svm: str = "E027",
    m2_exp: Optional[str] = "E020",
    ens_exp: str = "E028",
    use_extra: bool = True,
    extra_paths: Optional[Sequence[Path]] = None,
    require_unpadded_m2: bool = True,
    test_limit: Optional[int] = None,
) -> str:
    """Fit on all labeled data, blend M1+M2 if available, calibrate, predict test, write submission."""
    t0 = time.time()
    df       = load_sample()
    test     = load_test()
    if test_limit is not None and test_limit < len(test):
        test = test.head(test_limit).copy()
    template = load_submission_template()
    if len(template) != len(test):
        template = template.head(len(test)).copy()
    all_real_ids = df["id"].values.tolist()

    real_tech_count = techniques_to_multihot(df["techniques"]).sum(axis=1)
    log.info("Real rows label x technique-count x action==0:\n%s",
             pd.crosstab(df["label"], real_tech_count,
                         values=df["action"].eq(0), aggfunc="mean").round(3))

    # 1. Load promoted extra data (LLM + pseudo-labels) if enabled
    processed_extra = None
    if use_extra:
        from src.pseudo_label import load_pseudo_labels
        from src.training.train_llm_aug import (apply_class_cap,
                                               compute_near_real_ids,
                                               load_extra)

        parts = []
        if extra_paths:
            for p in extra_paths:
                parts.append(load_extra(Path(p), default_weight=0.25))
        else:
            llm_p = config.ROOT / "data" / "extra" / "llm_labels_v1.jsonl"
            pseudo_p = config.ROOT / "data" / "extra" / "pseudo_labels_v1.jsonl"
            if llm_p.exists():
                parts.append(load_extra(llm_p, default_weight=0.30))
            if pseudo_p.exists():
                parts.append(load_pseudo_labels(pseudo_p, weight=0.20))

        if parts:
            comb = pd.concat(parts, ignore_index=True)
            capped = apply_class_cap(comb, max_per_class=400, seed=config.SEED)
            processed_extra = compute_near_real_ids(capped, df)
            near_count = (processed_extra["near_real_ids"].apply(len) > 0).sum()
            log.info("Loaded %d extra rows for training (%d with near-real matches)",
                     len(processed_extra), near_count)

            filtered_llm = processed_extra[
                processed_extra["source"].astype(str).eq("llm")
                & processed_extra["near_real_ids"].apply(lambda ids: len(ids) == 0)
            ]
            if not filtered_llm.empty:
                llm_tech_count = techniques_to_multihot(filtered_llm["techniques"]).sum(axis=1)
                log.info("Filtered LLM rows label x technique-count x action==0:\n%s",
                         pd.crosstab(filtered_llm["label"], llm_tech_count,
                                     values=filtered_llm["action"].eq(0), aggfunc="mean").round(3))

    log.info("Running %d-fold × %d-repeat grouped CV (LR: %s & LinearSVC: %s)...",
             config.N_SPLITS, config.N_REPEATS, exp_lr, exp_svm)
    res_lr, sum_lr, oof_lr = _cv_oof(
        df, exp_name=exp_lr, C=DEFAULT_C, model_type="lr",
        extra_df=processed_extra, extra_weight=0.25, aug_extra=True, all_real_ids=all_real_ids
    )
    _, sum_svm, oof_svm    = _cv_oof(
        df, exp_name=exp_svm, C=DEFAULT_SVM_C, model_type="svm",
        extra_df=processed_extra, extra_weight=0.25, aug_extra=True, all_real_ids=all_real_ids
    )
    res_lr.to_csv(config.VALIDATION_DIR / f"{exp_lr}_cv.csv", index=False)

    y_a = df["label"].values
    Y_true = techniques_to_multihot(df["techniques"])
    y_c = df["action"].values
    folds_arr = oof_lr["folds"]["fold_r0"].values

    p_lr, p_svm = oof_lr["clean"]["proba"], oof_svm["clean"]["proba"]
    w_lr, fa_m1, m1_a = tune_blend_weight(p_lr, p_svm, y_a, macro_f1_multiclass)
    log.info("LR/SVM OOF blend weight: w_lr=%.2f, F_A=%.4f", w_lr, fa_m1)
    m1_b = oof_lr["clean"]["proba_b"]
    m1_c = oof_lr["clean"]["proba_c"]

    m2_oof_path = (config.OOF_DIR / f"{m2_exp}_m2.npz") if m2_exp else None
    m2_test_path = (config.PREDICTION_DIR / f"{m2_exp}_m2_test.npz") if m2_exp else None
    nr, no = _noise_stats(df["text"])

    m2_test = None
    if m2_oof_path and m2_test_path and m2_oof_path.exists() and m2_test_path.exists():
        log.info("Loading Model 2 OOF & test probabilities (%s_m2)...", m2_exp)
        m2_oof = np.load(m2_oof_path, allow_pickle=True)
        m2_test_data = np.load(m2_test_path, allow_pickle=True)
        assert np.array_equal(m2_oof["ids"], df["id"].to_numpy(dtype=np.int64))
        assert np.array_equal(m2_test_data["ids"], test["id"].to_numpy(dtype=np.int64))

        # Check padding on Model 2 test arrays
        A_test = m2_test_data["A"]
        n_padded = int((A_test == A_test[-1]).all(axis=1).sum())
        if n_padded > 1000:
            msg = (
                f"Model 2 test arrays in {m2_test_path.name} are padded ({n_padded} identical rows)! "
                "To generate full unpadded embeddings on Kaggle GPU, run:\n"
                "    python -m src.training.train_embeddings --full-test"
            )
            if require_unpadded_m2:
                log.error(msg)
                raise RuntimeError(msg)
            else:
                log.warning("%s\nProceeding with M1 (LR+SVM) test predictions to prevent class skew.", msg)
                m2_test = None
        else:
            m2_test = m2_test_data

        w_m1_a, fa_ens, ens_a = tune_blend_weight(m1_a, m2_oof["A"], y_a, macro_f1_multiclass)
        w_m1_b, _, ens_b = tune_blend_weight(m1_b, m2_oof["B"], Y_true, macro_f1_techniques, is_multilabel=True)
        w_m1_c, _, ens_c = tune_blend_weight(m1_c, m2_oof["C"], y_c, macro_f1_multiclass)
        dis = model_disagreement(m1_a, m2_oof["A"])
        extra = np.column_stack([nr, dis, m2_oof["ood_sim"]])

        np.savez_compressed(
            config.OOF_DIR / f"{ens_exp}_ens.npz",
            ids=m2_oof["ids"], repeat=m2_oof["repeat"],
            A=ens_a, B=ens_b, C=ens_c,
        )
        log.info("M1+M2 Ensemble OOF F_A=%.4f (w_m1_a=%.2f, w_m1_b=%.2f, w_m1_c=%.2f)",
                 fa_ens, w_m1_a, w_m1_b, w_m1_c)
    else:
        log.warning("Model 2 files not found or disabled; using M1 (LR+SVM) only.")
        ens_a, ens_b, ens_c = m1_a, m1_b, m1_c
        w_m1_a = w_m1_b = w_m1_c = 1.0
        dis = model_disagreement(p_lr, p_svm)
        extra = np.column_stack([nr, dis])

    thrs = tune_task_b_thresholds(ens_b, Y_true, folds_arr)
    nested_scores = []
    for f in np.unique(folds_arr):
        tr = folds_arr != f
        va = ~tr
        fold_thrs = np.asarray(tune_task_b_thresholds(ens_b[tr], Y_true[tr], folds_arr[tr]), dtype=float)
        pred_col = (ens_b[va] >= fold_thrs).astype(int)
        nested_scores.append(macro_f1_techniques(Y_true[va], pred_col))
    nested_fb = float(np.mean(nested_scores))
    log.info("Task B thresholds: %s (nested cross-fitted F_B estimate=%.4f)", thrs, nested_fb)

    cal = fit_confidence(ens_a, y_a, extra_feats=extra, folds=folds_arr, noise_only=no)

    # Fit final model on all labeled data (+ Set A aug + extra rows; F-07 train-only IDF)
    log.info("Fitting final model on %d labeled rows (+ %s extras)...",
             len(df), len(processed_extra) if processed_extra is not None else 0)
    fitted = fit_all(
        df,
        df_test=None,
        blend_w=w_lr,
        task_b_thresholds=thrs,
        calibrator=cal,
        extra_df=processed_extra,
        extra_weight=0.25,
        aug_extra=True,
    )

    # Predict test
    log.info("Predicting %d test rows...", len(test))
    sub = predict_df(test, fitted, m2_test=m2_test, w_m1_a=w_m1_a, w_m1_b=w_m1_b, w_m1_c=w_m1_c)

    if list(sub["id"].values) != list(template["id"].values):
        raise RuntimeError("Submission IDs do not match the template order!")
    if len(sub) != len(template):
        raise RuntimeError(
            f"Row count mismatch: {len(sub)} (ours) vs {len(template)} (template)"
        )

    exp_id   = datetime.now(timezone.utc).strftime("E%m%d_%H%M")
    out_path = config.PREDICTION_DIR / f"submission_{exp_id}.csv"
    sub.to_csv(out_path, index=False)
    elapsed  = (time.time() - t0) / 60
    log.info("Done → %s (%.1f min)", out_path, elapsed)
    log.info("Predicted class mix:\n%s",
             sub["label"].value_counts(normalize=True).sort_index().round(3))
    log.info("Predicted action mix:\n%s",
             sub["action"].value_counts(normalize=True).sort_index().round(3))
    return str(out_path)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile tools/validate_submission.py
"""Strict validator for the ScamShield BD 5-column Kaggle submission.

CLI:
    python tools/validate_submission.py <csv> [--sample data/sample_submission.csv] [--test data/test.csv]
Exit code 0 = valid, 1 = invalid (or unreadable).

Import:
    from tools.validate_submission import validate, check
    errors = validate("outputs/predictions/e001_1200.csv")          # list[str], empty = OK
    errors, warnings = check("outputs/predictions/e001_1200.csv")   # also returns warnings
"""
from __future__ import annotations

import argparse
import csv
import io
import re
import sys
from pathlib import Path
from typing import Optional, Sequence, Tuple

import pandas as pd

ROOT = Path(__file__).resolve().parents[1]
DEFAULT_SAMPLE = ROOT / "data" / "sample_submission.csv"
DEFAULT_TEST = ROOT / "data" / "test.csv"

EXPECTED_COLS = ["id", "label", "techniques", "action", "confidence"]
VALID_TECH = {f"T{i}" for i in range(1, 7)}
BOM = b"\xef\xbb\xbf"
MAX_SHOWN = 5


def _show(ids: pd.Series) -> str:
    """Format up to MAX_SHOWN offending ids."""
    shown = ", ".join(map(str, ids.head(MAX_SHOWN).tolist()))
    more = f" (+{len(ids) - MAX_SHOWN} more)" if len(ids) > MAX_SHOWN else ""
    return f"[{shown}]{more}"


def _fail(errors: list, msg: str, ids: Optional[pd.Series] = None) -> None:
    if ids is not None:
        msg = f"{msg}: {len(ids):,} row(s), e.g. ids {_show(ids)}"
    errors.append(msg)


def _read_header(path: Path) -> Optional[list]:
    with open(path, "r", encoding="utf-8-sig", newline="") as f:
        return next(csv.reader(f), None)


def _read_ids(path: Path) -> pd.Series:
    df = pd.read_csv(path, usecols=["id"], dtype=str, keep_default_na=False)
    return df["id"]


def _load_submission(path: Path, errors: list) -> Optional[pd.DataFrame]:
    """Read the raw file as strings. Returns None if it cannot be parsed."""
    try:
        raw = path.read_bytes()
    except OSError as e:
        errors.append(f"cannot read file: {e}")
        return None

    if raw.startswith(BOM):
        errors.append("file starts with a UTF-8 BOM; save as plain UTF-8 without BOM")
        raw = raw[len(BOM):]
    try:
        text = raw.decode("utf-8")
    except UnicodeDecodeError as e:
        errors.append(f"file is not valid UTF-8: {e}")
        return None

    rows = list(csv.reader(io.StringIO(text, newline="")))
    while rows and not rows[-1]:          # ignore trailing blank lines
        rows.pop()
    if not rows:
        errors.append("file is empty")
        return None

    header, body = rows[0], rows[1:]
    blank = [i + 2 for i, r in enumerate(body) if not r]
    if blank:
        errors.append(f"blank line(s) inside the file at line(s) {blank[:MAX_SHOWN]}")
        body = [r for r in body if r]

    ragged = [i + 2 for i, r in enumerate(body) if len(r) != len(header)]
    if ragged:
        errors.append(
            f"{len(ragged):,} row(s) do not have {len(header)} fields "
            f"(first at file line(s) {ragged[:MAX_SHOWN]})"
        )
        return None
    return pd.DataFrame(body, columns=header, dtype=str)


def check(
    path,
    sample: Optional[str] = None,
    test: Optional[str] = None,
) -> Tuple[list, list]:
    """Validate a submission. Returns (errors, warnings)."""
    errors: list = []
    warnings: list = []
    path = Path(path)

    df = _load_submission(path, errors)
    if df is None:
        return errors, warnings

    # 1. Header (and index-column detection)
    expected = EXPECTED_COLS
    sample_path = Path(sample) if sample else DEFAULT_SAMPLE
    if sample_path.exists():
        hdr = _read_header(sample_path)
        if hdr:
            expected = hdr
    elif sample:
        errors.append(f"sample submission not found: {sample_path}")
    else:
        warnings.append(f"sample_submission.csv not found at {sample_path}; using built-in header")

    cols = list(df.columns)
    if cols and cols[0] in ("", "Unnamed: 0", "index"):
        errors.append(f"first column {cols[0]!r} looks like a pandas index column; write with index=False")
    if cols != expected:
        errors.append(f"header {cols} != expected {expected} (names and order must match exactly)")
        return errors, warnings   # later checks need the right columns

    # 2. Row count and ids vs test.csv
    test_path = Path(test) if test else DEFAULT_TEST
    if test_path.exists():
        try:
            test_ids = _read_ids(test_path)
        except Exception as e:  # noqa: BLE001
            errors.append(f"cannot read test ids from {test_path}: {e}")
            test_ids = None
        if test_ids is not None:
            if len(df) != len(test_ids):
                errors.append(f"row count {len(df):,} != test.csv rows {len(test_ids):,}")
            ids = df["id"]
            dup = ids[ids.duplicated(keep=False)].drop_duplicates()
            if len(dup):
                _fail(errors, "duplicate ids", dup)
            test_set = set(test_ids)
            sub_set = set(ids)
            unknown = pd.Series(sorted(sub_set - test_set))
            missing = pd.Series(sorted(test_set - sub_set))
            if len(unknown):
                _fail(errors, "ids not present in test.csv", unknown)
            if len(missing):
                _fail(errors, "test ids missing from submission", missing)
    elif test:
        errors.append(f"test.csv not found: {test_path}")
    else:
        warnings.append(f"test.csv not found at {test_path}; skipped row-count and id checks")

    # 3. Empty cells (whitespace-only counts as empty)
    row_ids = df["id"]
    for col in df.columns:
        empty = df[col].str.strip() == ""
        if empty.any():
            _fail(errors, f"empty cell(s) in column '{col}'", row_ids[empty])
    nonempty = {c: df[c].str.strip() != "" for c in df.columns}

    # 4. label and action: integers 0-5 (reject 2.0, text, signs, spaces)
    for col in ("label", "action"):
        bad = nonempty[col] & ~df[col].str.fullmatch(r"[0-5]")
        if bad.any():
            _fail(errors, f"'{col}' must be an integer 0-5 (no decimals, text or spaces)", row_ids[bad])

    # 5. techniques
    def _tech_ok(v: str) -> bool:
        if v == "NONE":
            return True
        parts = v.split("|")
        return (
            all(p in VALID_TECH for p in parts)
            and len(set(parts)) == len(parts)
            and parts == sorted(parts)
        )

    tech = df["techniques"]
    bad = nonempty["techniques"] & ~tech.map(_tech_ok)
    if bad.any():
        _fail(
            errors,
            "'techniques' must be NONE or sorted unique T1-T6 joined by '|' (NONE never mixed with codes)",
            row_ids[bad],
        )

    # 6. confidence: numeric, finite, in [0, 1]
    conf = pd.to_numeric(df["confidence"], errors="coerce")
    bad = nonempty["confidence"] & ~((conf >= 0.0) & (conf <= 1.0))   # NaN/inf/text all fail
    if bad.any():
        _fail(errors, "'confidence' must be a number in [0, 1] (no NaN/inf/text)", row_ids[bad])

    # 7. Warnings only
    good_conf = conf[(conf >= 0.0) & (conf <= 1.0)]
    if len(good_conf) > 1 and good_conf.nunique() == 1:
        warnings.append(f"confidence is constant ({good_conf.iloc[0]}); calibration will not help")
    lab = df.loc[nonempty["label"] & df["label"].str.fullmatch(r"[0-5]"), "label"].astype(int)
    if len(lab):
        counts = lab.value_counts()
        never = [c for c in range(6) if c not in counts.index]
        if never:
            warnings.append(f"class(es) never predicted: {never}")
        top_share = counts.iloc[0] / len(lab)
        if top_share > 0.90:
            warnings.append(f"{top_share:.1%} of rows are predicted as class {counts.index[0]}")

    return errors, warnings


def validate(
    path,
    sample: Optional[str | Path] = None,
    test: Optional[str | Path] = None,
    sample_path: Optional[str | Path] = None,
    test_path: Optional[str | Path] = None,
) -> list:
    """Return a list of error strings; empty list means the file is valid."""
    s = sample if sample is not None else sample_path
    t = test if test is not None else test_path
    return check(path, sample=s, test=t)[0]


def main(argv: Optional[Sequence[str]] = None) -> int:
    ap = argparse.ArgumentParser(description="Validate a ScamShield BD submission CSV.")
    ap.add_argument("csv", help="submission file to check")
    ap.add_argument("--sample", default=None, help=f"sample_submission.csv (default {DEFAULT_SAMPLE})")
    ap.add_argument("--test", default=None, help=f"test.csv (default {DEFAULT_TEST})")
    args = ap.parse_args(argv)

    errors, warnings = check(args.csv, sample=args.sample, test=args.test)
    for w in warnings:
        print(f"WARNING: {w}")
    if errors:
        for e in errors:
            print(f"ERROR: {e}")
        print(f"\nFAILED: {len(errors)} problem(s) in {args.csv}")
        return 1
    print(f"OK: {args.csv} passed all checks.")
    return 0


if __name__ == "__main__":
    sys.exit(main())


In [ ]:
%%writefile data/extra/llm_labels_v1.jsonl
{"id": 90001, "text": "bKash support: your account will be paused tomorrow. Reply with the 6-digit code we just sent to confirm you are the owner.", "response": {"label": 1, "techniques": ["T1", "T2", "T5"], "action": 1, "confidence": 0.95, "language": "en"}, "model": "gpt-4o-mini", "key": "90001"}
{"id": 90002, "text": "Nagad helpline: apnar account e sobchhe chole gese. KYC update korte PIN ar OTP ta ei number e pathan, na hole block hobe.", "response": {"label": 1, "techniques": ["T1", "T2", "T5"], "action": 1, "confidence": 0.95, "language": "banglish"}, "model": "gpt-4o-mini", "key": "90002"}
{"id": 90003, "text": "S: We detected a login on your wallet ||| R: I am not at home ||| S: Then share the code we sent to keep access", "response": {"label": 1, "techniques": ["T1", "T2", "T5"], "action": 1, "confidence": 0.95, "language": "en"}, "model": "gpt-4o-mini", "key": "90003"}
{"id": 90004, "text": "bKash: Your new device login was approved at 18:42. If this was not you, call 16247 from the app. Never share your OTP with anyone, including our staff.", "response": {"label": 0, "techniques": [], "action": 0, "confidence": 0.95, "language": "en"}, "model": "gpt-4o-mini", "key": "90004"}
{"id": 90005, "text": "বিকাশ: আপনার অ্যাকাউন্টে নতুন ডিভাইস যোগ করা হয়েছে। যদি আপনি না করেন, অ্যাপ থেকে ১৬২৪৭ কল করুন। কাউকে ওটিপি বা পিন দেবেন না।", "response": {"label": 0, "techniques": [], "action": 0, "confidence": 0.95, "language": "bn"}, "model": "gpt-4o-mini", "key": "90005"}
{"id": 90006, "text": "S: bKash support here. We will never ask for your OTP. ||| R: I got a code, should I read it? ||| S: No. Keep it private and report the message in the app.", "response": {"label": 0, "techniques": [], "action": 0, "confidence": 0.95, "language": "en"}, "model": "gpt-4o-mini", "key": "90006"}
{"id": 90007, "text": "Daraz refund desk: Tk 1,250 refund approved. Send card number, expiry, CVV and OTP to release it.", "response": {"label": 1, "techniques": ["T1", "T2", "T5"], "action": 1, "confidence": 0.95, "language": "en"}, "model": "gpt-4o-mini", "key": "90007"}
{"id": 90008, "text": "Bhai, bhul kore apnar number e 5,000 taka gese. Ferot pathat hole bKash PIN ta bolen.", "response": {"label": 1, "techniques": ["T2", "T5"], "action": 1, "confidence": 0.95, "language": "banglish"}, "model": "gpt-4o-mini", "key": "90008"}
{"id": 90009, "text": "S: I sent money to the wrong number by mistake ||| R: I did not receive it ||| S: Please check and send it back; verify with your PIN first", "response": {"label": 1, "techniques": ["T2", "T5"], "action": 1, "confidence": 0.95, "language": "en"}, "model": "gpt-4o-mini", "key": "90009"}
{"id": 90010, "text": "bKash: Tk 1,250 refund from Daraz was credited to your wallet. TrxID BX91K7. No action needed; we never ask for PIN or OTP.", "response": {"label": 0, "techniques": [], "action": 0, "confidence": 0.95, "language": "en"}, "model": "gpt-4o-mini", "key": "90010"}
{"id": 90011, "text": "আপনার বিকাশ ওয়ালেটে ১,২৫০ টাকা রিফান্ড জমা হয়েছে। ট্রানজেকশন আইডি BX91K7। কোনো পিন বা ওটিপি দিতে হবে না।", "response": {"label": 0, "techniques": [], "action": 0, "confidence": 0.95, "language": "bn"}, "model": "gpt-4o-mini", "key": "90011"}
{"id": 90012, "text": "S: Did you receive the wrong-number transfer? ||| R: Yes, I will return it through bKash myself. ||| S: Good. Never ask anyone for your PIN to fix it.", "response": {"label": 0, "techniques": [], "action": 0, "confidence": 0.95, "language": "en"}, "model": "gpt-4o-mini", "key": "90012"}
{"id": 90013, "text": "SA Courier: parcel CX8821 stuck at customs. Pay Tk 450 clearance fee at hxxps://sa-clearing[.]top before 6 pm.", "response": {"label": 4, "techniques": ["T1", "T3", "T5"], "action": 2, "confidence": 0.95, "language": "en"}, "model": "gpt-4o-mini", "key": "90013"}
{"id": 90014, "text": "আপনার পার্সেল আটকে আছে। আজই ৫০ টাকা ডেলিভারি ফি দিতে লিংকে ক্লিক করুন: hxxps://parcel-fee[.]top", "response": {"label": 4, "techniques": ["T1", "T3", "T5"], "action": 2, "confidence": 0.95, "language": "bn"}, "model": "gpt-4o-mini", "key": "90014"}
{"id": 90015, "text": "S: DHL here, customs hold ||| R: I did not order anything ||| S: Pay Tk 900 duty today or the parcel returns", "response": {"label": 4, "techniques": ["T1", "T3", "T5"], "action": 2, "confidence": 0.95, "language": "en"}, "model": "gpt-4o-mini", "key": "90015"}
{"id": 90016, "text": "Pathao Parcel: delivery attempt 2 for tracking BD4471. Reschedule from the official app. Our riders never ask for advance payment by SMS link.", "response": {"label": 0, "techniques": [], "action": 0, "confidence": 0.95, "language": "en"}, "model": "gpt-4o-mini", "key": "90016"}
{"id": 90017, "text": "পাঠাও পার্সেল: আপনার BD4471 প্যারাসেল আজ বিকেল ৪টায় ডেলিভারি হবে। সময় বদলাতে অ্যাপ ব্যবহার করুন; এসএমএস লিংকে টাকা দেবেন না।", "response": {"label": 0, "techniques": [], "action": 0, "confidence": 0.95, "language": "bn"}, "model": "gpt-4o-mini", "key": "90017"}
{"id": 90018, "text": "S: Courier text says pay a fee via a link ||| R: Did you check the tracking number in the app? ||| S: Yes, it is already out for delivery. The link was fake.", "response": {"label": 0, "techniques": [], "action": 0, "confidence": 0.95, "language": "en"}, "model": "gpt-4o-mini", "key": "90018"}
{"id": 90019, "text": "Lucky draw: your number won a Tk 50,000 voucher. Claim at hxxps://win-bd[.]top within 60 minutes.", "response": {"label": 2, "techniques": ["T4", "T5"], "action": 3, "confidence": 0.95, "language": "en"}, "model": "gpt-4o-mini", "key": "90019"}
{"id": 90020, "text": "Oi apni 10 lakh taka jitechen. Claim korte age 300 tk processing fee pathan.", "response": {"label": 2, "techniques": ["T3", "T4", "T5"], "action": 2, "confidence": 0.95, "language": "banglish"}, "model": "gpt-4o-mini", "key": "90020"}
{"id": 90021, "text": "S: Congrats, you won a smartphone ||| R: really? ||| S: Pay Tk 500 delivery charge and confirm your address", "response": {"label": 2, "techniques": ["T3", "T4"], "action": 2, "confidence": 0.95, "language": "en"}, "model": "gpt-4o-mini", "key": "90021"}
{"id": 90022, "text": "Robi: EID bonus 2 GB added to your number. Check balance with *121*3#. No claim link or fee is required.", "response": {"label": 5, "techniques": [], "action": 0, "confidence": 0.95, "language": "en"}, "model": "gpt-4o-mini", "key": "90022"}
{"id": 90023, "text": "জিপির ঈদ অফার: ১ জিবি কিনলে ১ জিবি ফ্রি। *১২১*৩# ডায়াল করে দেখুন; কোনো ফি বা লিংক নেই।", "response": {"label": 5, "techniques": [], "action": 0, "confidence": 0.95, "language": "bn"}, "model": "gpt-4o-mini", "key": "90023"}
{"id": 90024, "text": "S: Is the cashback message real? ||| R: I checked the operator app, it is a genuine 10% top-up cashback. ||| S: Good, no need to click the SMS link.", "response": {"label": 5, "techniques": [], "action": 0, "confidence": 0.95, "language": "en"}, "model": "gpt-4o-mini", "key": "90024"}
{"id": 90025, "text": "Part-time task: rate 5 apps daily, earn Tk 800. Join hxxps://t[.]me/task_bd and deposit Tk 2,000 as security.", "response": {"label": 3, "techniques": ["T3", "T6"], "action": 2, "confidence": 0.95, "language": "en"}, "model": "gpt-4o-mini", "key": "90025"}
{"id": 90026, "text": "Freelance job: protidin 3000 tk income. WhatsApp e join korte fee 1500 tk pathan.", "response": {"label": 3, "techniques": ["T3", "T6"], "action": 2, "confidence": 0.95, "language": "banglish"}, "model": "gpt-4o-mini", "key": "90026"}
{"id": 90027, "text": "S: We saw your CV, data entry role ||| R: how much? ||| S: Tk 40,000 monthly, pay Tk 3,000 training fee first", "response": {"label": 3, "techniques": ["T1", "T3", "T6"], "action": 2, "confidence": 0.95, "language": "en"}, "model": "gpt-4o-mini", "key": "90027"}
{"id": 90028, "text": "Bdjobs: Your application for Data Entry Officer was received. Employer will contact you through Bdjobs messages only. Never pay to get a job.", "response": {"label": 0, "techniques": [], "action": 0, "confidence": 0.95, "language": "en"}, "model": "gpt-4o-mini", "key": "90028"}
{"id": 90029, "text": "বিডিজবস: আপনার আবেদন গৃহীত হয়েছে। নিয়োগকর্তা শুধু বিডিজবস মেসেজে যোগাযোগ করবেন। টাকা দিয়ে চাকরি কেনা যাবে না।", "response": {"label": 0, "techniques": [], "action": 0, "confidence": 0.95, "language": "bn"}, "model": "gpt-4o-mini", "key": "90029"}
{"id": 90030, "text": "S: Recruiter asking for a training fee ||| R: That is a scam; real employers never charge you ||| S: Thanks, I reported the chat.", "response": {"label": 0, "techniques": [], "action": 0, "confidence": 0.95, "language": "en"}, "model": "gpt-4o-mini", "key": "90030"}
{"id": 90031, "text": "Invest Tk 10,000 in our crypto bot, get 25% weekly profit guaranteed. Send to bKash 01XXXXXXXXX now.", "response": {"label": 3, "techniques": ["T3", "T5", "T6"], "action": 2, "confidence": 0.95, "language": "en"}, "model": "gpt-4o-mini", "key": "90031"}
{"id": 90032, "text": "Ami investment group e 300% profit dekhechi. Join korte Tk 5,000 fee dite hobe.", "response": {"label": 3, "techniques": ["T3", "T6"], "action": 4, "confidence": 0.95, "language": "banglish"}, "model": "gpt-4o-mini", "key": "90032"}
{"id": 90033, "text": "S: Sir, double your money in 7 days ||| R: is it safe? ||| S: 100% guaranteed, transfer today only", "response": {"label": 3, "techniques": ["T3", "T5", "T6"], "action": 2, "confidence": 0.95, "language": "en"}, "model": "gpt-4o-mini", "key": "90033"}
{"id": 90034, "text": "Bangladesh Bank advisory: verify any investment offer against the licensed list. High 'guaranteed' returns are a warning sign.", "response": {"label": 0, "techniques": [], "action": 0, "confidence": 0.95, "language": "en"}, "model": "gpt-4o-mini", "key": "90034"}
{"id": 90035, "text": "বাংলাদেশ ব্যাংক সতর্কতা: বিনিয়োগের আগে লাইসেন্স তালিকা যাচাই করুন। 'নিশ্চিত লাভ' প্রতারণার লক্ষণ হতে পারে।", "response": {"label": 0, "techniques": [], "action": 0, "confidence": 0.95, "language": "bn"}, "model": "gpt-4o-mini", "key": "90035"}
{"id": 90036, "text": "S: This investment chat promises guaranteed profit ||| R: Do not transfer; check the regulator list first ||| S: Agreed, I will report it.", "response": {"label": 0, "techniques": [], "action": 0, "confidence": 0.95, "language": "en"}, "model": "gpt-4o-mini", "key": "90036"}
{"id": 90037, "text": "Robi: your SIM will be blocked tomorrow for unverified KYC. Reply with NID number and OTP to keep it active.", "response": {"label": 1, "techniques": ["T1", "T2", "T5"], "action": 1, "confidence": 0.95, "language": "en"}, "model": "gpt-4o-mini", "key": "90037"}
{"id": 90038, "text": "নগদ: অ্যাকাউন্ট বন্ধ হতে পারে। এখনই পিন নম্বর পাঠান।", "response": {"label": 1, "techniques": ["T1", "T2", "T5"], "action": 1, "confidence": 0.95, "language": "bn"}, "model": "gpt-4o-mini", "key": "90038"}
{"id": 90039, "text": "S: Telecom support here ||| R: why? ||| S: Confirm your SIM by sharing the OTP we just sent", "response": {"label": 1, "techniques": ["T1", "T2", "T5"], "action": 1, "confidence": 0.95, "language": "en"}, "model": "gpt-4o-mini", "key": "90039"}
{"id": 90040, "text": "Titas Gas bill reminder: Tk 860 due on 30 Nov. Pay through the official app or bank branch. Ignore SMS links asking for OTP.", "response": {"label": 0, "techniques": [], "action": 0, "confidence": 0.95, "language": "en"}, "model": "gpt-4o-mini", "key": "90040"}
{"id": 90041, "text": "টিটাস গ্যাস: আপনার বিল ৮৬০ টাকা ৩০ নভেম্বরের আগে দিতে হবে। অফিশিয়াল অ্যাপ বা ব্যাংক ব্যবহার করুন; ওটিপি চাওয়া বার্তা এড়িয়ে চলুন।", "response": {"label": 0, "techniques": [], "action": 0, "confidence": 0.95, "language": "bn"}, "model": "gpt-4o-mini", "key": "90041"}
{"id": 90042, "text": "S: Gas bill text says pay via link ||| R: I paid in the app; the link was not official ||| S: Good, never share OTP for bill payment.", "response": {"label": 0, "techniques": [], "action": 0, "confidence": 0.95, "language": "en"}, "model": "gpt-4o-mini", "key": "90042"}
{"id": 90043, "text": "Nagad app update required. Download hxxps://nagad-update[.]top/app.apk and enter old PIN to keep wallet active.", "response": {"label": 1, "techniques": ["T1", "T2", "T4", "T5"], "action": 4, "confidence": 0.95, "language": "en"}, "model": "gpt-4o-mini", "key": "90043"}
{"id": 90044, "text": "Voucher activate korte app install korun hxxps://gp-voucher[.]top/a.apk ar PIN din.", "response": {"label": 1, "techniques": ["T1", "T2", "T4"], "action": 4, "confidence": 0.95, "language": "banglish"}, "model": "gpt-4o-mini", "key": "90044"}
{"id": 90045, "text": "S: Support says install an update ||| R: from where? ||| S: this link, then share your old PIN", "response": {"label": 1, "techniques": ["T1", "T2", "T4"], "action": 4, "confidence": 0.95, "language": "en"}, "model": "gpt-4o-mini", "key": "90045"}
{"id": 90046, "text": "bKash: Scheduled maintenance tonight 01:00-03:00. Details: hxxps://bkash[.]com/notice. No app download or PIN entry is needed.", "response": {"label": 0, "techniques": [], "action": 0, "confidence": 0.95, "language": "en"}, "model": "gpt-4o-mini", "key": "90046"}
{"id": 90047, "text": "বিকাশ: আজ রাত ১টা থেকে ৩টা রক্ষণাবেক্ষণ। বিবরণ hxxps://bkash[.]com/notice; কোনো পিন বা অ্যাপ ইনস্টলের প্রয়োজন নেই।", "response": {"label": 0, "techniques": [], "action": 0, "confidence": 0.95, "language": "bn"}, "model": "gpt-4o-mini", "key": "90047"}
{"id": 90048, "text": "S: Maintenance notice has a link ||| R: I opened the official domain, not the forwarded one ||| S: Correct; never install apps from SMS links.", "response": {"label": 0, "techniques": [], "action": 0, "confidence": 0.95, "language": "en"}, "model": "gpt-4o-mini", "key": "90048"}
{"id": 90049, "text": "Mama, ami bipode. Amar bKash e 3,000 taka nai. Tomar PIN ta dao, ami send kore nei.", "response": {"label": 1, "techniques": ["T2", "T5"], "action": 1, "confidence": 0.95, "language": "banglish"}, "model": "gpt-4o-mini", "key": "90049"}
{"id": 90050, "text": "Bhai, ami emergency e achhi. OTP ta patha diben, amar account block hoye jabe.", "response": {"label": 1, "techniques": ["T2", "T5"], "action": 1, "confidence": 0.95, "language": "banglish"}, "model": "gpt-4o-mini", "key": "90050"}
{"id": 90051, "text": "S: It is me, new number ||| R: really? ||| S: yes, send the OTP I just texted you", "response": {"label": 1, "techniques": ["T2", "T5"], "action": 1, "confidence": 0.95, "language": "en"}, "model": "gpt-4o-mini", "key": "90051"}
{"id": 90052, "text": "Ami ekta fake bKash message dekhechi. Kauke OTP ba PIN bolben na; ami amar number change korini.", "response": {"label": 0, "techniques": [], "action": 0, "confidence": 0.95, "language": "banglish"}, "model": "gpt-4o-mini", "key": "90052"}
{"id": 90053, "text": "আমি একটা জাল bKash বার্তা দেখেছি। কাউকে ওটিপি বা পিন বলবেন না; আমি আমার নম্বর পরিবর্তন করিনি।", "response": {"label": 0, "techniques": [], "action": 0, "confidence": 0.95, "language": "bn"}, "model": "gpt-4o-mini", "key": "90053"}
{"id": 90054, "text": "S: Bro, someone texted from my number asking for OTP ||| R: I did not share it ||| S: Good, block and report that number.", "response": {"label": 0, "techniques": [], "action": 0, "confidence": 0.95, "language": "en"}, "model": "gpt-4o-mini", "key": "90054"}
{"id": 90055, "text": "You won Tk 7,000 cashback. Confirm by clicking hxxps://cb-bd[.]top and pay Tk 250 verification fee.", "response": {"label": 2, "techniques": ["T3", "T4", "T5"], "action": 2, "confidence": 0.95, "language": "en"}, "model": "gpt-4o-mini", "key": "90055"}
{"id": 90056, "text": "Cashback 5,000 tk jitechen. Link e click kore address confirm korun.", "response": {"label": 2, "techniques": ["T4", "T5"], "action": 3, "confidence": 0.95, "language": "banglish"}, "model": "gpt-4o-mini", "key": "90056"}
{"id": 90057, "text": "S: Congrats, cashback winner ||| R: wow ||| S: Click this link and pay Tk 200 processing", "response": {"label": 2, "techniques": ["T3", "T4"], "action": 3, "confidence": 0.95, "language": "en"}, "model": "gpt-4o-mini", "key": "90057"}
{"id": 90058, "text": "Aarong EID offer: 20% off on all dresses this week. Valid in store and on the official app; no fee or link needed.", "response": {"label": 5, "techniques": [], "action": 0, "confidence": 0.95, "language": "en"}, "model": "gpt-4o-mini", "key": "90058"}
{"id": 90059, "text": "আরংগ ঈদ অফার: সব পোশাকে ২০% ছাড়। দোকান ও অফিশিয়াল অ্যাপে কার্যকর; কোনো ফি বা লিংক নেই।", "response": {"label": 5, "techniques": [], "action": 0, "confidence": 0.95, "language": "bn"}, "model": "gpt-4o-mini", "key": "90059"}
{"id": 90060, "text": "S: Is this cashback real? ||| R: I checked the official app, yes, 10% top-up cashback. ||| S: Great, no need to reply to the SMS.", "response": {"label": 5, "techniques": [], "action": 0, "confidence": 0.95, "language": "en"}, "model": "gpt-4o-mini", "key": "90060"}
{"id": 90061, "text": "Courier tracking BD9911 failed. Confirm address at hxxps://bd-track[.]top; pay Tk 60 redelivery now.", "response": {"label": 4, "techniques": ["T1", "T3", "T5"], "action": 2, "confidence": 0.95, "language": "en"}, "model": "gpt-4o-mini", "key": "90061"}
{"id": 90062, "text": "আপনার পার্সেলের ঠিকানা ভুল। hxxps://bd-track[.]top এ ক্লিক করে ৬০ টাকা দিন।", "response": {"label": 4, "techniques": ["T1", "T3"], "action": 3, "confidence": 0.95, "language": "bn"}, "model": "gpt-4o-mini", "key": "90062"}
{"id": 90063, "text": "S: Courier here ||| R: I have no parcel ||| S: It is in your name, pay Tk 60 customs today", "response": {"label": 4, "techniques": ["T1", "T3", "T5"], "action": 2, "confidence": 0.95, "language": "en"}, "model": "gpt-4o-mini", "key": "90063"}
{"id": 90064, "text": "Sundarban Courier: tracking BD9911 out for delivery today. Check status on the official website; no payment requested.", "response": {"label": 0, "techniques": [], "action": 0, "confidence": 0.95, "language": "en"}, "model": "gpt-4o-mini", "key": "90064"}
{"id": 90065, "text": "সুন্দরবন কুরিয়ার: BD9911 আজ ডেলিভারিতে আছে। অফিশিয়াল ওয়েবসাইটে দেখুন; কোনো টাকা চাওয়া হয়নি।", "response": {"label": 0, "techniques": [], "action": 0, "confidence": 0.95, "language": "bn"}, "model": "gpt-4o-mini", "key": "90065"}
{"id": 90066, "text": "S: Delivery message asks for a fee ||| R: Official tracking says out for delivery ||| S: Then it is phishing; ignore the link.", "response": {"label": 0, "techniques": [], "action": 0, "confidence": 0.95, "language": "en"}, "model": "gpt-4o-mini", "key": "90066"}
{"id": 90067, "text": "HR: your salary is on hold. Share bank account and OTP to release Tk 45,000 today.", "response": {"label": 1, "techniques": ["T1", "T2", "T5"], "action": 1, "confidence": 0.95, "language": "en"}, "model": "gpt-4o-mini", "key": "90067"}
{"id": 90068, "text": "HR office: salary release korte account details ar OTP pathan.", "response": {"label": 1, "techniques": ["T1", "T2"], "action": 1, "confidence": 0.95, "language": "banglish"}, "model": "gpt-4o-mini", "key": "90068"}
{"id": 90069, "text": "S: Payroll team here ||| R: why OTP? ||| S: to verify salary account, send now", "response": {"label": 1, "techniques": ["T1", "T2", "T5"], "action": 1, "confidence": 0.95, "language": "en"}, "model": "gpt-4o-mini", "key": "90069"}
{"id": 90070, "text": "Payroll: salary Tk 45,000 credited to account ending 4412. Payslip is in the office portal. HR never asks for OTP.", "response": {"label": 0, "techniques": [], "action": 0, "confidence": 0.95, "language": "en"}, "model": "gpt-4o-mini", "key": "90070"}
{"id": 90071, "text": "বেতন: ৪৫,০০০ টাকা ৪৪১২ নম্বর হিসাব জমা হয়েছে। পেসলিপ অফিস পোর্টালে আছে; এইচআর কখনো ওটিপি চায় না।", "response": {"label": 0, "techniques": [], "action": 0, "confidence": 0.95, "language": "bn"}, "model": "gpt-4o-mini", "key": "90071"}
{"id": 90072, "text": "S: HR text asks for bank OTP ||| R: Never share it; I checked the portal ||| S: Reported to security.", "response": {"label": 0, "techniques": [], "action": 0, "confidence": 0.95, "language": "en"}, "model": "gpt-4o-mini", "key": "90072"}
{"id": 90073, "text": "Customer loyalty survey: you qualified for Tk 15,000 gift. Pay Tk 100 handling fee at hxxps://survey-bd[.]top to claim.", "response": {"label": 2, "techniques": ["T1", "T3", "T4"], "action": 2, "confidence": 0.95, "language": "en"}, "model": "gpt-4o-mini", "key": "90073"}
{"id": 90074, "text": "Survey e 15,000 tk gift. Claim korte hxxps://survey-bd[.]top e click korun.", "response": {"label": 2, "techniques": ["T4", "T5"], "action": 3, "confidence": 0.95, "language": "banglish"}, "model": "gpt-4o-mini", "key": "90074"}
{"id": 90075, "text": "S: Loyalty reward selected ||| R: what? ||| S: Pay Tk 100 shipping for your gift", "response": {"label": 2, "techniques": ["T3", "T4"], "action": 2, "confidence": 0.95, "language": "en"}, "model": "gpt-4o-mini", "key": "90075"}
{"id": 90076, "text": "Customer survey: tell us about network quality in the app and get 500 bonus points. No payment or personal data required.", "response": {"label": 5, "techniques": [], "action": 0, "confidence": 0.95, "language": "en"}, "model": "gpt-4o-mini", "key": "90076"}
{"id": 90077, "text": "কাস্টমার সার্ভে: অ্যাপে নেটওয়ার্ক মতামত দিন ৫০০ বোনাস পয়েন্ট পাবেন। কোনো টাকা বা ব্যক্তিগত তথ্য লাগবে না।", "response": {"label": 5, "techniques": [], "action": 0, "confidence": 0.95, "language": "bn"}, "model": "gpt-4o-mini", "key": "90077"}
{"id": 90078, "text": "S: Survey offers bonus points ||| R: I filled it in the official app ||| S: Good, no link or payment needed.", "response": {"label": 5, "techniques": [], "action": 0, "confidence": 0.95, "language": "en"}, "model": "gpt-4o-mini", "key": "90078"}
{"id": 90079, "text": "bKash support: apnar РIN expire today. Reply with currentP1N and new PIN to continue.", "response": {"label": 1, "techniques": ["T1", "T2", "T5"], "action": 1, "confidence": 0.95, "language": "mixed"}, "model": "gpt-4o-mini", "key": "90079"}
{"id": 90080, "text": "N0gad helpline: ver1fy account, send 0TP + P1N nowwww.", "response": {"label": 1, "techniques": ["T1", "T2", "T5"], "action": 1, "confidence": 0.95, "language": "mixed"}, "model": "gpt-4o-mini", "key": "90080"}
{"id": 90081, "text": "S: Assalamu alaikum, bKash agent ||| R: kotha bolun ||| S: apnar account verify korte PIN bolen ||| R: no ||| S: na hole SIM block", "response": {"label": 1, "techniques": ["T1", "T2", "T5"], "action": 1, "confidence": 0.95, "language": "mixed"}, "model": "gpt-4o-mini", "key": "90081"}
{"id": 90082, "text": "bKash notice: your PIN was changed today at 14:10. If not you, call 16247. We will never ask for OTP in chat.", "response": {"label": 0, "techniques": [], "action": 0, "confidence": 0.95, "language": "en"}, "model": "gpt-4o-mini", "key": "90082"}
{"id": 90083, "text": "বিকাশ নোটিশ: আজ ১৪:১০ এ পিন বদল হয়েছে। আপনি না করলে ১৬২৪৭ কল করুন। চ্যাটে কখনো ওটিপি চাইব না।", "response": {"label": 0, "techniques": [], "action": 0, "confidence": 0.95, "language": "bn"}, "model": "gpt-4o-mini", "key": "90083"}
{"id": 90084, "text": "S: Message says SIM will block ||| R: I checked in app, nothing pending ||| S: Good, delete it and never send OTP.", "response": {"label": 0, "techniques": [], "action": 0, "confidence": 0.95, "language": "en"}, "model": "gpt-4o-mini", "key": "90084"}


In [ ]:
%%writefile data/extra/pseudo_labels_v1.jsonl
{"id": 47239, "text": "security alert: new device logged in to your mcash. if this was you, ignore. if not, call 16247. we never ask for your pin or otp.", "response": {"label": 0, "techniques": [], "action": 0, "confidence": 0.9410691765752959}, "source": "pseudo", "weight": 0.2}
{"id": 68547, "text": "We hope this message finds you well. A: I forgot my bkash PIN. How can I reset it? ||| B: Dial *247# and choose Reset PIN. Never share your PIN with anyone, including our staff. ||| A: Thank you Please do not reply to this automated message.", "response": {"label": 0, "techniques": [], "action": 0, "confidence": 0.9395417148307175}, "source": "pseudo", "weight": 0.2}
{"id": 66579, "text": "security alert: new device logged in to your tap. if this was you, ignore. if not, call 16247. we never ask for your pin or otp.", "response": {"label": 0, "techniques": [], "action": 0, "confidence": 0.938500168642851}, "source": "pseudo", "weight": 0.2}
{"id": 42156, "text": "Security alert: new device logged in to your bKash. If this was you, ignore. If not, call 16268. We never ask for your PIN or OTP.", "response": {"label": 0, "techniques": [], "action": 0, "confidence": 0.9378027420597057}, "source": "pseudo", "weight": 0.2}
{"id": 57635, "text": "security alert: new device logged in to your bkash. if this was you, ignore. if not, call 16268. we never ask for your pin or otp. 😊", "response": {"label": 0, "techniques": [], "action": 0, "confidence": 0.9378027420597057}, "source": "pseudo", "weight": 0.2}
{"id": 30565, "text": "security alert: new device logged in to your bkash. if this was you, ignore. if not, call 16268. we never ask for your pin or otp. 🙏", "response": {"label": 0, "techniques": [], "action": 0, "confidence": 0.9378027420597057}, "source": "pseudo", "weight": 0.2}
{"id": 43779, "text": "Security alert: new device logged in to your OK Wallet. If this was you, ignore. If not, call 16247. We never ask for your PIN or OTP. 🔥", "response": {"label": 0, "techniques": [], "action": 0, "confidence": 0.9363752379870232}, "source": "pseudo", "weight": 0.2}
{"id": 36842, "text": "bkash: Apnar Tk 400 cash-out successful. Balance Tk 75,913. TrxID PY88N4K.", "response": {"label": 0, "techniques": [], "action": 0, "confidence": 0.9356681707458666}, "source": "pseudo", "weight": 0.2}
{"id": 33881, "text": "bKash: Apnar Tk 2,200 cash-out successful. Balance Tk 43,709. TrxID 71PCJ4ASH9.", "response": {"label": 0, "techniques": [], "action": 0, "confidence": 0.9344812526043063}, "source": "pseudo", "weight": 0.2}
{"id": 67214, "text": "a: i forgot my pathao pay pin. how can i reset it? ||| b: dial *247# and choose reset pin. never share your pin with anyone, including our staff. ||| a: thank you 📲", "response": {"label": 0, "techniques": [], "action": 0, "confidence": 0.9328834798006869}, "source": "pseudo", "weight": 0.2}
{"id": 42578, "text": "A: I forgot my Pathao Pay PIN. How can I reset it? ||| B: Dial *247# and choose Reset PIN. Never share your PIN with anyone, including our staff. ||| A: Thank you 💰", "response": {"label": 0, "techniques": [], "action": 0, "confidence": 0.9328834798006869}, "source": "pseudo", "weight": 0.2}
{"id": 43277, "text": "A: Is my order here? ||| B: Rider called, will deliver at 10 ||| A: ok I'll pay cash on delivery Please do not reply to this automated message.", "response": {"label": 0, "techniques": [], "action": 0, "confidence": 0.9328198539927308}, "source": "pseudo", "weight": 0.2}
{"id": 22254, "text": "bkash: apnar tk 4,500 cash-out successful. balance tk 69,707. trxid qhmn65uz.", "response": {"label": 0, "techniques": [], "action": 0, "confidence": 0.9326561375512563}, "source": "pseudo", "weight": 0.2}
{"id": 35088, "text": "bkash: apnar tk 2,200 cash-out successful. balance tk 32,597. trxid fvzlhl8b7.", "response": {"label": 0, "techniques": [], "action": 0, "confidence": 0.9312375191002937}, "source": "pseudo", "weight": 0.2}
{"id": 32684, "text": "A: I forgot my mCash PIN. How can I reset it? ||| B: Dial *322# and choose Reset PIN. Never share your PIN with anyone, including our staff. ||| A: Thank u Plz do not reply to this automated message.", "response": {"label": 0, "techniques": [], "action": 0, "confidence": 0.9305806656200182}, "source": "pseudo", "weight": 0.2}
{"id": 37971, "text": "bKash: Apnar Tk 200 cash-out successful. Balance Tk 9,593. TrxID V5MV16K42.", "response": {"label": 0, "techniques": [], "action": 0, "confidence": 0.9302255019646354}, "source": "pseudo", "weight": 0.2}
{"id": 39060, "text": "BKASH: APNAR TK 400 CASH-OUT SUCCESSFUL. BALANCE TK 90,904. TRXID Y0NJSPFP4S.", "response": {"label": 0, "techniques": [], "action": 0, "confidence": 0.9296623721992952}, "source": "pseudo", "weight": 0.2}
{"id": 48347, "text": "A: I forgot my bKash PIN. How can I reset it? ||| B: Dial *322# and choose Reset PIN. Never share your PIN with anyone, including our staff. ||| A: Thank you ⚠️", "response": {"label": 0, "techniques": [], "action": 0, "confidence": 0.9295923277580312}, "source": "pseudo", "weight": 0.2}
{"id": 23588, "text": "bkash: Apnar Tk 250 cash-out successful. Balance Tk 52,349. TrxID 1ULKF81M88.", "response": {"label": 0, "techniques": [], "action": 0, "confidence": 0.929549304222244}, "source": "pseudo", "weight": 0.2}
{"id": 41872, "text": "bKash: Apnar Tk 700 cash-out successful. Balance Tk 87,999. TrxID 06FX3NDZ3Y.", "response": {"label": 0, "techniques": [], "action": 0, "confidence": 0.9294727675702967}, "source": "pseudo", "weight": 0.2}
{"id": 25410, "text": "bk​ash: apnar tk 150cash-out successful. balance tk 29,588. trxid 9sq4996bbc.", "response": {"label": 0, "techniques": [], "action": 0, "confidence": 0.9294248790572097}, "source": "pseudo", "weight": 0.2}
{"id": 60187, "text": "Security alert: new device logged in to your Rocket. If this was you, ignore. If not, call 16268. We never ask for your PIN or OTP. 🎁", "response": {"label": 0, "techniques": [], "action": 0, "confidence": 0.9287460660737027}, "source": "pseudo", "weight": 0.2}
{"id": 44067, "text": "bkash: apnar tk 150 cash-out successful. balance tk 67,470. trxid 2lrbjk7rm.", "response": {"label": 0, "techniques": [], "action": 0, "confidence": 0.9279449293084103}, "source": "pseudo", "weight": 0.2}
{"id": 59993, "text": "BKASH: APNAR TK 400 CASH-OUT SUCCESSFUL. BALANCE TK 70,752. TRXID 601NLJ4W.", "response": {"label": 0, "techniques": [], "action": 0, "confidence": 0.9279123132059897}, "source": "pseudo", "weight": 0.2}
{"id": 41387, "text": "a: i forgot my surecash pin. how can i reset it? ||| b: dial *247# and choose reset pin. never share your pin with anyone, including our staff. ||| a: thank you", "response": {"label": 0, "techniques": [], "action": 0, "confidence": 0.9277847500918249}, "source": "pseudo", "weight": 0.2}
{"id": 34369, "text": "bkash: apnar tk 350 cash-out successful. balance tk 52,206. trxid 7ptebfjcw2. 😊", "response": {"label": 0, "techniques": [], "action": 0, "confidence": 0.9276561361640887}, "source": "pseudo", "weight": 0.2}
{"id": 29494, "text": "security alert: new device logged in to your tap. if this was you, ignore. if not, call 16268. we never ask for your pin or otp.", "response": {"label": 0, "techniques": [], "action": 0, "confidence": 0.927574198852664}, "source": "pseudo", "weight": 0.2}
{"id": 38222, "text": "bKash: Apnar Tk 600 cash-out successful. Balance Tk 68,153. TrxID SGEPEG9. 📲", "response": {"label": 0, "techniques": [], "action": 0, "confidence": 0.9275011639745653}, "source": "pseudo", "weight": 0.2}
{"id": 43949, "text": "Pathao Pay: Apnar Tk 100 cash-out successful. Balance Tk 14,647. TrxID 5YY2E9M34. 📲", "response": {"label": 0, "techniques": [], "action": 0, "confidence": 0.9274907632432834}, "source": "pseudo", "weight": 0.2}
{"id": 31025, "text": "Security alert: new device logged in to your mCash. If this was you, ignore. If not, call 16167. We never ask for your PIN or OTP.", "response": {"label": 0, "techniques": [], "action": 0, "confidence": 0.9274765371111051}, "source": "pseudo", "weight": 0.2}
{"id": 61808, "text": "bKash: Apnar Tk 200 cash-out successful. Balance Tk 36,207. TrxID ZW3JFP5LE.", "response": {"label": 0, "techniques": [], "action": 0, "confidence": 0.9270975734295249}, "source": "pseudo", "weight": 0.2}
{"id": 46848, "text": "bkash: apnar tk 350 cash-out successful. balance tk 77,422. trxid 4wuelkn.", "response": {"label": 0, "techniques": [], "action": 0, "confidence": 0.9267499788709751}, "source": "pseudo", "weight": 0.2}
{"id": 40464, "text": "bKash: Apnar Tk 700 cash-out successful. Balance Tk 77,021. TrxID L4X7L2PBDK.", "response": {"label": 0, "techniques": [], "action": 0, "confidence": 0.926553448958279}, "source": "pseudo", "weight": 0.2}
{"id": 38780, "text": "Security alert: new device logged in to your Pathao Pay If this was you, ignore If not, call 16167 We never ask for your PIN or OTP 📲", "response": {"label": 0, "techniques": [], "action": 0, "confidence": 0.9264371568948006}, "source": "pseudo", "weight": 0.2}
{"id": 53406, "text": "bkash: apnar tk 2,200 cash-out successful. balance tk 42,700. trxid lx1jm1ml.", "response": {"label": 0, "techniques": [], "action": 0, "confidence": 0.9263612407169397}, "source": "pseudo", "weight": 0.2}
{"id": 53079, "text": "Security alert: new device logged in to your SureCash. If this was you, ignore. If not, call 16167. We never ask for your PIN or OTP.", "response": {"label": 0, "techniques": [], "action": 0, "confidence": 0.9260949648023266}, "source": "pseudo", "weight": 0.2}
{"id": 27344, "text": "xkcdpl A: Is 📲 my ordеr here? ||| B: Rider called, will deliver at 9 qzxvbn ||| A: ok I'll pay cash on delivery", "response": {"label": 0, "techniques": [], "action": 0, "confidence": 0.9259761251020681}, "source": "pseudo", "weight": 0.2}
{"id": 50684, "text": "A: Is my order here? ||| B: Rider called, will deliver at 9 ||| A: ok I'll pay cash on delivery ⚠️", "response": {"label": 0, "techniques": [], "action": 0, "confidence": 0.9259761251020681}, "source": "pseudo", "weight": 0.2}
{"id": 55611, "text": "A: Is my order here? ||| B: Rider called, will deliver at 9 ||| A: ok I'll pay cash on delivery 😊", "response": {"label": 0, "techniques": [], "action": 0, "confidence": 0.9259761251020681}, "source": "pseudo", "weight": 0.2}
{"id": 52349, "text": "Security alert: new device logged in to your bkash If this was you, ignore If not, call 16167 We never ask for your PIN or OTP", "response": {"label": 0, "techniques": [], "action": 0, "confidence": 0.9259643914625848}, "source": "pseudo", "weight": 0.2}
{"id": 34320, "text": "bkash: Apnar Tk 1,200 cash-out successful. Balance Tk 95,455. TrxID NY1B6LT.", "response": {"label": 0, "techniques": [], "action": 0, "confidence": 0.9258699293676512}, "source": "pseudo", "weight": 0.2}
{"id": 35813, "text": "Pathao Pay: Apnar Tk 400 cash-out successful. Balance Tk 74,681. TrxID 5KKPUE8.", "response": {"label": 0, "techniques": [], "action": 0, "confidence": 0.9256237186394334}, "source": "pseudo", "weight": 0.2}
{"id": 45914, "text": "A: I FORGOT MY PATHAO PAY PIN. HOW CAN I RESET IT? ||| B: DIAL *322# AND CHOOSE RESET PIN. NEVER SHARE YOUR PIN WITH ANYONE, INCLUDING OUR STAFF. ||| A: THANK YOU", "response": {"label": 0, "techniques": [], "action": 0, "confidence": 0.9255785311000294}, "source": "pseudo", "weight": 0.2}
{"id": 31135, "text": "a: i forgot my pathao pay pin. how can i reset it? ||| b: dial *322# and choose reset pin. never share your pin with anyone, including our staff. ||| a: thank you", "response": {"label": 0, "techniques": [], "action": 0, "confidence": 0.9255785311000294}, "source": "pseudo", "weight": 0.2}
{"id": 33411, "text": "A: I forgot my SureCash PIN How can I reset it? ||| B: Dial *247# and choose Reset PIN Never share your PIN with anyone, including our staff ||| A: Thank you", "response": {"label": 0, "techniques": [], "action": 0, "confidence": 0.9249250813303993}, "source": "pseudo", "weight": 0.2}
{"id": 33019, "text": "Security alert: new device logged in to your Tap. If this was you, ignore. If not, call 16167. We never ask for your PIN or OTP.", "response": {"label": 0, "techniques": [], "action": 0, "confidence": 0.9242221734465973}, "source": "pseudo", "weight": 0.2}
{"id": 59983, "text": "SECURITY ALERT: NEW DEVICE LOGGED IN TO YOUR TAP. IF THIS WAS YOU, IGNORE. IF NOT, CALL 16167. WE NEVER ASK FOR YOUR PIN OR OTP.", "response": {"label": 0, "techniques": [], "action": 0, "confidence": 0.9242221734465973}, "source": "pseudo", "weight": 0.2}
{"id": 37389, "text": "A: I forgot my Rocket PIN. How can I reset it? ||| B: Dial *247# and choose Reset PIN. Never share your PIN with anyone, including our staff. ||| A: Thank you", "response": {"label": 0, "techniques": [], "action": 0, "confidence": 0.9240631563957248}, "source": "pseudo", "weight": 0.2}
{"id": 23033, "text": "A: I asdkjh forgot my bkash PIN. How can I reset 1t? ||| B:Dial *247# ANDchoose Reset PIN. Never share your PIN 📲 wth anyone, including our staff. ||| A: Thank you", "response": {"label": 0, "techniques": [], "action": 0, "confidence": 0.9230564284096258}, "source": "pseudo", "weight": 0.2}
{"id": 57581, "text": "Caller: Security alert: new device logged in to your Pathao Pay. ||| You: right ||| Caller: If this was you, ignore. ||| You: yes ||| Caller: If not, call 16167. ||| You: ok ||| Caller: We never ask for your PIN or OTP.", "response": {"label": 0, "techniques": [], "action": 0, "confidence": 0.9229358011155346}, "source": "pseudo", "weight": 0.2}
{"id": 39519, "text": "bkash: Apnar Tk450 cash-out suc​cessful. Balance Tk 25,911. xkcdpl TrxID S0NRHE4W. 🎁", "response": {"label": 0, "techniques": [], "action": 0, "confidence": 0.9227250656321785}, "source": "pseudo", "weight": 0.2}
{"id": 34120, "text": "We hope this message finds you well. A: I forgot my Pathao Pay PIN. How can I reset it? ||| B: Dial *167# and choose Reset PIN. Never share your PIN with anyone, including our staff. ||| A: Thank you Please do not reply to this automated message.", "response": {"label": 0, "techniques": [], "action": 0, "confidence": 0.9217395954755305}, "source": "pseudo", "weight": 0.2}
{"id": 27014, "text": "bkash: Apnar Tk 500 cash-out successful. Balance Tk 28,333. TrxID B8JSW59BU.", "response": {"label": 0, "techniques": [], "action": 0, "confidence": 0.921738736050916}, "source": "pseudo", "weight": 0.2}
{"id": 56196, "text": "We hope this message finds you well. A: I forgot my bkash PIN. How can I reset it? ||| B: Dial *167# and choose Reset PIN. Never share your PIN with anyone, including our staff. ||| A: Thank you", "response": {"label": 0, "techniques": [], "action": 0, "confidence": 0.921717276427274}, "source": "pseudo", "weight": 0.2}
{"id": 31748, "text": "bkash: Apnar Tk 600 cash-out successful. Balance Tk 69,045. TrxID 6X4SMEK.", "response": {"label": 0, "techniques": [], "action": 0, "confidence": 0.9211602430227323}, "source": "pseudo", "weight": 0.2}
{"id": 66556, "text": "mCash: Apnar Tk 2,200 cash-out successful. Balance Tk 91,284. TrxID SUQXA2UAEM.", "response": {"label": 0, "techniques": [], "action": 0, "confidence": 0.9211502407024299}, "source": "pseudo", "weight": 0.2}
{"id": 33430, "text": "surecash: apnar tk 100 cash-out successful. balance tk 37,620. trxid deb8yqfx9. 🔥", "response": {"label": 0, "techniques": [], "action": 0, "confidence": 0.9209397854925111}, "source": "pseudo", "weight": 0.2}
{"id": 28929, "text": "bKash: Apnar Tk 900 cash-out successful. Balance Tk 34,423. TrxID 7T7WTQLBK.", "response": {"label": 0, "techniques": [], "action": 0, "confidence": 0.9207954810955195}, "source": "pseudo", "weight": 0.2}
{"id": 42649, "text": "A: I forgot my mCash PIN. How can I reset it? ||| B: Dial *322# and choose Reset PIN. Never share your PIN with anyone, including our staff. ||| A: Thank you", "response": {"label": 0, "techniques": [], "action": 0, "confidence": 0.9206400997438302}, "source": "pseudo", "weight": 0.2}
{"id": 67638, "text": "Security alert: new device logged in to your Nagad. If this was you, ignore. If not, call 16167. We never ask for your PIN or OTP.", "response": {"label": 0, "techniques": [], "action": 0, "confidence": 0.9205509579970603}, "source": "pseudo", "weight": 0.2}
{"id": 63560, "text": "apnar account e suspicious login detect hoyeche. card number, expiry ar CVV den. na hole account suspend hobe. - Nagad Security", "response": {"label": 1, "techniques": ["T1", "T2", "T5"], "action": 1, "confidence": 0.9697869625858447}, "source": "pseudo", "weight": 0.2}
{"id": 54843, "text": "mnbvcx Bai, apnar account e SUSpiciOUSlogni detect hoyeche. card number, expiry arCVV den. na hole aCCount suspend hobe.", "response": {"label": 1, "techniques": ["T1", "T2", "T5"], "action": 1, "confidence": 0.9618128310961841}, "source": "pseudo", "weight": 0.2}
{"id": 31306, "text": "Bhai, apnar account 24 ghontar moddhe band hoye jabe. card number, expiry ar CVV de. nahole ассount suspend hobe.", "response": {"label": 1, "techniques": ["T1", "T2", "T5"], "action": 1, "confidence": 0.9618082793672356}, "source": "pseudo", "weight": 0.2}
{"id": 38216, "text": "Nagad theke bolchi. apnar account e suspicious login detect hoyeche. card number, expiry ar CVV den.", "response": {"label": 1, "techniques": ["T1", "T2", "T5"], "action": 1, "confidence": 0.9614194615535311}, "source": "pseudo", "weight": 0.2}
{"id": 45607, "text": "Sabdhan! Rocket: apnar account 24 ghontar moddhe band hoye jabe. card number, expiry ar CVV den. na hole account suspend hobe.", "response": {"label": 1, "techniques": ["T1", "T2", "T5"], "action": 1, "confidence": 0.9578345881794237}, "source": "pseudo", "weight": 0.2}
{"id": 57295, "text": "bkash theke bolchi. apnar KYC update lagbe. card number, expiry ar CVV den. na hole account suspend hobe.", "response": {"label": 1, "techniques": ["T1", "T2", "T5"], "action": 1, "confidence": 0.9572843823524358}, "source": "pseudo", "weight": 0.2}
{"id": 48552, "text": "sabdhan! bkash: apnar account e suspicious login detect hoyeche. card number, expiry ar cvv den. na hole account suspend hobe.", "response": {"label": 1, "techniques": ["T1", "T2", "T5"], "action": 1, "confidence": 0.9558052020368597}, "source": "pseudo", "weight": 0.2}
{"id": 66144, "text": "Sabdhan! Upay: apnar account e suspicious login detect hoyeche. card number, expiry ar CVV den. na hole account suspend hobe. 🎁", "response": {"label": 1, "techniques": ["T1", "T2", "T5"], "action": 1, "confidence": 0.9552715202829968}, "source": "pseudo", "weight": 0.2}
{"id": 57853, "text": "Sabdhan! bKash: apnar account 24 ghontar moddhe band hoye jabe card number, expiry ar CVV den na hole account suspend hobe 🔥", "response": {"label": 1, "techniques": ["T1", "T2", "T5"], "action": 1, "confidence": 0.9550873747013944}, "source": "pseudo", "weight": 0.2}
{"id": 65452, "text": "upay theke bolchi. apnar kyc update lagbe. card number, expiry ar cvv den. na hole account suspend hobe.", "response": {"label": 1, "techniques": ["T1", "T2", "T5"], "action": 1, "confidence": 0.9548257536901337}, "source": "pseudo", "weight": 0.2}
{"id": 60060, "text": "Sabdhan! Tap: apnar account e suspicious login detect hoyeche. card number, expiry ar CVV den. na hole account suspend hobe.", "response": {"label": 1, "techniques": ["T1", "T2", "T5"], "action": 1, "confidence": 0.9546208224687941}, "source": "pseudo", "weight": 0.2}
{"id": 39431, "text": "S: Tap theke bolchi. apnar account 24 ghontar moddhe band hoye jabe. card number, expiry ar CVV den. na hole account suspend hobe. Dhonyobad.", "response": {"label": 1, "techniques": ["T1", "T2", "T5"], "action": 1, "confidence": 0.9537733719229572}, "source": "pseudo", "weight": 0.2}
{"id": 46449, "text": "S: Apu, ekta kotha chilo ||| R: bolen ||| S: mCash theke bolchi. apnar account e suspicious login detect hoyeche. card number, expiry ar CVV den. ||| R: really? ||| S: na hole account suspend hobe. - mCash Security ||| R: ki hoyeche?", "response": {"label": 1, "techniques": ["T1", "T2", "T5"], "action": 1, "confidence": 0.9535332894607921}, "source": "pseudo", "weight": 0.2}
{"id": 32236, "text": "apnar card flag hoyeche. card number, expiry ar cvv den. na hole account suspend hobe.", "response": {"label": 1, "techniques": ["T1", "T2", "T5"], "action": 1, "confidence": 0.9534273596952824}, "source": "pseudo", "weight": 0.2}
{"id": 23371, "text": "Priyo Rocket grahok, apnar KYC update lagbe. card number, expiry ar CVV den. na hole account suspend hobe.", "response": {"label": 1, "techniques": ["T1", "T2", "T5"], "action": 1, "confidence": 0.9529335719837546}, "source": "pseudo", "weight": 0.2}
{"id": 21960, "text": "bhai, apnar account e suspicious login detect hoyeche. card number, expiry ar cvv den.", "response": {"label": 1, "techniques": ["T1", "T2", "T5"], "action": 1, "confidence": 0.9523920179992181}, "source": "pseudo", "weight": 0.2}
{"id": 51522, "text": "Islami Bank security department theke. apnar account e suspicious login detect hoyeche. card number, expiry ar CVV den. na hole account suspend hobe.", "response": {"label": 1, "techniques": ["T1", "T2", "T5"], "action": 1, "confidence": 0.9523425384783446}, "source": "pseudo", "weight": 0.2}
{"id": 33571, "text": "bkash theke bolchi. apnar account e suspicious login detect hoyeche. card number, expiry ar cvv den.", "response": {"label": 1, "techniques": ["T1", "T2", "T5"], "action": 1, "confidence": 0.9518825160630158}, "source": "pseudo", "weight": 0.2}
{"id": 49895, "text": "bkash theke bolchi. apnar account 24 ghontar moddhe band hoye jabe. card number, expiry ar cvv den.", "response": {"label": 1, "techniques": ["T1", "T2", "T5"], "action": 1, "confidence": 0.9508826764786061}, "source": "pseudo", "weight": 0.2}
{"id": 24571, "text": "Pathao Pay customer care: apnar account e suspicious login detect hoyeche. card number, expiry ar CVV den. na hole account suspend hobe.", "response": {"label": 1, "techniques": ["T1", "T2", "T5"], "action": 1, "confidence": 0.9501943981098917}, "source": "pseudo", "weight": 0.2}
{"id": 36421, "text": "Prime Bank security department theke. apnar account 24 ghontar moddhe band hoye jabe. card number, expiry ar CVV den. na hole account suspend hobe.", "response": {"label": 1, "techniques": ["T1", "T2", "T5"], "action": 1, "confidence": 0.9491589257902839}, "source": "pseudo", "weight": 0.2}
{"id": 21148, "text": "Bhai, apnar wallet limit barano hobe card number, expiry ar CVV den na hole account suspend hobe", "response": {"label": 1, "techniques": ["T1", "T2", "T5"], "action": 1, "confidence": 0.9485538672185525}, "source": "pseudo", "weight": 0.2}
{"id": 69447, "text": "bkash theke bolchi. apnar card flag hoyeche. card number, expiry ar cvv den. na hole account suspend hobe.", "response": {"label": 1, "techniques": ["T1", "T2", "T5"], "action": 1, "confidence": 0.9472471773110035}, "source": "pseudo", "weight": 0.2}
{"id": 55610, "text": "Priyo Nagad grahok, apnar number e Tk 400 refund ashche. PIN ta ekhoni reply korun. na hole account suspend hobe.", "response": {"label": 1, "techniques": ["T1", "T2", "T5"], "action": 1, "confidence": 0.9461490859568631}, "source": "pseudo", "weight": 0.2}
{"id": 62242, "text": "Rocket theke bolchi. apnar card flag hoyeche. card number, expiry ar CVV den. na hole account suspend hobe.", "response": {"label": 1, "techniques": ["T1", "T2", "T5"], "action": 1, "confidence": 0.9455996894817352}, "source": "pseudo", "weight": 0.2}
{"id": 61391, "text": "Nagad theke bolchi. apnar account 24 ghontar moddhe band hoye jabe. PIN ta ekhoni reply korun. na hole account suspend hobe. - Nagad Security", "response": {"label": 1, "techniques": ["T1", "T2", "T5"], "action": 1, "confidence": 0.9453131420543498}, "source": "pseudo", "weight": 0.2}
{"id": 52861, "text": "upay theke bolchi. apnar account e suspicious login detect hoyeche. otp ta ekhoni share korun. na hole account suspend hobe.", "response": {"label": 1, "techniques": ["T1", "T2", "T5"], "action": 1, "confidence": 0.9435184937102095}, "source": "pseudo", "weight": 0.2}
{"id": 28537, "text": "S: Bha​i, apnar account zzqwpo ⚠️ 24 ghontar moddhe band hoye jabe.N​ID number ar jonmo tarikhpthan. na hole account suspend hobe. ||| xkcdpl R: ki bolchen?", "response": {"label": 1, "techniques": ["T1", "T2", "T5"], "action": 1, "confidence": 0.9430044768809636}, "source": "pseudo", "weight": 0.2}
{"id": 25511, "text": "apnar account 24 ghontar moddhe band hoye jabe. hxxps://rokomarikyc[.]cc/login e giye pin din. na hole account suspend hobe.", "response": {"label": 1, "techniques": ["T1", "T2", "T5"], "action": 1, "confidence": 0.9424660815316344}, "source": "pseudo", "weight": 0.2}
{"id": 67006, "text": "S: Apu, ekta kotha chilo ||| R: bolen ||| S: প্রিয় গ্রাহক, আপনার Nagad অ্যাকাউন্ট বন্ধ হয়ে যাবে। এখনই আপনার পিন নম্বর পাঠান। ||| R: let me check 😊", "response": {"label": 1, "techniques": ["T1", "T2", "T5"], "action": 1, "confidence": 0.9408701620133457}, "source": "pseudo", "weight": 0.2}
{"id": 49686, "text": "Priyo mCash grahok, apnar account 24 ghontar moddhe band hoye jabe. card number, expiry ar CVV den.", "response": {"label": 1, "techniques": ["T1", "T2", "T5"], "action": 1, "confidence": 0.9406918045775327}, "source": "pseudo", "weight": 0.2}
{"id": 58220, "text": "Assalamu alaikum, ami Rocket officer. apnar account 24 ghontar moddhe band hoye jabe. card number, expiry ar CVV den.", "response": {"label": 1, "techniques": ["T1", "T2", "T5"], "action": 1, "confidence": 0.9404212522132154}, "source": "pseudo", "weight": 0.2}
{"id": 46311, "text": "S: প্রিয় গ্রাহক, আপনার Nagad অ্যাকাউন্ট বন্ধ হয়ে যাবে। এখনই আপনার পিন নম্বর পাঠান। ||| R: really?", "response": {"label": 1, "techniques": ["T1", "T2", "T5"], "action": 1, "confidence": 0.939459905490276}, "source": "pseudo", "weight": 0.2}
{"id": 28952, "text": "Priyo SureCash grahok, apnar account e suspicious login detect hoyeche. card number, expiry ar CVV den.", "response": {"label": 1, "techniques": ["T1", "T2", "T5"], "action": 1, "confidence": 0.9392610413791317}, "source": "pseudo", "weight": 0.2}
{"id": 61880, "text": "SureCash theke bolchi. apnar account 24 ghontar moddhe band hoye jabe. OTP ta ekhoni share korun. na hole account suspend hobe.", "response": {"label": 1, "techniques": ["T1", "T2", "T5"], "action": 1, "confidence": 0.9392588429892422}, "source": "pseudo", "weight": 0.2}
{"id": 49189, "text": "Priyo Nagad grahok, apnar account e suspicious login detect hoyeche. NID number ar jonmo tarikh pathan. na hole account suspend hobe. - Nagad Security", "response": {"label": 1, "techniques": ["T1", "T2", "T5"], "action": 1, "confidence": 0.9389449194046783}, "source": "pseudo", "weight": 0.2}
{"id": 51305, "text": "[Sender]: Good morning sir ||| [You]: Good morning, who's this? ||| [Sender]: apnar account 24 ghontar moddhe band hoye jabe. card number, expiry ar CVV den. na hole account suspend hobe. 🔥", "response": {"label": 1, "techniques": ["T1", "T2", "T5"], "action": 1, "confidence": 0.93852417087132}, "source": "pseudo", "weight": 0.2}
{"id": 21856, "text": "priyo nagad grahok, apnar sim registration adhura. card number, expiry ar cvv den. na hole account suspend hobe.", "response": {"label": 1, "techniques": ["T1", "T2", "T5"], "action": 1, "confidence": 0.9378082440320304}, "source": "pseudo", "weight": 0.2}
{"id": 45967, "text": "assalamu alaikum, ami rocket officer. apnar account e suspicious login detect hoyeche. card number, expiry ar cvv den.", "response": {"label": 1, "techniques": ["T1", "T2", "T5"], "action": 1, "confidence": 0.9374744135526952}, "source": "pseudo", "weight": 0.2}
{"id": 51259, "text": "sabdhan! nagad: apnar account 24 ghontar moddhe band hoye jabe. pin ta ekhoni reply korun. na hole account suspend hobe.", "response": {"label": 1, "techniques": ["T1", "T2", "T5"], "action": 1, "confidence": 0.9372131780157436}, "source": "pseudo", "weight": 0.2}
{"id": 47092, "text": "Priyo mCash grahok, apnar wallet limit barano hobe. card number, expiry ar CVV den. na hole account suspend hobe.", "response": {"label": 1, "techniques": ["T1", "T2", "T5"], "action": 1, "confidence": 0.9354938138141087}, "source": "pseudo", "weight": 0.2}
{"id": 27257, "text": "Bhai, apnar account e suspicious login detect hoyeche. bit[.]ly/0qp2b7p e giye PIN din. na hole account suspend hobe.", "response": {"label": 1, "techniques": ["T1", "T2", "T5"], "action": 1, "confidence": 0.9346497165926119}, "source": "pseudo", "weight": 0.2}
{"id": 60353, "text": "Subject: Your bKash update. Dear Valued Customer, প্রিয় গ্রাহক, আপনার bKash অ্যাকাউন্ট বন্ধ হয়ে যাবে। এখনই আপনার পিন নম্বর পাঠান। Sincerely, Support Team", "response": {"label": 1, "techniques": ["T1", "T2", "T5"], "action": 1, "confidence": 0.9345322430012468}, "source": "pseudo", "weight": 0.2}
{"id": 45941, "text": "Assalamu alaikum, ami mCash officer. apnar account e suspicious login detect hoyeche. PIN ta ekhoni reply korun. na hole account suspend hobe.", "response": {"label": 1, "techniques": ["T1", "T2", "T5"], "action": 1, "confidence": 0.93395239532825}, "source": "pseudo", "weight": 0.2}
{"id": 49433, "text": "Upay customer care: apnar wallet limit barano hobe card number, expiry ar CVV den na hole account suspend hobe", "response": {"label": 1, "techniques": ["T1", "T2", "T5"], "action": 1, "confidence": 0.933292563665315}, "source": "pseudo", "weight": 0.2}
{"id": 56313, "text": "Rocket customer care: apnar account e suspicious login detect hoyeche. card number, expiry ar CVV den.", "response": {"label": 1, "techniques": ["T1", "T2", "T5"], "action": 1, "confidence": 0.9325212721323546}, "source": "pseudo", "weight": 0.2}
{"id": 58713, "text": "Subject: Important notice regarding your account. Dear Sir/Madam, প্রিয় গ্রাহক, আপনার SureCash অ্যাকাউন্ট বন্ধ হয়ে যাবে। এখনই আপনার পিন নম্বর পাঠান। Sincerely, Support Team", "response": {"label": 1, "techniques": ["T1", "T2", "T5"], "action": 1, "confidence": 0.9324587949624504}, "source": "pseudo", "weight": 0.2}
{"id": 24754, "text": "assalamu alaikum, ami mcash officer. apnar card flag hoyeche. card number, expiry ar cvv den. na hole account suspend hobe.", "response": {"label": 1, "techniques": ["T1", "T2", "T5"], "action": 1, "confidence": 0.9319287682350126}, "source": "pseudo", "weight": 0.2}
{"id": 31138, "text": "Sir/Madam, apnar account e suspicious login detect hoyeche card number, expiry ar CVV den 30 minute er moddhe korun", "response": {"label": 1, "techniques": ["T1", "T2", "T5"], "action": 1, "confidence": 0.9318614116247861}, "source": "pseudo", "weight": 0.2}
{"id": 60153, "text": "Sabdhan! bKash: apnar KYC update lagbe. card number, expiry ar CVV den. na hole account suspend hobe.", "response": {"label": 1, "techniques": ["T1", "T2", "T5"], "action": 1, "confidence": 0.9315286180213824}, "source": "pseudo", "weight": 0.2}
{"id": 68713, "text": "Pathao Pay theke bolchi. apnar number e Tk 250 refund ashche. card number, expiry ar CVV den. na hole account suspend hobe.", "response": {"label": 1, "techniques": ["T1", "T2", "T5"], "action": 1, "confidence": 0.9311138473945699}, "source": "pseudo", "weight": 0.2}
{"id": 69261, "text": "Pathao Pay theke bolchi. apnar account e suspicious login detect hoyeche. card number, expiry ar CVV den. 😊", "response": {"label": 1, "techniques": ["T1", "T2", "T5"], "action": 1, "confidence": 0.9310620030592981}, "source": "pseudo", "weight": 0.2}
{"id": 67053, "text": "Caller: apnar account e suspicious login detect hoyeche. ||| You: yes ||| Caller: card number, expiry ar CVV den.", "response": {"label": 1, "techniques": ["T1", "T2", "T5"], "action": 1, "confidence": 0.9309784488854332}, "source": "pseudo", "weight": 0.2}
{"id": 24389, "text": "[Sender]: Sir, apni ki Upay use koren? ||| [You]: ha, keno? ||| [Sender]: প্রিয় গ্রাহক, আপনার Upay অ্যাকাউন্ট বন্ধ হয়ে যাবে। এখনই আপনার পিন নম্বর পাঠান।", "response": {"label": 1, "techniques": ["T1", "T2", "T5"], "action": 1, "confidence": 0.9305838114188395}, "source": "pseudo", "weight": 0.2}
{"id": 33489, "text": "Tap customer care: apnar account 24 ghontar moddhe band hoye jabe. PIN ta ekhoni reply korun. na hole account suspend hobe.", "response": {"label": 1, "techniques": ["T1", "T2", "T5"], "action": 1, "confidence": 0.9294581342842257}, "source": "pseudo", "weight": 0.2}
{"id": 65254, "text": "[Sender]: Apu, ekta kotha chilo ||| [You]: bolen ||| [Sender]: প্রিয় গ্রাহক, আপনার bKash অ্যাকাউন্ট বন্ধ হয়ে যাবে। এখনই আপনার পিন নম্বর পাঠান। ||| [You]: but why? ⚠️", "response": {"label": 1, "techniques": ["T1", "T2", "T5"], "action": 1, "confidence": 0.9290888512168842}, "source": "pseudo", "weight": 0.2}
{"id": 26079, "text": "pubali bank security department theke. apnar account 24 ghontar moddhe band hoye jabe. card number, expiry ar cvv den. na hole account suspend hobe. dhonyobad.", "response": {"label": 1, "techniques": ["T1", "T2", "T5"], "action": 1, "confidence": 0.9286339135750429}, "source": "pseudo", "weight": 0.2}
{"id": 40207, "text": "Priyo Nagad grahok, apnar card flag hoyeche. hxxp://islamibank-update[.]online/verify e giye PIN din. na hole account suspend hobe.", "response": {"label": 1, "techniques": ["T1", "T2", "T5"], "action": 1, "confidence": 0.9283986480373555}, "source": "pseudo", "weight": 0.2}
{"id": 44713, "text": "mCash customer care: apnar account e suspicious login detect hoyeche. OTP ta ekhoni share korun. na hole account suspend hobe.", "response": {"label": 1, "techniques": ["T1", "T2", "T5"], "action": 1, "confidence": 0.9278222746157151}, "source": "pseudo", "weight": 0.2}
{"id": 41718, "text": "Priyo SureCash grahok, apnar account 24 ghontar moddhe band hoye jabe. bit[.]ly/4sfy7 e giye PIN din. na hole account suspend hobe.", "response": {"label": 1, "techniques": ["T1", "T2", "T5"], "action": 1, "confidence": 0.9273516605360026}, "source": "pseudo", "weight": 0.2}
{"id": 26471, "text": "MCASH LUCKY DRAW: YOUR MCASH ACCOUNT WON A TK 5,000 REWARD. PAY TK 60 PROCESSING FEE TO RECEIVE IT. OFFER EXPIRES IN 1 HOUR!", "response": {"label": 2, "techniques": ["T1", "T3", "T4"], "action": 2, "confidence": 0.9618515354792374}, "source": "pseudo", "weight": 0.2}
{"id": 41139, "text": "Tap Lucky Draw: You have won a free smartphone! Pay Tk 500 processing fee to receive it.", "response": {"label": 2, "techniques": ["T1", "T3", "T4"], "action": 2, "confidence": 0.9596023421310239}, "source": "pseudo", "weight": 0.2}
{"id": 46967, "text": "bKash Lucky DrAW:yOU wOn cashbackof Tk 10,000! Paу 🎁 Tk 60delivery сhargе. Offr expires in 1 hour! Thankyou for yurcooooooperation. 🙏", "response": {"label": 2, "techniques": ["T1", "T4"], "action": 3, "confidence": 0.958476324158904}, "source": "pseudo", "weight": 0.2}
{"id": 50334, "text": "bKash Lucky Draw: You are eligible for a free gift: a free smartphone. Pay Tk 500 delivery charge. Offer expires in 1 hour!", "response": {"label": 2, "techniques": ["T3", "T4"], "action": 2, "confidence": 0.9582381780477484}, "source": "pseudo", "weight": 0.2}
{"id": 61329, "text": "অভিনন্দন! আপনি ১৫,০০০ টাকা ক্যাশব্যাক জিতেছেন। পেতে hxxps://rokomari-gift[.]online/claim এ ক্লিক করুন।", "response": {"label": 2, "techniques": ["T4", "T5"], "action": 3, "confidence": 0.9574746980080453}, "source": "pseudo", "weight": 0.2}
{"id": 48279, "text": "This is an automated notification from our system. SureCash Lucky Draw: You won cashback of Tk 5,000! Pay Tk 25 processing fee to receive it. Offer expires in 1 hour!", "response": {"label": 2, "techniques": ["T1", "T3", "T4"], "action": 2, "confidence": 0.9573357652689206}, "source": "pseudo", "weight": 0.2}
{"id": 42304, "text": "SureCash Lucky Draw: You have won a free smartphone! Pay Tk 300 processing fee to receive it.", "response": {"label": 2, "techniques": ["T1", "T3", "T4"], "action": 2, "confidence": 0.9568810681871238}, "source": "pseudo", "weight": 0.2}
{"id": 42283, "text": "SureCash Lucky Draw: You have won a laptop! Pay Tk 25 processing fee to receive it. Offer expires in 1 hour!", "response": {"label": 2, "techniques": ["T1", "T3", "T4"], "action": 2, "confidence": 0.9550565963105271}, "source": "pseudo", "weight": 0.2}
{"id": 56864, "text": "You w​on a free smartрhоnе in the luсky draw Send a phto xkcdpl of your NID to claim", "response": {"label": 2, "techniques": ["T4"], "action": 2, "confidence": 0.9510413397598589}, "source": "pseudo", "weight": 0.2}
{"id": 69874, "text": "upay lucky draw: you are eligible for a free gift: a laptop. pay tk 120 processing fee to receive it. ||| ki bolchen? ||| offer expires in 1 hour!", "response": {"label": 2, "techniques": ["T1", "T3", "T4"], "action": 2, "confidence": 0.9502015322724833}, "source": "pseudo", "weight": 0.2}
{"id": 29114, "text": "Upay Lucky Draw: Your number was selected for iPhone 15. Pay Tk 60 processing fee to receive it. Offer expires in 1 hour!", "response": {"label": 2, "techniques": ["T1", "T3", "T4"], "action": 2, "confidence": 0.9490073340392017}, "source": "pseudo", "weight": 0.2}
{"id": 47503, "text": "SureCash qzxvbn Lucky Draw: You are our 100000th customer and won a Toyota car Pay Tk3​00 deliverycharge Offer expires in 1 hour!", "response": {"label": 2, "techniques": ["T4"], "action": 2, "confidence": 0.948417829513627}, "source": "pseudo", "weight": 0.2}
{"id": 44603, "text": "Pathao Pay Lucky Draw: You have won a Samsung S24! Pay Tk 300 processing fee to receive it. Offer expires in 1 hour!", "response": {"label": 2, "techniques": ["T1", "T3", "T4"], "action": 2, "confidence": 0.9460379111595734}, "source": "pseudo", "weight": 0.2}
{"id": 53196, "text": "Yоu won a Tk 50,000 gift cad in the lucky draw. Send a photo of yourNID to claim.", "response": {"label": 2, "techniques": ["T4"], "action": 2, "confidence": 0.946003326352605}, "source": "pseudo", "weight": 0.2}
{"id": 34525, "text": "Rocket Lucky Draw: Apni a free smartphone jitechen! Pete Tk 300 processing fee din. Taratari korun.", "response": {"label": 2, "techniques": ["T1", "T3", "T4"], "action": 2, "confidence": 0.9449275741057794}, "source": "pseudo", "weight": 0.2}
{"id": 32377, "text": "mCahs Lucky Draw: You 🙏 won cashback of Tk 2,500! Pay Tk 250 d​elivery charge. Offer expires in 1 hour!", "response": {"label": 2, "techniques": ["T4"], "action": 2, "confidence": 0.9434409625807872}, "source": "pseudo", "weight": 0.2}
{"id": 47758, "text": "Tap Lucky Draw: You won cashback of Tk 800! Pay Tk 25 processing fee to receive it.", "response": {"label": 2, "techniques": ["T1", "T3", "T4"], "action": 2, "confidence": 0.9421497462751525}, "source": "pseudo", "weight": 0.2}
{"id": 51824, "text": "Tap Lucky Draw: Apni Tk 5,000 cashback jitechen. Delivery charge Tk 99 bkash korun. 1 ghontar moddhe claim korun.", "response": {"label": 2, "techniques": ["T4"], "action": 3, "confidence": 0.9410268423231624}, "source": "pseudo", "weight": 0.2}
{"id": 27637, "text": "xkcdpl Tар Lucky Draw: Anpi Tk 400 casssshback jitechen. Pete Tk 300 processing fee dinnnn. Taratari korun.", "response": {"label": 2, "techniques": ["T1", "T3", "T4"], "action": 2, "confidence": 0.9405356562621031}, "source": "pseudo", "weight": 0.2}
{"id": 65426, "text": "Rocket Lucky Draw: Apni a free smartphone jitechen! Pete Tk 25 processing fee din.", "response": {"label": 2, "techniques": ["T3", "T4"], "action": 2, "confidence": 0.940442542269646}, "source": "pseudo", "weight": 0.2}
{"id": 34975, "text": "PATHAO PAY LUCKY DRAW: YOUR NUMBER WAS SELECTED FOR IPHONE 15. PAY TK 45 PROCESSING FEE TO RECEIVE IT. OFFER EXPIRES IN 1 HOUR!", "response": {"label": 2, "techniques": ["T1", "T3", "T4"], "action": 2, "confidence": 0.9397718470056632}, "source": "pseudo", "weight": 0.2}
{"id": 25558, "text": "Upay Lucky Draw: Apni a Tk 50,000 gift card jitechen! Delivery charge Tk 150 bkash korun.", "response": {"label": 2, "techniques": ["T4"], "action": 2, "confidence": 0.939476087065014}, "source": "pseudo", "weight": 0.2}
{"id": 28395, "text": "nagad lucky draw: you are eligible for a free gift: iphone 15. pay tk 99 processing fee to receive it.", "response": {"label": 2, "techniques": ["T1", "T3", "T4"], "action": 2, "confidence": 0.9391092227484013}, "source": "pseudo", "weight": 0.2}
{"id": 27265, "text": "bkash lucky draw: you are eligible for a free gift: a tk 50,000 gift card. pay tk 20 processing fee to receive it. hurry, limited winners only. 🔥", "response": {"label": 2, "techniques": ["T1", "T3", "T4"], "action": 2, "confidence": 0.9390137942593977}, "source": "pseudo", "weight": 0.2}
{"id": 37749, "text": "bkash Lucky Draw: You won cashback of Tk 5,000! Pay Tk 80 delivery charge Hurry, limited winners only", "response": {"label": 2, "techniques": ["T3", "T4"], "action": 2, "confidence": 0.938885708222961}, "source": "pseudo", "weight": 0.2}
{"id": 61400, "text": "bKash Lucky Draw: You won cashback of Tk 🙏 5,000! Paу Tk 80delivery charge. Claim wkrjfejejrn within 24hours or it wlil be cancelled.", "response": {"label": 2, "techniques": ["T3", "T4"], "action": 3, "confidence": 0.9386261998008785}, "source": "pseudo", "weight": 0.2}
{"id": 31499, "text": "mCash Lucky Draw: Apni Tk 20,000 cashback jitechen. Delivery charge Tk 150 bkash korun.", "response": {"label": 2, "techniques": ["T4"], "action": 2, "confidence": 0.9376896744505531}, "source": "pseudo", "weight": 0.2}
{"id": 24703, "text": "mCash Lucky Draw: You won cashback of Tk 150! Pay Tk 80 processing fee to receive it.", "response": {"label": 2, "techniques": ["T3", "T4"], "action": 2, "confidence": 0.9368806612636378}, "source": "pseudo", "weight": 0.2}
{"id": 44649, "text": "mCash Anniversary: You have won a Toyota car! Pay Tk 99 processing fee to receive it. Offer expires in 1 hour!", "response": {"label": 2, "techniques": ["T1", "T4"], "action": 2, "confidence": 0.9366406524954455}, "source": "pseudo", "weight": 0.2}
{"id": 25408, "text": "bkash Lucky Draw: Apni a free smartphone jitechen! Pete Tk 20 processing fee din.", "response": {"label": 2, "techniques": ["T3", "T4"], "action": 2, "confidence": 0.9350905704438032}, "source": "pseudo", "weight": 0.2}
{"id": 38875, "text": "অভিনন্দন! আপনি ৫,০০০ টাকা ক্যাশব্যাক জিতেছেন। পেতে hxxp://gp-verify[.]live/claim এ ক্লিক করুন।", "response": {"label": 2, "techniques": ["T4", "T5"], "action": 3, "confidence": 0.934990830047526}, "source": "pseudo", "weight": 0.2}
{"id": 23332, "text": "Tap Lucky Draw: You are eligible for a free gift: a Tk 50,000 gift card. Pay Tk 120 delivery charge.", "response": {"label": 2, "techniques": ["T3", "T4"], "action": 2, "confidence": 0.9347034553004023}, "source": "pseudo", "weight": 0.2}
{"id": 25797, "text": "Tap Anniversary: You won cashback of Tk 600! Pay Tk 300 delivery charge. Offer expires in 1 hour! 📲", "response": {"label": 2, "techniques": ["T1", "T4"], "action": 3, "confidence": 0.9345354111470735}, "source": "pseudo", "weight": 0.2}
{"id": 46135, "text": "Rocket Lucky Draw: Apnar number ti a Tk 50,000 gift card er jonno select hoyeche. Delivery charge Tk 150 bkash korun.", "response": {"label": 2, "techniques": ["T1", "T4"], "action": 5, "confidence": 0.9334285603974014}, "source": "pseudo", "weight": 0.2}
{"id": 57031, "text": "asdkjh pAThaO Pay lUCKy Draw: Aрnі a Tk 50,000 gift card jitechen! Delivery charge Tk 250bkash korun.", "response": {"label": 2, "techniques": ["T4"], "action": 2, "confidence": 0.9332158586668389}, "source": "pseudo", "weight": 0.2}
{"id": 21740, "text": "mCash Lucky Draw: Apni a Tk 50,000 gift card jitechen! Delivery charge Tk 150 bkash korun. Taratari korun.", "response": {"label": 2, "techniques": ["T4"], "action": 3, "confidence": 0.933099048498537}, "source": "pseudo", "weight": 0.2}
{"id": 32437, "text": "Tap Lucky Draw: Your number was selected for a laptop. Pay Tk 99 processing fee to receive it. Offer expires in 1 hour! 🎁", "response": {"label": 2, "techniques": ["T1", "T3", "T4"], "action": 2, "confidence": 0.9315134778023396}, "source": "pseudo", "weight": 0.2}
{"id": 60374, "text": "Pathao Pay Lucky Draw: Apni a free smartphone jitechen! Pete Tk 60 processing fee din.", "response": {"label": 2, "techniques": ["T1", "T3", "T4"], "action": 2, "confidence": 0.9308696380210618}, "source": "pseudo", "weight": 0.2}
{"id": 58099, "text": "bkash lucky draw: apni a free smartphone jitechen! pete tk 80 processing fee din. 1 ghontar moddhe claim korun.", "response": {"label": 2, "techniques": ["T1", "T3", "T4"], "action": 2, "confidence": 0.9307688828633676}, "source": "pseudo", "weight": 0.2}
{"id": 41384, "text": "You won a free smartphone in the lucky draw Send a photo of your NID to claim", "response": {"label": 2, "techniques": ["T4"], "action": 2, "confidence": 0.9303865846619487}, "source": "pseudo", "weight": 0.2}
{"id": 29363, "text": "SureCash Lucky Draw: You won cashback of Tk 8,500! Pay Tk 40 processing fee to receive it.", "response": {"label": 2, "techniques": ["T1", "T3", "T4"], "action": 2, "confidence": 0.9298281146438417}, "source": "pseudo", "weight": 0.2}
{"id": 65647, "text": "Tap Lucky Draw: Your Tap account won a Tk 25,000 reward. Pay Tk 150 processing fee to receive it.", "response": {"label": 2, "techniques": ["T1", "T3", "T4"], "action": 2, "confidence": 0.9297697745178014}, "source": "pseudo", "weight": 0.2}
{"id": 57759, "text": "Rocket Lucky Draw: You have won a free smartphone! Pay Tk 40 delivery charge.", "response": {"label": 2, "techniques": ["T3", "T4"], "action": 2, "confidence": 0.9294262423880119}, "source": "pseudo", "weight": 0.2}
{"id": 63476, "text": "surecash lucky draw: apni tk 400 cashback jitechen. pete tk 150 processing fee din. taratari korun.", "response": {"label": 2, "techniques": ["T1", "T3", "T4"], "action": 2, "confidence": 0.9290173453747976}, "source": "pseudo", "weight": 0.2}
{"id": 38865, "text": "Upay Lucky Draw: You are our 100,000th customer and won a Tk 1,00,000 prize. Pay Tk 45 delivery charge. Offer expires in 1 hour!", "response": {"label": 2, "techniques": ["T1", "T3", "T4"], "action": 2, "confidence": 0.9283403431982844}, "source": "pseudo", "weight": 0.2}
{"id": 31875, "text": "caller: mcash lucky draw: you are eligible for a free gift: iphone 15. ||| you: i see ||| caller: pay tk 120 processing fee to receive it.", "response": {"label": 2, "techniques": ["T1", "T3", "T4"], "action": 2, "confidence": 0.9278784276937806}, "source": "pseudo", "weight": 0.2}
{"id": 21138, "text": "BKASH LUCKY DRAW: APNI TK 50,000 CASHBACK JITECHEN. PETE TK 300 PROCESSING FEE DIN. 1 GHONTAR MODDHE CLAIM KORUN.", "response": {"label": 2, "techniques": ["T1", "T3", "T4"], "action": 2, "confidence": 0.9274724234391012}, "source": "pseudo", "weight": 0.2}
{"id": 59456, "text": "surecash lucky draw: apnar number ti iphone 15 er jonno select hoyeche. pete tk 120 processing fee din. 1 ghontar moddhe claim korun.", "response": {"label": 2, "techniques": ["T1", "T4"], "action": 5, "confidence": 0.9274617437741558}, "source": "pseudo", "weight": 0.2}
{"id": 36895, "text": "Rocket Lucky Draw: You have won a Toyota car! Pay Tk 45 processing fee to receive it.", "response": {"label": 2, "techniques": ["T1", "T3", "T4"], "action": 2, "confidence": 0.927361722873302}, "source": "pseudo", "weight": 0.2}
{"id": 29931, "text": "bkash lucky draw: apni a free smartphone jitechen! delivery charge tk 20 bkash korun.", "response": {"label": 2, "techniques": ["T4"], "action": 2, "confidence": 0.9269220920207456}, "source": "pseudo", "weight": 0.2}
{"id": 29129, "text": "Rocket Lucky Draw: Apni Tk 4,000 cashback jitechen. Pete Tk 300 processing fee din. Taratari korun.", "response": {"label": 2, "techniques": ["T1", "T3", "T4"], "action": 2, "confidence": 0.9269084161177116}, "source": "pseudo", "weight": 0.2}
{"id": 69623, "text": "Upay Lucky Draw: You are our 100,000th customer and won a Toyota car. Pay Tk 99 delivery charge.", "response": {"label": 2, "techniques": ["T3", "T4"], "action": 2, "confidence": 0.9268981530424348}, "source": "pseudo", "weight": 0.2}
{"id": 44397, "text": "Nagad Lucky Draw: You are our 100,000th customer and won a Toyota car. Pay Tk 40 processing fee to receive it.", "response": {"label": 2, "techniques": ["T1", "T3", "T4"], "action": 2, "confidence": 0.9268047014771587}, "source": "pseudo", "weight": 0.2}
{"id": 49120, "text": "Pathao Pay Lucky Draw: You won cashback of Tk 8,500! Pay Tk 250 processing fee to receive it. Hurry, limited winners only.", "response": {"label": 2, "techniques": ["T3", "T4"], "action": 2, "confidence": 0.9264888005864084}, "source": "pseudo", "weight": 0.2}
{"id": 23624, "text": "অভিনন্দন! আপনি ১০,০০০ টাকা ক্যাশব্যাক জিতেছেন। পেতে hxxp://banglalinkbonus[.]click/verify এ ক্লিক করুন।", "response": {"label": 2, "techniques": ["T4", "T5"], "action": 3, "confidence": 0.92612527871982}, "source": "pseudo", "weight": 0.2}
{"id": 57036, "text": "You won a Tk 50,000 gift card in the lucky draw. Send a photo of your NID to claim. 🔥", "response": {"label": 2, "techniques": ["T4"], "action": 2, "confidence": 0.9260563176166563}, "source": "pseudo", "weight": 0.2}
{"id": 49777, "text": "bkash Lucky Draw: Apnar number ti a free smartphone er jonno select hoyeche. Pete Tk 60 processing fee din.", "response": {"label": 2, "techniques": ["T1", "T3", "T4"], "action": 2, "confidence": 0.9259485836291189}, "source": "pseudo", "weight": 0.2}
{"id": 28832, "text": "Tap Lucky Draw: Apni a free smartphone jitechen! Pete Tk 150 processing fee din. Offer ses hoye jabe!", "response": {"label": 2, "techniques": ["T1", "T3", "T4"], "action": 2, "confidence": 0.9247691579602014}, "source": "pseudo", "weight": 0.2}
{"id": 52394, "text": "অভিনন্দন! আপনি ১২,০০০ টাকা ক্যাশব্যাক জিতেছেন। পেতে hxxps://teletalk-gift[.]click/0dwu1 এ ক্লিক করুন।", "response": {"label": 2, "techniques": ["T4", "T5"], "action": 3, "confidence": 0.9243012189704353}, "source": "pseudo", "weight": 0.2}
{"id": 30752, "text": "অভিনন্দন! আপনি ৭,০‌০০ টাকাক্যাশ‌ব্যাক জিতেছেন। পেতে hxxps://robisupport[.]click/claim এক্লিক করুন।", "response": {"label": 2, "techniques": ["T4", "T5"], "action": 3, "confidence": 0.9240156877129679}, "source": "pseudo", "weight": 0.2}
{"id": 64184, "text": "Invest Tk 3,000 and get Tk 6,000 in 7 days, guaranteed. Pay Tk 300 registration fee to start.", "response": {"label": 3, "techniques": ["T3", "T6"], "action": 2, "confidence": 0.9697180802643477}, "source": "pseudo", "weight": 0.2}
{"id": 62335, "text": "ঘরে বসে প্রতিদিন ৩,০০০ টাকা আয় করুন। আজই হোয়াটসঅ্যাপে যোগ দিন। 📲", "response": {"label": 3, "techniques": ["T6"], "action": 4, "confidence": 0.9621303961249792}, "source": "pseudo", "weight": 0.2}
{"id": 57636, "text": "ঘরে বসে 🔥 প্রতিদিন ৫,০০০ টাকা আয় করুন।আজ‌ই হোয়াটসঅ্যাপে যোগ দিন।", "response": {"label": 3, "techniques": ["T6"], "action": 4, "confidence": 0.9607789751696237}, "source": "pseudo", "weight": 0.2}
{"id": 42801, "text": "qzxvbn ঘরে ব‌সে প্রতিদিন ৫,০০০ টাকা আয় করুন।আজই হোয়াটসঅ্যাপে যোগ দিন।", "response": {"label": 3, "techniques": ["T6"], "action": 4, "confidence": 0.9607789751696237}, "source": "pseudo", "weight": 0.2}
{"id": 35612, "text": "ঘরে বসে প্রতিদিন ৫,০০০ টাকা আয় করুন। আজই হোয়াটসঅ্যাপে যোগ দিন। ⚠️", "response": {"label": 3, "techniques": ["T6"], "action": 4, "confidence": 0.9607269669035438}, "source": "pseudo", "weight": 0.2}
{"id": 22446, "text": "[Group] Sarah: Friday at 5 ||| [Group] +8801XXXXXXXXX: ঘরে বসে প্রতিদিন ৭,০০০ টাকা আয় করুন। আজই হোয়াটসঅ্যাপে যোগ দিন।", "response": {"label": 3, "techniques": ["T6"], "action": 4, "confidence": 0.9595219232532716}, "source": "pseudo", "weight": 0.2}
{"id": 42300, "text": "invest tk 10,00and gt tk 3,0000000 in 🔥 7 days, guaranteed. deposittk 1,000 tounlock tasks.", "response": {"label": 3, "techniques": ["T3", "T6"], "action": 2, "confidence": 0.9570450047712347}, "source": "pseudo", "weight": 0.2}
{"id": 55739, "text": "ঘরে বসে প্রতিদিন ২০০০ 💰 টাকা আয‌় করুন। আজই হোয়াটসঅ্যাপে যোগ দ‌িন।", "response": {"label": 3, "techniques": ["T6"], "action": 4, "confidence": 0.9548130869352491}, "source": "pseudo", "weight": 0.2}
{"id": 51849, "text": "ঘ‌রে ⚠️ বসে প্রতিদিন ২৫,০০০ টাকা আয় করুন। আজই হোয়াটসঅ্যাপে যোগ দিন‌।", "response": {"label": 3, "techniques": ["T6"], "action": 4, "confidence": 0.9522179685205473}, "source": "pseudo", "weight": 0.2}
{"id": 46007, "text": "ঘরে বসে প্রতিদিন ২৫,০০০ টাকা আয় করুন। আজই হোয়াটসঅ্যাপে যোগ দিন। 📲", "response": {"label": 3, "techniques": ["T6"], "action": 4, "confidence": 0.9522179685205473}, "source": "pseudo", "weight": 0.2}
{"id": 64349, "text": "ঘরে বসে প্রতিদিন ৩,৫০০ টাকা আয় করুন। আজই হোয়াটসঅ্যাপে যোগ দিন। 🎁", "response": {"label": 3, "techniques": ["T6"], "action": 4, "confidence": 0.951752363988404}, "source": "pseudo", "weight": 0.2}
{"id": 66158, "text": "[Group] Nusrat: thanks bro ||| [Group] +8801XXXXXXXXX: ঘরে বসে প্রতিদিন ১২,০০০ টাকা আয় করুন। আজই হোয়াটসঅ্যাপে যোগ দিন।", "response": {"label": 3, "techniques": ["T6"], "action": 4, "confidence": 0.9513102220263168}, "source": "pseudo", "weight": 0.2}
{"id": 28900, "text": "ঘরে wkrjfejejrn বসে প্রতিদিন ১২,০০০ টাকা আ‌য় করুন। আজই হোয়াটসঅ্যাপে যো‌গ দিন।", "response": {"label": 3, "techniques": ["T6"], "action": 4, "confidence": 0.9493508082551652}, "source": "pseudo", "weight": 0.2}
{"id": 56932, "text": "invest tk 3,000 and get tk 6,000 in 2 days, guaranteed. pay tk 45 registration fee to start.", "response": {"label": 3, "techniques": ["T3", "T6"], "action": 2, "confidence": 0.9485989046969923}, "source": "pseudo", "weight": 0.2}
{"id": 42745, "text": "ঘরে বসে প্রতিদিন ৩০০ টাকা আয় করুন। আজই 🔥 হোয়াটসঅ্যাপে যোগ দিন।", "response": {"label": 3, "techniques": ["T6"], "action": 4, "confidence": 0.9483309352463083}, "source": "pseudo", "weight": 0.2}
{"id": 43479, "text": "ঘরে বসে প্রতিদিন ৭,০০০ টাকা আয় করুন। আজই হোয়াটসঅ্যাপে যোগ দিন। 💰", "response": {"label": 3, "techniques": ["T6"], "action": 4, "confidence": 0.9480450261553571}, "source": "pseudo", "weight": 0.2}
{"id": 25544, "text": "Invest Tk 3,000 and get Tk 9,000 in 3 days, guaranteed. Pay Tk 150 registration fee to start. Limited seats!", "response": {"label": 3, "techniques": ["T3", "T6"], "action": 2, "confidence": 0.9465127618774027}, "source": "pseudo", "weight": 0.2}
{"id": 22784, "text": "part-time job: like and share posts, tk 3,000 per task. deposit tk 3,000 to unlock tasks.", "response": {"label": 3, "techniques": ["T6"], "action": 4, "confidence": 0.9463974721932639}, "source": "pseudo", "weight": 0.2}
{"id": 34548, "text": "Invest 🎁 Tk 5,000 4nd get Tk 10,000 in 3 days, guaranteed. Pay Tk 45 registration feeto start.", "response": {"label": 3, "techniques": ["T3", "T6"], "action": 2, "confidence": 0.9463902339724171}, "source": "pseudo", "weight": 0.2}
{"id": 40182, "text": "Invest Tk 5,000 and get Tk 15,000 in 3 days, guaranteed. Deposit Tk 5,000 to unlock tasks.", "response": {"label": 3, "techniques": ["T3", "T6"], "action": 2, "confidence": 0.94545914985416}, "source": "pseudo", "weight": 0.2}
{"id": 46595, "text": "invest tk 3,000 and get tk 9,000 in 10 days, guaranteed. pay tk 20 registration fee to start. limited seats!", "response": {"label": 3, "techniques": ["T3", "T6"], "action": 2, "confidence": 0.9452371517056906}, "source": "pseudo", "weight": 0.2}
{"id": 45942, "text": "3,000 taka invest korun, 5 din e 9,000 taka pan, guaranteed. Registration fee 300 taka dite hobe. Seat kom!", "response": {"label": 3, "techniques": ["T3", "T6"], "action": 4, "confidence": 0.9449782166009082}, "source": "pseudo", "weight": 0.2}
{"id": 65398, "text": "mnbvcx ঘরে বসে প্রতিদিন ১২০০০ টাকা আয় করুন। আজই হোয়াটসঅ্যাপে যো‌গ দিন।", "response": {"label": 3, "techniques": ["T6"], "action": 4, "confidence": 0.9447441439217346}, "source": "pseudo", "weight": 0.2}
{"id": 46246, "text": "Invest Tk 20,000 and get Tk 60,000 in 10 days, guaranteed. Deposit Tk 20,000 to unlock tasks. 🔥", "response": {"label": 3, "techniques": ["T3", "T6"], "action": 2, "confidence": 0.9445550000397204}, "source": "pseudo", "weight": 0.2}
{"id": 67619, "text": "ঘরে বসে প্রতিদিন ১০০ টাকা আয় করুন। আজই হোয়াটসঅ্যাপে যোগ দিন। 😊", "response": {"label": 3, "techniques": ["T6"], "action": 4, "confidence": 0.9442476473103905}, "source": "pseudo", "weight": 0.2}
{"id": 20001, "text": "📲 ঘরে ব‌সে প্রতিদিন ১০০ টাকা আয‌় ক‌রুন।আজই হোয়াটসঅ্যাপে য‌োগ দি‌ন।", "response": {"label": 3, "techniques": ["T6"], "action": 4, "confidence": 0.9441177934191393}, "source": "pseudo", "weight": 0.2}
{"id": 42354, "text": "ঘরে বস‌ে প্র‌তিদিন ১৫০০০ টাকা আয় করুন। আজই হোয়াটসঅ্যা‌পে যোগ দিন।", "response": {"label": 3, "techniques": ["T6"], "action": 4, "confidence": 0.9438658265414018}, "source": "pseudo", "weight": 0.2}
{"id": 66917, "text": "ঘরে বসে প্রতিদিন 🙏 ২৫০০০ টাকা আয় করুন। xkcdpl আজই হোয়াটসঅ্যাপে যোগ দিন।", "response": {"label": 3, "techniques": ["T6"], "action": 4, "confidence": 0.9438308505121344}, "source": "pseudo", "weight": 0.2}
{"id": 37723, "text": "ঘরে বসে প্রতিদিন ২৫০০০ zzqwpo টাকা আয় করুন। আজই হোয়াটসঅ্যাপে যোগ দিন।", "response": {"label": 3, "techniques": ["T6"], "action": 4, "confidence": 0.9438308505121344}, "source": "pseudo", "weight": 0.2}
{"id": 67773, "text": "ঘর‌ে ব‌সে প্রত‌িদিন ৩,৫০০টাকা আয‌় করুন। আজ‌ই হোয়াটসঅ্যাপে যোগ দিন‌।", "response": {"label": 3, "techniques": ["T6"], "action": 4, "confidence": 0.9438114761344676}, "source": "pseudo", "weight": 0.2}
{"id": 68247, "text": "Invest Tk 12,000 and get Tk 24,000 in 7 days, guaranteed. Deposit Tk 12,000 to unlock tasks.", "response": {"label": 3, "techniques": ["T3", "T6"], "action": 2, "confidence": 0.9437764894799048}, "source": "pseudo", "weight": 0.2}
{"id": 69701, "text": "ঘরে বসে প্রতিদিন ১৫,০০০ টাকা আয় করুন। আজই হোয়াটসঅ্যাপে যোগ দিন। 🔥", "response": {"label": 3, "techniques": ["T6"], "action": 4, "confidence": 0.9430109343600568}, "source": "pseudo", "weight": 0.2}
{"id": 33421, "text": "asdkjh ঘরে বসে 🙏 প্‌রতিদিন ৭০০ টাকা আয় করু‌ন। আ‌জই হোয়াটসঅ‌্যাপে য‌োগ দিন।", "response": {"label": 3, "techniques": ["T6"], "action": 4, "confidence": 0.9420502773652056}, "source": "pseudo", "weight": 0.2}
{"id": 68099, "text": "[sender]: ঘরে বসে প্রতিদিন ১২,০০০ টাকা আয় করুন। আজই হোয়াটসঅ্যাপে যোগ দিন।", "response": {"label": 3, "techniques": ["T6"], "action": 4, "confidence": 0.9417279968699537}, "source": "pseudo", "weight": 0.2}
{"id": 60301, "text": "ঘরে বসে প্রতিদিন ৮০০ টাকা আয় করুন। আজই হোয়াটসঅ্যাপে যোগ দিন। 🔥", "response": {"label": 3, "techniques": ["T6"], "action": 4, "confidence": 0.9414323587274013}, "source": "pseudo", "weight": 0.2}
{"id": 48072, "text": "[Group] James: anyone has the slides? ||| [Group] +8801XXXXXXXXX: ঘরে বসে প্রতিদিন ১২,০০০ টাকা আয় করুন। আজই হোয়াটসঅ্যাপে যোগ দিন। 📲", "response": {"label": 3, "techniques": ["T6"], "action": 4, "confidence": 0.9406777765368454}, "source": "pseudo", "weight": 0.2}
{"id": 67866, "text": "Crypto bot trading, 30% daily profit guaranteed. Deposit Tk 3,000 to unlock tasks.", "response": {"label": 3, "techniques": ["T3", "T6"], "action": 2, "confidence": 0.9402789618100654}, "source": "pseudo", "weight": 0.2}
{"id": 31998, "text": "10,000 taka invest korun, 7 din e 30,000 taka pan, guaranteed. registration fee 300 taka dite hobe.", "response": {"label": 3, "techniques": ["T3", "T6"], "action": 2, "confidence": 0.9393798828209798}, "source": "pseudo", "weight": 0.2}
{"id": 43013, "text": "Invest Tk 50,000 and get Tk 100,000 in 7 days, guaranteed. Pay Tk 120 registration fee to start.", "response": {"label": 3, "techniques": ["T3", "T6"], "action": 2, "confidence": 0.9390499242165089}, "source": "pseudo", "weight": 0.2}
{"id": 67645, "text": "ঘরে বসে প্রতিদ‌িন ৫‌০০ টাকা 😊 আয় করুন। আজই হোয়াটসঅ্যাপে xkcdpl যোগ দিন।", "response": {"label": 3, "techniques": ["T6"], "action": 4, "confidence": 0.9389845700837793}, "source": "pseudo", "weight": 0.2}
{"id": 67521, "text": "ঘরে বসে প্রতিদিন ৫০০ টাকা আয় করুন। আজই হোয়াটসঅ্যাপে যোগ দিন। ||| ok", "response": {"label": 3, "techniques": ["T6"], "action": 4, "confidence": 0.9387409374320624}, "source": "pseudo", "weight": 0.2}
{"id": 56993, "text": "🎁 ঘরে বস‌ে প্রতিদিন ১,০০০ টা‌কা আয় করুন।আ‌জই হোয়াটসঅ্যাপে যোগ দিন।", "response": {"label": 3, "techniques": ["T6"], "action": 4, "confidence": 0.9383899432778828}, "source": "pseudo", "weight": 0.2}
{"id": 25599, "text": "ঘরে বসে প্রতিদিন ১,০০০ টাকা আয় করুন। আজই asdkjh হোয়াটসঅ্যাপে যোগ দিন।", "response": {"label": 3, "techniques": ["T6"], "action": 4, "confidence": 0.9383326987018229}, "source": "pseudo", "weight": 0.2}
{"id": 54864, "text": "ঘরে বসে প্রতিদিন ১,০০০ টাকা আয় করুন। আজই হোয়াটসঅ্যাপে যোগ দিন। 💰", "response": {"label": 3, "techniques": ["T6"], "action": 4, "confidence": 0.9383326987018229}, "source": "pseudo", "weight": 0.2}
{"id": 54964, "text": "ঘরে বসে প্রতিদিন xkcdpl ৪৫০০ ট‌াকা আয় ক‌রুন। আজ‌ই হোয়াটসঅ্যাপে যোগ দিন।", "response": {"label": 3, "techniques": ["T6"], "action": 4, "confidence": 0.9382286103004739}, "source": "pseudo", "weight": 0.2}
{"id": 29199, "text": "10,000 taka invest korun, 7 din e 20,000 taka pan, guaranteed. Registration fee 120 taka dite hobe.", "response": {"label": 3, "techniques": ["T3", "T6"], "action": 2, "confidence": 0.9382055515182393}, "source": "pseudo", "weight": 0.2}
{"id": 37735, "text": "Invest Tk 4,000 and get Tk 12,000 in 7 days, guaranteed. Deposit Tk 4,000 to unlock tasks. 😊", "response": {"label": 3, "techniques": ["T3", "T6"], "action": 2, "confidence": 0.9381769990270211}, "source": "pseudo", "weight": 0.2}
{"id": 38534, "text": "S: ঘরে বসে প্রতিদিন ১,০০০ টাকা আয় করুন। আজই হোয়াটসঅ্যাপে যোগ দিন।", "response": {"label": 3, "techniques": ["T6"], "action": 4, "confidence": 0.9378170427273695}, "source": "pseudo", "weight": 0.2}
{"id": 49339, "text": "invest tk 25,000 and get tk 75,000 in 10 days, guaranteed. deposit tk 25,000 to unlock tasks.", "response": {"label": 3, "techniques": ["T3", "T6"], "action": 2, "confidence": 0.9374436456595322}, "source": "pseudo", "weight": 0.2}
{"id": 45762, "text": "Invest Tk 20,000 and get Tk 60,000 in 7 days, guaranteed Pay Tk 500 registration fee to start Limited seats!", "response": {"label": 3, "techniques": ["T3", "T6"], "action": 2, "confidence": 0.9374121295817821}, "source": "pseudo", "weight": 0.2}
{"id": 44668, "text": "caller: [sender]: ঘরে বসে প্রতিদিন ৭,০০০ টাকা আয় করুন। আজই হোয়াটসঅ্যাপে যোগ দিন।", "response": {"label": 3, "techniques": ["T6"], "action": 4, "confidence": 0.9370809340956652}, "source": "pseudo", "weight": 0.2}
{"id": 29273, "text": "ঘরে বসে প্রতিদিন ২০০টাকা আয় 💰 করুন। আজই হোয়াটসঅ্যাপে যোগ দিন। 😊", "response": {"label": 3, "techniques": ["T6"], "action": 4, "confidence": 0.9365363014385635}, "source": "pseudo", "weight": 0.2}
{"id": 44465, "text": "invest tk 4,000 and get tk 8,000 in 3 days, guaranteed. pay tk 20 registration fee to start.", "response": {"label": 3, "techniques": ["T3", "T6"], "action": 2, "confidence": 0.9361433586779018}, "source": "pseudo", "weight": 0.2}
{"id": 38099, "text": "ঘরে বসে প্রতিদিন ৩৫০ টাকা আয় করুন। আজই হোয়াটসঅ্যাপে যোগ দিন। 📲", "response": {"label": 3, "techniques": ["T6"], "action": 4, "confidence": 0.9357721290804537}, "source": "pseudo", "weight": 0.2}
{"id": 31755, "text": "ঘরে বসে প্রতিদিন ২৫,০০০ টাকা আয় করুন। আজই হোয়াটসঅ্যাপেযোগ দি‌ন।", "response": {"label": 3, "techniques": ["T6"], "action": 4, "confidence": 0.9353709393857513}, "source": "pseudo", "weight": 0.2}
{"id": 60819, "text": "ঘরে বসে প্রতিদিন ২,০০০ zzqwpo টাকা 😊 আয‌়করুন। আজই হোয়াটসঅ্যাপে যোগ দিন।", "response": {"label": 3, "techniques": ["T6"], "action": 4, "confidence": 0.9351733657601943}, "source": "pseudo", "weight": 0.2}
{"id": 58600, "text": "Invest Tk 100 and get Tk 200 in 7 days, guaranteed. Pay Tk 80 registration fee to start.", "response": {"label": 3, "techniques": ["T3", "T6"], "action": 2, "confidence": 0.9345543146373247}, "source": "pseudo", "weight": 0.2}
{"id": 33532, "text": "Invest Tk 8,500 and get Tk 25,500 in 7 days, guaranteed. Pay Tk 300 registration fee to start.", "response": {"label": 3, "techniques": ["T3", "T6"], "action": 2, "confidence": 0.9344131138820577}, "source": "pseudo", "weight": 0.2}
{"id": 51781, "text": "S: ঘর‌ে বসে প্রতিদিন ৭,০০০টাকা আ‌য় করুন। আজই হোয়াটসঅ্যাপে যোগ দিন‌। ||| wkrjfejejrn R: 📲 ok", "response": {"label": 3, "techniques": ["T6"], "action": 4, "confidence": 0.9343423262947554}, "source": "pseudo", "weight": 0.2}
{"id": 69109, "text": "ঘরে বসে প্রতিদিন ৯‌০০ টাকা আয় করুন। qzxvbn আজই হোয়াটসঅ্যাপে যোগ দিন।", "response": {"label": 3, "techniques": ["T6"], "action": 4, "confidence": 0.9333267170290283}, "source": "pseudo", "weight": 0.2}
{"id": 59961, "text": "Janani Courier: Parcel CN31005 could not be delivered. Confirm address and pay Tk 40 redelivery charge: hxxps://skitto-fee[.]top/5evp47 Unpaid parcels will be destroyed.", "response": {"label": 4, "techniques": ["T1", "T3"], "action": 3, "confidence": 0.9791771387252621}, "source": "pseudo", "weight": 0.2}
{"id": 36882, "text": "আপনার পার্সেল আটকে আছে। ডেলিভারি চার্জ ১৫০ টাকা পরিশোধ করতে hxxps://ecourier-reward97[.]top এ ক্লিক করুন।", "response": {"label": 4, "techniques": ["T3"], "action": 2, "confidence": 0.9751439956700308}, "source": "pseudo", "weight": 0.2}
{"id": 56663, "text": "আপনার পার্সেল আটকে আছে। ডেলিভারি চার্জ ৪৫ টাকা পরিশোধ করতে hxxps://jananicourier-refund45[.]click এ ক্লিক করুন।", "response": {"label": 4, "techniques": ["T3"], "action": 2, "confidence": 0.9709964653238721}, "source": "pseudo", "weight": 0.2}
{"id": 25310, "text": "আপনার পার্সেল আটকে আছে। ডেলিভারি চার্জ ১৫০ টাকা পরিশোধ করতে hxxps://pathaocourier-track[.]xyz/1tui এ ক্লিক করুন।", "response": {"label": 4, "techniques": ["T3"], "action": 2, "confidence": 0.9703502267030168}, "source": "pseudo", "weight": 0.2}
{"id": 57056, "text": "Pathao Courier: Parcel CN90735 could not be delivered. Pay customs duty Tk 300 via bKash 017XXXXXX45. Act within 24 hours or it will be returned.", "response": {"label": 4, "techniques": ["T1", "T3", "T5"], "action": 2, "confidence": 0.9700646747857622}, "source": "pseudo", "weight": 0.2}
{"id": 43713, "text": "আপনার পার্সেল আটকে আছে। ডেলিভারি চার্জ ৯৯ টাকা পরিশোধ করতে hxxps://saparibahangift[.]live এ ক্লিক করুন।", "response": {"label": 4, "techniques": ["T3"], "action": 2, "confidence": 0.9700220507589076}, "source": "pseudo", "weight": 0.2}
{"id": 66864, "text": "pathao courier: parcel rx54122 could not be delivered. pay customs duty tk 250 via bkash 015xxxxxx20. act within 24 hours or it will be returned.", "response": {"label": 4, "techniques": ["T1", "T5"], "action": 2, "confidence": 0.9693132611000288}, "source": "pseudo", "weight": 0.2}
{"id": 29356, "text": "আপনার পার্সেল আটকে আছে। ডেলিভারি চার্জ ২০ টাকা পরিশোধ করতে hxxps://ucb-update[.]site/pay এ ক্লিক করুন।", "response": {"label": 4, "techniques": ["T3"], "action": 2, "confidence": 0.969131061973284}, "source": "pseudo", "weight": 0.2}
{"id": 27776, "text": "আপনার পার্সেল আটকে আছে। ডেলিভারি চার্জ ৪৫ টাকা পরিশোধ করতে hxxps://yellow-login[.]xyz/pay এ ক্লিক করুন।", "response": {"label": 4, "techniques": ["T3"], "action": 2, "confidence": 0.9688001186527129}, "source": "pseudo", "weight": 0.2}
{"id": 45985, "text": "আপনার পার্সেল আটকে আছে। ডেলিভারি চার্জ ২৫ টাকা পরিশোধ করতে hxxps://fee-saparibahan[.]live/login এ ক্লিক করুন।", "response": {"label": 4, "techniques": ["T3"], "action": 2, "confidence": 0.9684445942001043}, "source": "pseudo", "weight": 0.2}
{"id": 35419, "text": "caller: আপনার পার্সেল আটকে আছে। ডেলিভারি চার্জ ৪০ টাকা পরিশোধ করতে hxxps://shebadelivery-track[.]xyz/gr011 এ ক্লিক করুন।", "response": {"label": 4, "techniques": ["T3"], "action": 2, "confidence": 0.9683469340429562}, "source": "pseudo", "weight": 0.2}
{"id": 61513, "text": "আপনার পার্সেল আটকে আছে। ডেলিভারি চার্জ ২৫০ টাকা পরিশোধ করতে hxxps://fedex-update[.]icu/pay এ ক্লিক করুন।", "response": {"label": 4, "techniques": ["T3"], "action": 2, "confidence": 0.9680944047926958}, "source": "pseudo", "weight": 0.2}
{"id": 64866, "text": "আপনার পার্সেল আটকে আছে। ডেলিভারি চার্জ ১৫০ টাকা পরিশোধ করতে hxxps://ecourierverify[.]xyz/skjsrk এ ক্লিক করুন।", "response": {"label": 4, "techniques": ["T3"], "action": 2, "confidence": 0.9680444161215481}, "source": "pseudo", "weight": 0.2}
{"id": 23910, "text": "Pathao Courier: Your parcel EC98294 is on hold. Confirm address and pay Tk 20 redelivery charge: hxxps://ajkerdeal-refund17[.]top Act within 24 hours or it will be returned.", "response": {"label": 4, "techniques": ["T1", "T5"], "action": 3, "confidence": 0.9678840194024888}, "source": "pseudo", "weight": 0.2}
{"id": 40872, "text": "আপনার পার্সেল আটকে আছে। ডেলিভারি চার্জ ২৫ টাকা পরিশোধ করতে hxxps://fee-paperfly[.]xyz এ ক্লিক করুন।", "response": {"label": 4, "techniques": ["T3"], "action": 2, "confidence": 0.967206901842962}, "source": "pseudo", "weight": 0.2}
{"id": 52090, "text": "আপনার পার্সেল আটকে আছে। ডেলিভারি চার্জ ৬০ টাকা পরিশোধ করতে hxxps://tap-help60[.]live এ ক্লিক করুন।", "response": {"label": 4, "techniques": ["T3"], "action": 2, "confidence": 0.9672038773643926}, "source": "pseudo", "weight": 0.2}
{"id": 32849, "text": "আপনা‌র প‌ার্সেল আটক‌ে আছে। ডেলিভারি চার্জ ২৫টাকা পরিশোধ ক‌রতে hxxps://skitto-track[]link এ ক্লিক qzxvbn করুন।", "response": {"label": 4, "techniques": ["T3"], "action": 2, "confidence": 0.9668711061747772}, "source": "pseudo", "weight": 0.2}
{"id": 49061, "text": "আপনার পার্সেল আটকে আছে। ডেলিভারি চার্জ ৪০ টাকা পরিশোধ করতে hxxps://upay-help[.]click/pay এ ক্লিক করুন।", "response": {"label": 4, "techniques": ["T3"], "action": 2, "confidence": 0.9667179356168675}, "source": "pseudo", "weight": 0.2}
{"id": 27000, "text": "SA Paribahan: Parcel PF73870 could not be delivered. Pay customs duty Tk 900 via bKash 015XXXXXX15. Act within 24 hours or it will be returned.", "response": {"label": 4, "techniques": ["T1", "T3", "T5"], "action": 2, "confidence": 0.9661463998465151}, "source": "pseudo", "weight": 0.2}
{"id": 37103, "text": "Pathao Courier: Parcel EC87729 could not be delivered. Confirm address and pay Tk 45 redelivery charge: hxxp://login-airtel[.]info/7cp7gze Act within 24 hours or it will be returned.", "response": {"label": 4, "techniques": ["T1", "T5"], "action": 3, "confidence": 0.9660700574652986}, "source": "pseudo", "weight": 0.2}
{"id": 66457, "text": "আপনার পার্সেল আটকে আছে। ডেলিভারি চার্জ ৩০০ টাকা পরিশোধ করতে hxxps://sundarbancourier-help48[.]link/0qdv এ ক্লিক করুন।", "response": {"label": 4, "techniques": ["T3"], "action": 2, "confidence": 0.9659538122382769}, "source": "pseudo", "weight": 0.2}
{"id": 49004, "text": "Pathao Courier: Parcel PC57089 could not be delivered Pay customs duty Tk 20,000 via bKash +880 018XXXXXX43 Act within 24 hours or it will be returned", "response": {"label": 4, "techniques": ["T1", "T3", "T5"], "action": 2, "confidence": 0.9659130495622497}, "source": "pseudo", "weight": 0.2}
{"id": 22406, "text": "আপনার পার্সেল আটকে আছে। ডেলিভারি চার্জ ৮০ টাকা পরিশোধ করতে hxxps://saparibahan-login[.]net এ ক্লিক করুন।", "response": {"label": 4, "techniques": ["T3"], "action": 2, "confidence": 0.9657988766705514}, "source": "pseudo", "weight": 0.2}
{"id": 63490, "text": "আপনার পার্সেল আটকে আছে। ডেলিভারি চার্জ ১৫০ টাকা পরিশোধ করতে hxxps://fee-robi[.]cc এ ক্লিক করুন।", "response": {"label": 4, "techniques": ["T3"], "action": 2, "confidence": 0.9657719485018678}, "source": "pseudo", "weight": 0.2}
{"id": 21136, "text": "S: আপনার পার্সেল আটকে আছে। ডেলিভারি চার্জ ২০ টাকা পরিশোধ করতে hxxps://pathaocourier-help[.]cc/22uj এ ক্লিক করুন।", "response": {"label": 4, "techniques": ["T3"], "action": 2, "confidence": 0.9655271253368845}, "source": "pseudo", "weight": 0.2}
{"id": 40556, "text": "আপনার পার্সেল আটকে আছে। ডেলিভারি চার্জ ৪৫ টাকা পরিশোধ করতে hxxps://robibonus[.]icu এ ক্লিক করুন।", "response": {"label": 4, "techniques": ["T3"], "action": 2, "confidence": 0.9654996220178839}, "source": "pseudo", "weight": 0.2}
{"id": 63329, "text": "Pathao Courier: Parcel PC56332 could not be delivered. Confirm address and pay Tk 45 redelivery charge: hxxps://pathaocourier-claim62[.]com/verify Unpaid parcels will be destroyed.", "response": {"label": 4, "techniques": ["T1", "T5"], "action": 3, "confidence": 0.9651249863264275}, "source": "pseudo", "weight": 0.2}
{"id": 60625, "text": "আপনার পার্সেল আটকে আছে। ডেলিভারি চার্জ ৩০০ টাকা পরিশোধ করতে hxxps://sundarbancourier-bonus25[.]live এ ক্লিক করুন।", "response": {"label": 4, "techniques": ["T3"], "action": 2, "confidence": 0.9648972005320916}, "source": "pseudo", "weight": 0.2}
{"id": 64553, "text": "আপনার পার্সেল আটকে আছে। ডেলিভারি চার্জ ১২০ টাকা পরিশোধ করতে hxxps://dhl-fee98[.]org এ ক্লিক করুন।", "response": {"label": 4, "techniques": ["T3"], "action": 2, "confidence": 0.9648318586516881}, "source": "pseudo", "weight": 0.2}
{"id": 62325, "text": "janani courier: parcel pc93800 er address adhura. customs duty tk 400 bkash korun +880 018xxxxxx76 e.", "response": {"label": 4, "techniques": ["T1", "T3", "T5"], "action": 5, "confidence": 0.9643993017633807}, "source": "pseudo", "weight": 0.2}
{"id": 60478, "text": "আপনার পার্সেল আটকে আছে। ডেলিভারি চার্জ ২৫০ টাকা পরিশোধ করতে hxxps://sundarbancourier-login[.]online/pay এ ক্লিক করুন।", "response": {"label": 4, "techniques": ["T3"], "action": 2, "confidence": 0.9643375143858155}, "source": "pseudo", "weight": 0.2}
{"id": 32693, "text": "আপনার পার্সেল আটকে আছে। ডেলিভারি চার্জ ৯৯ টাকা পরিশোধ করতে hxxps://foodpanda-bonus72[.]net/pay এ ক্লিক করুন।", "response": {"label": 4, "techniques": ["T3"], "action": 2, "confidence": 0.9642568828756591}, "source": "pseudo", "weight": 0.2}
{"id": 59891, "text": "আপনার পার্সেল আটকে আছে। ডেলিভারি চার্জ ৫০০ টাকা পরিশোধ করতে hxxps://update-airtel[.]org/pay এ ক্লিক করুন।", "response": {"label": 4, "techniques": ["T3"], "action": 2, "confidence": 0.9642009102740475}, "source": "pseudo", "weight": 0.2}
{"id": 37995, "text": "আপনার পার্সেল আটকে আছে। ডেলিভারি চার্জ ৯৯ টাকা পরিশোধ করতে hxxps://saparibahanhelp[.]site/claim এ ক্লিক করুন।", "response": {"label": 4, "techniques": ["T3"], "action": 2, "confidence": 0.963876728096033}, "source": "pseudo", "weight": 0.2}
{"id": 35150, "text": "আপনার পার্সেল আটকে আছে। ডেলিভারি চার্জ ২০ টাকা পরিশোধ করতে hxxps://track-robi[.]xyz/pay এ ক্লিক করুন।", "response": {"label": 4, "techniques": ["T3"], "action": 2, "confidence": 0.9637600704108323}, "source": "pseudo", "weight": 0.2}
{"id": 35871, "text": "Sundarban Courier: The address on parcel RX90296 is incomplete. Pay customs duty Tk 2,500 via bKash 018XXXXXX80. Act within 24 hours or it will be returned.", "response": {"label": 4, "techniques": ["T1", "T5"], "action": 2, "confidence": 0.9636823775785397}, "source": "pseudo", "weight": 0.2}
{"id": 65359, "text": "Janani Courier: The address on parcel RX35927 is incomplete. Pay customs duty Tk 1,800 via bKash 018XXXXXX40. Act within 24 hours or it will be returned.", "response": {"label": 4, "techniques": ["T1", "T3", "T5"], "action": 2, "confidence": 0.9635679452816883}, "source": "pseudo", "weight": 0.2}
{"id": 41619, "text": "আপনার পার্সেল আটকে আছে। ডেলিভারি চার্জ ২০ টাকা পরিশোধ করতে hxxps://fee-tap[.]xyz/hsjpz এ ক্লিক করুন।", "response": {"label": 4, "techniques": ["T3"], "action": 2, "confidence": 0.9634758559062434}, "source": "pseudo", "weight": 0.2}
{"id": 45609, "text": "আপনার পার্সেল আটকে আছে। ডেলিভারি চার্জ ৮০ টাকা পরিশোধ করতে hxxps://login-redx[.]top/7jre7rj এ ক্লিক করুন।", "response": {"label": 4, "techniques": ["T3"], "action": 2, "confidence": 0.9632124853329501}, "source": "pseudo", "weight": 0.2}
{"id": 55279, "text": "আপনার পার্সেল আটকে আছে। ডেলিভারি চার্জ ৯৯ টাকা পরিশোধ করতে hxxps://steadfast-refund[.]cc/pay এ ক্লিক করুন।", "response": {"label": 4, "techniques": ["T3"], "action": 2, "confidence": 0.9630976930159805}, "source": "pseudo", "weight": 0.2}
{"id": 46768, "text": "আপনার পার্সেল আটকে আছে। ডেলিভারি চার্জ ১২০ টাকা পরিশোধ করতে hxxps://bonus-nagad[.]xyz/pay এ ক্লিক করুন।", "response": {"label": 4, "techniques": ["T3"], "action": 2, "confidence": 0.9629479381683442}, "source": "pseudo", "weight": 0.2}
{"id": 33803, "text": "আপনার পার্সেল আটকে আছে। ডেলিভারি চার্জ ১৫০ টাকা পরিশোধ করতে hxxps://ecouriertrack[.]org/login এ ক্লিক করুন।", "response": {"label": 4, "techniques": ["T3"], "action": 2, "confidence": 0.9628845403427732}, "source": "pseudo", "weight": 0.2}
{"id": 39614, "text": "আপনার পার্সেল আটকে আছে। 📲 ডেলিভারি zzqwpo চা‌র্জ ২৫ টাকা পরিশোধ করতে hxxps://rocketgift[.]top/7ssf এ ক্লিক করুন।", "response": {"label": 4, "techniques": ["T3"], "action": 2, "confidence": 0.9628397631395346}, "source": "pseudo", "weight": 0.2}
{"id": 21887, "text": "আপনার পার্সেল আটকে আছে। ডেলিভারি চার্জ ৫০০ টাকা পরিশোধ করতে hxxps://saparibahanrefund[.]net/ucax এ ক্লিক করুন।", "response": {"label": 4, "techniques": ["T3"], "action": 2, "confidence": 0.9626355670541576}, "source": "pseudo", "weight": 0.2}
{"id": 68736, "text": "আপনার পার্সেল আটকে আছে। ডেলিভারি চার্জ ৯৯ টাকা পরিশোধ করতে hxxps://robitrack[.]vip/pay এ ক্লিক করুন।", "response": {"label": 4, "techniques": ["T3"], "action": 2, "confidence": 0.9625715243674278}, "source": "pseudo", "weight": 0.2}
{"id": 32873, "text": "আপনার পার্সেল আটকে আছে। ডেলিভারি চার্জ ১২০ টাকা পরিশোধ করতে hxxps://pubalibank-update22[.]icu/84xvn এ ক্লিক করুন।", "response": {"label": 4, "techniques": ["T3"], "action": 2, "confidence": 0.962533763010583}, "source": "pseudo", "weight": 0.2}
{"id": 48066, "text": "Janani Courier: Parcel RX25172 could not be delivered. Confirm address and pay Tk 250 redelivery charge: hxxps://dutchbanglabank-track[.]live/915vy3 Unpaid parcels will be destroyed.", "response": {"label": 4, "techniques": ["T1", "T3"], "action": 3, "confidence": 0.96235282417305}, "source": "pseudo", "weight": 0.2}
{"id": 20067, "text": "আপনার পার্সেল আটকে আছে। ডেলিভারি চার্জ ১৫০ টাকা পরিশোধ করতে hxxps://tap-track[.]site/7be4 এ ক্লিক করুন।", "response": {"label": 4, "techniques": ["T3"], "action": 2, "confidence": 0.9623064785085778}, "source": "pseudo", "weight": 0.2}
{"id": 28974, "text": "আপনার পার্সেল আটকে আছে। ডেলিভারি চার্জ ২৫ টাকা পরিশোধ করতে hxxps://fee-aarong[.]icu/9dnahn এ ক্লিক করুন।", "response": {"label": 4, "techniques": ["T3"], "action": 2, "confidence": 0.9622177732723025}, "source": "pseudo", "weight": 0.2}
{"id": 33451, "text": "আপনার পার্সেল আটকে আছে। ডেলিভারি চার্জ ৬০ টাকা পরিশোধ করতে hxxps://standardcharteredgift[.]org এ ক্লিক করুন।", "response": {"label": 4, "techniques": ["T3"], "action": 2, "confidence": 0.9619436669084572}, "source": "pseudo", "weight": 0.2}
{"id": 54724, "text": "janani courier: apnar parcel rx95832 customs e atke ase. tk 99 fee din.", "response": {"label": 4, "techniques": ["T1", "T3"], "action": 2, "confidence": 0.9619066455342354}, "source": "pseudo", "weight": 0.2}
{"id": 30433, "text": "SA Paribahan: Parcel CN33486 could not be delivered. Confirm address and pay Tk 500 redelivery charge: hxxps://robikyc[.]org Act within 24 hours or it will be returned.", "response": {"label": 4, "techniques": ["T1", "T5"], "action": 3, "confidence": 0.961807439033723}, "source": "pseudo", "weight": 0.2}
{"id": 61606, "text": "আপনার পার্সেল আটকে আছে। ডেলিভারি চার্জ ২৫০ টাকা পরিশোধ করতে hxxps://banglalink-fee42[.]top/rw68 এ ক্লিক করুন।", "response": {"label": 4, "techniques": ["T3"], "action": 2, "confidence": 0.9618012220033004}, "source": "pseudo", "weight": 0.2}
{"id": 42153, "text": "আপনার পার্সেল আটকে আছে। ডেলিভারি চার্জ ৬০ টাকা পরিশোধ করতে hxxps://airtellogin[.]info/verify এ ক্লিক করুন।", "response": {"label": 4, "techniques": ["T3"], "action": 2, "confidence": 0.9617594621485331}, "source": "pseudo", "weight": 0.2}
{"id": 63832, "text": "আপনার পার্সেল আটকে আছে। ডেলিভারি চার্জ ১২০ টাকা পরিশোধ করতে hxxps://sundarbancourier-gift[.]co এ ক্লিক করুন।", "response": {"label": 4, "techniques": ["T3"], "action": 2, "confidence": 0.9616334808528829}, "source": "pseudo", "weight": 0.2}
{"id": 48887, "text": "আপনার পার্সেল আটকে আছে। ডেলিভারি চার্জ ৪৫ টাকা পরিশোধ করতে hxxps://shwapno-track[.]site/nfcrzx এ ক্লিক করুন।", "response": {"label": 4, "techniques": ["T3"], "action": 2, "confidence": 0.9615870353349631}, "source": "pseudo", "weight": 0.2}
{"id": 20456, "text": "আপনার পার্সেল আটকে আছে। ডেলিভারি চার্জ ৮০ টাকা পরিশোধ করতে hxxps://help-yellow[.]online এ ক্লিক করুন।", "response": {"label": 4, "techniques": ["T3"], "action": 2, "confidence": 0.9615362516814642}, "source": "pseudo", "weight": 0.2}
{"id": 26512, "text": "আপনার পার্সেল আটকে আছে। ডেলিভারি চার্জ ৬০ টাকা পরিশোধ করতে hxxps://ecourier-reward15[.]net/claim এ ক্লিক করুন।", "response": {"label": 4, "techniques": ["T3"], "action": 2, "confidence": 0.9614800041377269}, "source": "pseudo", "weight": 0.2}
{"id": 64677, "text": "আপনার পার্সেল আটকে আছে। ডেলিভারি চার্জ ২৫ টাকা পরিশোধ করতে hxxps://mcashkyc[.]site/pay এ ক্লিক করুন।", "response": {"label": 4, "techniques": ["T3"], "action": 2, "confidence": 0.961464521227643}, "source": "pseudo", "weight": 0.2}
{"id": 22350, "text": "eCourier: The address on parcel PC61024 is incomplete. Pay customs duty Tk 450 via bKash +880 017XXXXXX23. Act within 24 hours or it will be returned.", "response": {"label": 4, "techniques": ["T1", "T3", "T5"], "action": 2, "confidence": 0.9613438315959644}, "source": "pseudo", "weight": 0.2}
{"id": 32815, "text": "Upay: Eid offer! Sob item e 30% discount. Dial *121*7#.", "response": {"label": 5, "techniques": [], "action": 0, "confidence": 0.9727580268081861}, "source": "pseudo", "weight": 0.2}
{"id": 55655, "text": "eid offer! sob item e 30% discount. code: new25.", "response": {"label": 5, "techniques": [], "action": 0, "confidence": 0.9720793552422061}, "source": "pseudo", "weight": 0.2}
{"id": 63781, "text": "Eid offer! Sob item e 30% discount. Code: FLASH1545.", "response": {"label": 5, "techniques": [], "action": 0, "confidence": 0.9692879318597348}, "source": "pseudo", "weight": 0.2}
{"id": 31165, "text": "eіd offer! sob item e 30% discount.🙏", "response": {"label": 5, "techniques": [], "action": 0, "confidence": 0.9688905952106086}, "source": "pseudo", "weight": 0.2}
{"id": 58019, "text": "GP: Eid offer! Sob item e 30% discount.", "response": {"label": 5, "techniques": [], "action": 0, "confidence": 0.9647066102589666}, "source": "pseudo", "weight": 0.2}
{"id": 44099, "text": "Robi: Eid offer! Sob item e 20% discount. Code: FLASH1568.", "response": {"label": 5, "techniques": [], "action": 0, "confidence": 0.9646103182482306}, "source": "pseudo", "weight": 0.2}
{"id": 61652, "text": "Airtel: Eid offer! Sob item e 30% discount. Code: FLASH1589.", "response": {"label": 5, "techniques": [], "action": 0, "confidence": 0.9631450955719085}, "source": "pseudo", "weight": 0.2}
{"id": 41467, "text": "Eid offer! Sob item e 20% discount Code: FLASH15", "response": {"label": 5, "techniques": [], "action": 0, "confidence": 0.9629691863032841}, "source": "pseudo", "weight": 0.2}
{"id": 55745, "text": "eid offer! sob item e 30% discount. 16247 e call korun. ||| yellow: admission cholche spoken english course e. dial *121*3#. offer 30 nov porjonto. 🎁", "response": {"label": 5, "techniques": [], "action": 0, "confidence": 0.9623847600665463}, "source": "pseudo", "weight": 0.2}
{"id": 44580, "text": "Sonali Bank: Eid offer! Sob item e 30% discount. Code: BDFEST85.", "response": {"label": 5, "techniques": [], "action": 0, "confidence": 0.9621542455259425}, "source": "pseudo", "weight": 0.2}
{"id": 35243, "text": "Banglalink: Eid offer! Sob item e 50% discount. Code: EID10. Offer 30 Nov porjonto.", "response": {"label": 5, "techniques": [], "action": 0, "confidence": 0.9621332034168242}, "source": "pseudo", "weight": 0.2}
{"id": 50947, "text": "Jamuna Bank: Eid offer! Sob item e 20% discount. Code: EID10.", "response": {"label": 5, "techniques": [], "action": 0, "confidence": 0.9619011585716196}, "source": "pseudo", "weight": 0.2}
{"id": 43682, "text": "S: OK Wallet: Get 50% cashback on bill payments. Call 09611-XXXXXX. ||| S: Winter sale is live! Up to 50% off. Visit your nearest outlet. Offer ends Friday.", "response": {"label": 5, "techniques": [], "action": 0, "confidence": 0.9617626035346818}, "source": "pseudo", "weight": 0.2}
{"id": 23373, "text": "airtel: eid offеr! sob item e 40% discount code: eid10 xkcdpl offer fridayрorjontо", "response": {"label": 5, "techniques": [], "action": 0, "confidence": 0.9606693311862707}, "source": "pseudo", "weight": 0.2}
{"id": 58547, "text": "Rocket: Eid offer! Sob item e 30% discount. Dial *121*2#. Shortto projojjo.", "response": {"label": 5, "techniques": [], "action": 0, "confidence": 0.9604588019056951}, "source": "pseudo", "weight": 0.2}
{"id": 26518, "text": "Chaldal: Eid offer! Sob item e 50% discount. Code: SAVE2094.", "response": {"label": 5, "techniques": [], "action": 0, "confidence": 0.9591361670387816}, "source": "pseudo", "weight": 0.2}
{"id": 28341, "text": "UCB: Eid offer! Sob item e 20% discount.", "response": {"label": 5, "techniques": [], "action": 0, "confidence": 0.95910098204206}, "source": "pseudo", "weight": 0.2}
{"id": 67779, "text": "Eіd 📲 of​fer! Sоb іtem e 10% discount 📲", "response": {"label": 5, "techniques": [], "action": 0, "confidence": 0.9581100055130705}, "source": "pseudo", "weight": 0.2}
{"id": 24474, "text": "Bata: 🙏 Eid offer! Sob ITEM e 30% discount 09611-XXXXXX e call kourn Limited time!", "response": {"label": 5, "techniques": [], "action": 0, "confidence": 0.9560525475276821}, "source": "pseudo", "weight": 0.2}
{"id": 45560, "text": "Eid offer! Sob item e 10% discount Dial *123*3#", "response": {"label": 5, "techniques": [], "action": 0, "confidence": 0.9560298570449233}, "source": "pseudo", "weight": 0.2}
{"id": 69465, "text": "Eid offer! Sob item e 30% discount. Code: BDFEST38. Shortto projojjo.", "response": {"label": 5, "techniques": [], "action": 0, "confidence": 0.9548310162680925}, "source": "pseudo", "weight": 0.2}
{"id": 39955, "text": "StartEcH: ঈদ উপলক্ষে সব পণ্যে ২০% mnbvcx ⚠️ ছাড়! আজই কিনুন।", "response": {"label": 5, "techniques": [], "action": 0, "confidence": 0.9547510019602564}, "source": "pseudo", "weight": 0.2}
{"id": 66290, "text": "Eid offer! Sob item e 40% discount. Code: BDFEST. Offer Friday porjonto.", "response": {"label": 5, "techniques": [], "action": 0, "confidence": 0.9545233420533907}, "source": "pseudo", "weight": 0.2}
{"id": 41969, "text": "Eid mnbvcx offer! Sob item e 40% discount", "response": {"label": 5, "techniques": [], "action": 0, "confidence": 0.9539824168508815}, "source": "pseudo", "weight": 0.2}
{"id": 57791, "text": "Upay: Eid offer! Sob item e 40% discount. Dial *121*4#. Offer 30 Nov porjonto.", "response": {"label": 5, "techniques": [], "action": 0, "confidence": 0.9539496903734305}, "source": "pseudo", "weight": 0.2}
{"id": 42076, "text": "UCB: Eid offer! Sob item e 30% discount. Code: FLASH15. Limited time!", "response": {"label": 5, "techniques": [], "action": 0, "confidence": 0.9539147294658651}, "source": "pseudo", "weight": 0.2}
{"id": 69553, "text": "admission going on for spoken english course. use code flash15. t&c apply. ||| aarong: book your dream flat, easy installment. visit your nearest outlet. t&c apply.", "response": {"label": 5, "techniques": [], "action": 0, "confidence": 0.9538473733452734}, "source": "pseudo", "weight": 0.2}
{"id": 61524, "text": "skitto: eid offer! sob item e 40% discount. code: new25.", "response": {"label": 5, "techniques": [], "action": 0, "confidence": 0.9538383016077927}, "source": "pseudo", "weight": 0.2}
{"id": 43902, "text": "Eid offer! Sob item e 30% discount. Code: BDFEST. Shortto projojjo.", "response": {"label": 5, "techniques": [], "action": 0, "confidence": 0.9537888700353806}, "source": "pseudo", "weight": 0.2}
{"id": 54811, "text": "DARAZ: ঈদ উপলক্ষে সব পণ্যে ২০% ছাড়! আজই কিনুন।", "response": {"label": 5, "techniques": [], "action": 0, "confidence": 0.9535041576230439}, "source": "pseudo", "weight": 0.2}
{"id": 21202, "text": "Robi: Eid offer! Sob item e 40% discount.", "response": {"label": 5, "techniques": [], "action": 0, "confidence": 0.9532749803328041}, "source": "pseudo", "weight": 0.2}
{"id": 26244, "text": "eid offer! sob item e 15% discount. dial *121*1#.", "response": {"label": 5, "techniques": [], "action": 0, "confidence": 0.9527328671227779}, "source": "pseudo", "weight": 0.2}
{"id": 39761, "text": "Bata: ঈদ উপলক্ষে সব পণ্যে ২০% ছাড়! আজই কিনুন।", "response": {"label": 5, "techniques": [], "action": 0, "confidence": 0.9526680612848746}, "source": "pseudo", "weight": 0.2}
{"id": 63555, "text": "Eid offer! Sob item e 30% discount. Dial *123*3#. Offer 15 Dec porjonto.", "response": {"label": 5, "techniques": [], "action": 0, "confidence": 0.9522978477621424}, "source": "pseudo", "weight": 0.2}
{"id": 29654, "text": "Star Tech: ঈদ উপলক্ষে সব পণ্যে ২০% ছাড়! আজই কিনুন।", "response": {"label": 5, "techniques": [], "action": 0, "confidence": 0.9521290745349511}, "source": "pseudo", "weight": 0.2}
{"id": 65285, "text": "Tap: Eid offer! Sob item e 10% discount. Code: SAVE20.", "response": {"label": 5, "techniques": [], "action": 0, "confidence": 0.951428807149455}, "source": "pseudo", "weight": 0.2}
{"id": 41491, "text": "bkash: Eid offer! Sob item e 30% discount. Code: WINTER30.", "response": {"label": 5, "techniques": [], "action": 0, "confidence": 0.9511303404911298}, "source": "pseudo", "weight": 0.2}
{"id": 27241, "text": "Yellow:ঈদ উপলক্ষে সব পণ‌্যে ২০% ছাড়! আ‌জই কিনুন। 💰", "response": {"label": 5, "techniques": [], "action": 0, "confidence": 0.95095413785625}, "source": "pseudo", "weight": 0.2}
{"id": 41858, "text": "Eid offer! Sob item e 40% discount. Dial *166*7#.", "response": {"label": 5, "techniques": [], "action": 0, "confidence": 0.9508984209825003}, "source": "pseudo", "weight": 0.2}
{"id": 58001, "text": "Chaldal: Eid offer! Sob item e 10% discount. Dial *166*3#.", "response": {"label": 5, "techniques": [], "action": 0, "confidence": 0.9500470332213141}, "source": "pseudo", "weight": 0.2}
{"id": 67859, "text": "Islami Bank: Eid offer! Sob item e 20% discount. Code: SAVE2081.", "response": {"label": 5, "techniques": [], "action": 0, "confidence": 0.9497911913376283}, "source": "pseudo", "weight": 0.2}
{"id": 23133, "text": "Aarong: Eid offer! Sob item e 20% discount. Dial *166*5#. Offer 30 Nov porjonto.", "response": {"label": 5, "techniques": [], "action": 0, "confidence": 0.9486805187326989}, "source": "pseudo", "weight": 0.2}
{"id": 51404, "text": "Eid offer! Sob іtem e 4​0% discount ⚠️ Code: FLASH15 Offer Sundayporjonto", "response": {"label": 5, "techniques": [], "action": 0, "confidence": 0.948620735999526}, "source": "pseudo", "weight": 0.2}
{"id": 41701, "text": "Bata: Eid offer! Sob item e 40% discount. 📲", "response": {"label": 5, "techniques": [], "action": 0, "confidence": 0.9484830492029765}, "source": "pseudo", "weight": 0.2}
{"id": 29997, "text": "s: weekend sale shuru, 30% porjonto chhar. dial *123*7#. offer sunday porjonto. ||| s: eid offer! sob item e 30% discount.", "response": {"label": 5, "techniques": [], "action": 0, "confidence": 0.9482386976403652}, "source": "pseudo", "weight": 0.2}
{"id": 44971, "text": "Eid offer! Sob item e 40% discount. Dial *123*7#.", "response": {"label": 5, "techniques": [], "action": 0, "confidence": 0.9479169264832624}, "source": "pseudo", "weight": 0.2}
{"id": 39247, "text": "eid offer! sob item e 50% discount. code: flash1531. offer sunday porjonto. 😊", "response": {"label": 5, "techniques": [], "action": 0, "confidence": 0.9475846692701958}, "source": "pseudo", "weight": 0.2}
{"id": 41923, "text": "aarong: ঈদ উপলক্ষে সব পণ্যে ১০% ছাড়! আজই কিনুন। 🔥", "response": {"label": 5, "techniques": [], "action": 0, "confidence": 0.9475563622110597}, "source": "pseudo", "weight": 0.2}
{"id": 44216, "text": "S: Airtel: Eid offer! Sob item e 50% discount. Dial *121*7#. ||| S: Airtel: Weekend sale shuru, 50% porjonto chhar. Offer Friday porjonto.", "response": {"label": 5, "techniques": [], "action": 0, "confidence": 0.9475401221327344}, "source": "pseudo", "weight": 0.2}
{"id": 36660, "text": "BRAC Bak: Eid offer! Sob item e 30% asdkjh discount Code: DEAL50 Sh​ortto рrоjоjjo", "response": {"label": 5, "techniques": [], "action": 0, "confidence": 0.9471358326877463}, "source": "pseudo", "weight": 0.2}
{"id": 45671, "text": "Dutch-Bangla Bank: Eid offer! Sob item e 10% discount. ||| Skitto: Eid offer! Sob item e 10% discount. Code: NEW25. Offer 30 Nov porjonto.", "response": {"label": 5, "techniques": [], "action": 0, "confidence": 0.9470168631952789}, "source": "pseudo", "weight": 0.2}
{"id": 50760, "text": "S:Get 15% cashback on bill payments Open the app to apply Offer 😊 ends Friday ||| S: Book your dream 💰 flat, easy installmеnt. Visit your nearest outlet. T&Capply.", "response": {"label": 5, "techniques": [], "action": 0, "confidence": 0.9467394286388447}, "source": "pseudo", "weight": 0.2}
{"id": 39337, "text": "eid offer! sob item e 20% discount. code: flash15. limited time!", "response": {"label": 5, "techniques": [], "action": 0, "confidence": 0.946590889024304}, "source": "pseudo", "weight": 0.2}
{"id": 60838, "text": "PICKABOO: EID OFFER! SOB ITEM E 10% DISCOUNT.", "response": {"label": 5, "techniques": [], "action": 0, "confidence": 0.9460496759316357}, "source": "pseudo", "weight": 0.2}
{"id": 62810, "text": "Pickaboo: Eid offer! Sob item e 30% discount. Offer Sunday porjonto.", "response": {"label": 5, "techniques": [], "action": 0, "confidence": 0.946015605561318}, "source": "pseudo", "weight": 0.2}
{"id": 35902, "text": "Foodpanda: Eid offer! Sob item e 40% discount. Code: SAVE20.", "response": {"label": 5, "techniques": [], "action": 0, "confidence": 0.9458068088772612}, "source": "pseudo", "weight": 0.2}
{"id": 47414, "text": "Chaldal: ঈদ উপলক্ষে সব 📲 পণ্যে ৩০% ছাড়! আজই কিনুন।", "response": {"label": 5, "techniques": [], "action": 0, "confidence": 0.9457233753477446}, "source": "pseudo", "weight": 0.2}
{"id": 35960, "text": "Chaldal: ঈদ উপলক্ষে সব পণ্যে ৩০% ছাড়! আজই কিনুন।", "response": {"label": 5, "techniques": [], "action": 0, "confidence": 0.9457233753477446}, "source": "pseudo", "weight": 0.2}
{"id": 50905, "text": "Chaldal: Eid offer! Sob item e 50% discount. Offer Friday porjonto.", "response": {"label": 5, "techniques": [], "action": 0, "confidence": 0.9450699112026101}, "source": "pseudo", "weight": 0.2}
{"id": 53903, "text": "Aarong: ঈদ উপলক্ষে সব পণ্যে ৪০% ছাড়! আজই ক‌িনুন। 📲", "response": {"label": 5, "techniques": [], "action": 0, "confidence": 0.9449539741444325}, "source": "pseudo", "weight": 0.2}


In [ ]:
%%writefile outputs/validation_results/folds.csv
id,group,cluster,fold_r0,fold_r1,fold_r2
10001,0,1,0,0,0
10002,1,0,1,1,1
10003,2,1,2,2,2
10004,3,1,4,4,4
10005,4,0,4,0,0
10006,5,0,0,1,1
10007,6,0,1,2,2
10008,7,0,2,4,4
10009,8,1,0,0,0
10010,9,0,2,1,1
10011,10,1,3,3,2
10012,11,1,1,2,3
10013,12,2,2,4,0
10014,13,1,4,4,4
10015,14,2,3,0,2
10016,15,0,0,1,2
10017,16,1,1,2,1
10018,17,1,2,2,3
10019,18,2,4,3,4
10020,19,0,3,3,2
10021,20,0,4,2,0
10022,21,0,0,0,4
10023,22,0,1,1,3
10024,23,0,2,3,1
10025,24,0,0,4,0
10026,25,0,3,0,3
10027,26,1,4,1,2
10028,27,2,1,2,1
10029,28,0,4,4,4
10030,29,2,0,0,0
10031,30,1,1,3,3
10032,31,2,2,1,1
10033,32,0,3,3,3
10034,33,0,3,0,4
10035,34,2,2,2,0
10036,35,1,2,3,2
10037,36,1,0,4,4
10038,37,1,4,4,0
10039,38,0,0,3,1
10040,39,2,1,0,2
10041,40,0,1,1,3
10042,41,1,4,1,4
10043,42,1,3,2,1
10044,43,0,0,4,0
10045,44,0,2,4,2
10046,45,1,3,0,3
10047,46,1,1,1,0
10048,47,1,4,3,3
10049,48,0,0,2,4
10050,49,0,4,2,1
10051,50,0,1,3,2
10052,51,1,1,0,1
10053,52,1,2,3,2
10054,53,1,2,2,4
10055,54,1,3,1,0
10056,55,0,0,4,2
10057,56,2,3,0,3
10058,57,0,4,1,1
10059,58,0,1,4,3
10060,59,1,2,2,4
10061,60,2,2,4,4
10062,61,0,3,4,1
10063,62,1,0,3,2
10064,63,0,4,0,0
10065,64,0,1,1,3
10066,65,0,3,2,4
10067,66,2,0,3,3
10068,67,2,2,1,1
10069,68,0,3,0,4
10070,69,1,3,2,0
10071,70,0,4,3,2
10072,71,1,4,4,1
10073,72,0,2,3,3
10074,73,2,0,0,2
10075,74,1,1,1,0
10076,75,0,0,0,3
10077,76,1,1,4,4
10078,77,0,0,2,3
10079,78,0,4,3,0
10080,79,2,3,2,0
10081,57,0,4,1,1
10082,80,0,2,4,4
10083,81,0,3,1,1
10084,82,0,4,0,0
10085,83,0,0,0,2
10086,84,1,1,1,1


In [ ]:
import importlib
import logging
import os
import random
import sys
import time
import tracemalloc
from pathlib import Path

import numpy as np
import pandas as pd

if "." not in sys.path:
    sys.path.insert(0, ".")

logging.basicConfig(level=logging.INFO, format="%(levelname)s:%(name)s:%(message)s")

from src import config, data_loader, prediction, submission
from src.training import cross_validation, train_embeddings
from tools import validate_submission

SEED = getattr(config, "SEED", 42)
if hasattr(config, "set_seed"):
    config.set_seed(SEED)
else:
    logging.warning("config.set_seed() not found; falling back to local random/numpy seeding.")
    random.seed(SEED)
    np.random.seed(SEED)

print(f"Modules imported successfully. SEED={SEED}, IS_KAGGLE={getattr(config, 'IS_KAGGLE', False)}")


In [ ]:
if hasattr(config, "find_data_dir"):
    DATA_DIR = Path(config.find_data_dir())
else:
    logging.warning("config.find_data_dir() not found; falling back to Path('data').")
    DATA_DIR = Path("data")

SAMPLE_PATH = DATA_DIR / "sample.csv"
TEST_PATH = DATA_DIR / "test.csv"
SAMPLE_SUB_PATH = DATA_DIR / "sample_submission.csv"

for p in (SAMPLE_PATH, TEST_PATH, SAMPLE_SUB_PATH):
    assert p.is_file(), f"Required data file missing: {p}"

TEST_SLICE_ENV = os.getenv("SCAMSHIELD_TEST_SLICE", "").strip()
TEST_SLICE = int(TEST_SLICE_ENV) if TEST_SLICE_ENV.isdigit() else None

if "path" in data_loader.load_sample.__code__.co_varnames:
    df_labeled = data_loader.load_sample(SAMPLE_PATH)
else:
    df_labeled = data_loader.load_sample()

df_test_full = pd.read_csv(TEST_PATH, dtype={"id": str})

if TEST_SLICE is not None and TEST_SLICE < len(df_test_full):
    logging.warning("SCAMSHIELD_TEST_SLICE=%d active — slicing test set for smoke test.", TEST_SLICE)
    df_test = df_test_full.head(TEST_SLICE).copy()
else:
    df_test = df_test_full

print(f"DATA_DIR:     {DATA_DIR}")
print(f"Labeled rows: {len(df_labeled)}")
print(f"Test rows:    {len(df_test)} (full={len(df_test_full)})")


In [ ]:
t0 = time.perf_counter()
tracemalloc.start()

M2_EXP = "E020"
M2_OOF_PATH = config.OOF_DIR / f"{M2_EXP}_m2.npz"
M2_TEST_PATH = config.PREDICTION_DIR / f"{M2_EXP}_m2_test.npz"
expected_m2_rows = TEST_SLICE if TEST_SLICE is not None else len(df_test_full)

need_m2 = not M2_OOF_PATH.exists() or not M2_TEST_PATH.exists()
if not need_m2:
    embedded_ids = np.load(M2_TEST_PATH, allow_pickle=True)["ids"]
    need_m2 = len(embedded_ids) != expected_m2_rows

if need_m2:
    cuda_available = False
    try:
        import torch
        cuda_available = torch.cuda.is_available()
    except ImportError:
        pass
    if not cuda_available and TEST_SLICE is None:
        logging.warning(
            "CUDA unavailable: embedding 500,000 rows on CPU takes roughly 3 hours; "
            "use a Kaggle GPU for the full clean run."
        )
    logging.info("Generating Model 2 OOF and test probabilities (%s)...", M2_EXP)
    train_embeddings.run(
        exp_id=M2_EXP,
        test_limit=TEST_SLICE if TEST_SLICE is not None else None,
    )
    if TEST_SLICE is not None:
        m2_data = np.load(M2_TEST_PATH, allow_pickle=True)
        np.savez_compressed(
            M2_TEST_PATH,
            **{key: m2_data[key][:TEST_SLICE] for key in m2_data.files},
        )

assert M2_OOF_PATH.exists() and M2_TEST_PATH.exists(), "Model 2 artifacts were not generated"
assert len(np.load(M2_TEST_PATH, allow_pickle=True)["ids"]) == expected_m2_rows

print("Running final grouped CV, M1+M2 blend, calibration, final fit, and prediction...")
FINAL_SUBMISSION_PATH = prediction.main(test_limit=TEST_SLICE)
cv_elapsed = time.perf_counter() - t0
print(f"Final pipeline completed in {cv_elapsed:.2f}s")


In [ ]:
t1 = time.perf_counter()
sub_df = pd.read_csv(FINAL_SUBMISSION_PATH)
pred_elapsed = time.perf_counter() - t1

out_dir = Path(getattr(config, "OUT_DIR", Path(".")))
out_dir.mkdir(parents=True, exist_ok=True)
SUBMISSION_PATH = out_dir / "submission.csv"
sub_df.to_csv(SUBMISSION_PATH, index=False, encoding="utf-8")

root_sub = Path("submission.csv")
if root_sub.resolve() != SUBMISSION_PATH.resolve():
    sub_df.to_csv(root_sub, index=False, encoding="utf-8")

sample_for_check = SAMPLE_SUB_PATH
test_for_check = TEST_PATH
if TEST_SLICE is not None:
    sample_for_check = out_dir / "validation_results" / "sample_submission_smoke.csv"
    test_for_check = out_dir / "validation_results" / "test_smoke.csv"
    pd.read_csv(SAMPLE_SUB_PATH).head(TEST_SLICE).to_csv(sample_for_check, index=False)
    df_test_full.head(TEST_SLICE).to_csv(test_for_check, index=False)

errors = validate_submission.validate(
    SUBMISSION_PATH,
    sample_path=sample_for_check,
    test_path=test_for_check,
)
if errors:
    raise RuntimeError("Submission validation failed:\n" + "\n".join(f"  - {e}" for e in errors))

_, peak_mem_bytes = tracemalloc.get_traced_memory()
tracemalloc.stop()
total_elapsed = time.perf_counter() - t0

print(f"VALIDATION PASSED: {SUBMISSION_PATH}")
print(
    f"Prediction load/validate: {pred_elapsed:.2f}s | Total wall time: {total_elapsed:.2f}s | "
    f"Peak traced RAM: {peak_mem_bytes / (1024 ** 2):.1f} MB"
)
print("\nClass distribution:")
print(sub_df["label"].value_counts(normalize=True).sort_index().round(3))
print("\nConfidence summary:")
print(sub_df["confidence"].describe())


In [ ]:
compare_env = os.getenv("SCAMSHIELD_COMPARE_SUBMISSION", "").strip()
if compare_env:
    reference_path = Path(compare_env)
    if not reference_path.is_file():
        raise FileNotFoundError(f"SCAMSHIELD_COMPARE_SUBMISSION not found: {reference_path}")
    reference_df = pd.read_csv(reference_path)
    print(f"DataFrame.equals vs {reference_path}: {sub_df.equals(reference_df)}")
    print(f"Byte equality vs {reference_path}: {SUBMISSION_PATH.read_bytes() == reference_path.read_bytes()}")
else:
    print("SCAMSHIELD_COMPARE_SUBMISSION not set; skipped CSV comparison.")
